# RSNA Knee — complément 320 (séries manquantes)

**Shard 0/2** `proto` (CPU, internet off). **320×320** uint8.

Les séries fluid 320 sont déjà le clone du 224 (mêmes UID, même crop, mêmes coupes). Ce kernel ne les refait pas.

UID **figés** depuis `manifest_proto_part` du crop 130 mm 224 (T1 coronal et axial, hors picks fluid et hors T1 sagittal). Même recette spatiale :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 % si spacing absent
2. bande profondeur **20–80 %**
3. resample **24** coupes, resize **320×320**, uint8, percentiles 1–99

Sortie dans `/kaggle/working/` :
- `volumes_crop130_320_proto_part0.zip`
- `manifest_proto_part0.csv`
- `RECIPE.json`

A déposer dans `data/crop130_320/` à côté des zip fluid.

In [ ]:
SHARD_ID = 0
N_SHARDS = 2

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        # debug local : CSV seulement, pas de DICOM
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
# Kaggle : train_series/<StudyUID>/<SeriesUID>/*.dcm
# (parfois a la racine COMP_DIR, parfois un cran plus bas)
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806","series":"1.2.826.0.1.3680043.8.498.87979090750692080284624781117007957118","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260","series":"1.2.826.0.1.3680043.8.498.23084836536722595275828690293168736174","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513","series":"1.2.826.0.1.3680043.8.498.71462627071432782612166533162526483355","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231","series":"1.2.826.0.1.3680043.8.498.30827030879314217305902637111255984309","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801","series":"1.2.826.0.1.3680043.8.498.10709481291181856834545021650181350504","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018","series":"1.2.826.0.1.3680043.8.498.11984659184847340623187455521474478239","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806","series":"1.2.826.0.1.3680043.8.498.12061584096631077508632129628209972719","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018","series":"1.2.826.0.1.3680043.8.498.70860619701802043401860561907550358128","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10088707526476055079083082931629995449","series":"1.2.826.0.1.3680043.8.498.20043323759072541095414617174838138298","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364","series":"1.2.826.0.1.3680043.8.498.30785636453554791359320199635948074779","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636","series":"1.2.826.0.1.3680043.8.498.85751867415695932706674441547035128463","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343","series":"1.2.826.0.1.3680043.8.498.23233651069805614070606540865878503229","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882","series":"1.2.826.0.1.3680043.8.498.36258630042254758729305084838122618231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471","series":"1.2.826.0.1.3680043.8.498.96789827286986302126324549422969988462","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863","series":"1.2.826.0.1.3680043.8.498.79414559606170408228037412242959219968","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166","series":"1.2.826.0.1.3680043.8.498.18262986451824141381443413686909829481","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10136486417446492428821313562961937182","series":"1.2.826.0.1.3680043.8.498.51084084454936728856990915694601621680","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10148517526101576084961355066637531019","series":"1.2.826.0.1.3680043.8.498.37237025409410099950666378948790724886","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205","series":"1.2.826.0.1.3680043.8.498.10180173995522375163398189474717875046","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973","series":"1.2.826.0.1.3680043.8.498.11063373555867596933273578209763066062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10182565688010898600749222617373997197","series":"1.2.826.0.1.3680043.8.498.12262601433106960284639393383887092405","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725","series":"1.2.826.0.1.3680043.8.498.30408508872190527653389633705344809821","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10162168547770229150261479353227398478","series":"1.2.826.0.1.3680043.8.498.26549824353145138262302671279541386781","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10189392162056736163704940747328177364","series":"1.2.826.0.1.3680043.8.498.11740732272602659379392281696328258247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773","series":"1.2.826.0.1.3680043.8.498.14742042201485078869478491511180615133","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10218285921271312931891912812808052375","series":"1.2.826.0.1.3680043.8.498.16630902380428122462742659928321447586","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10201660732108909756259359440298787775","series":"1.2.826.0.1.3680043.8.498.36856803420888463595920193264903955784","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115","series":"1.2.826.0.1.3680043.8.498.10908568304623504035467450279262604082","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10247930792182001003497189021027466833","series":"1.2.826.0.1.3680043.8.498.11659433169986303483172819201894143082","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115","series":"1.2.826.0.1.3680043.8.498.93287543218012611484019840234197182808","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10211034309703220222647821582770542129","series":"1.2.826.0.1.3680043.8.498.12382624143288964745669413836123425766","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414","series":"1.2.826.0.1.3680043.8.498.92471874072301720761553891425666714628","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10252163031705285141538200795896549203","series":"1.2.826.0.1.3680043.8.498.20013967067892078473845526499599848515","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766","series":"1.2.826.0.1.3680043.8.498.73229709248183617446995179304669316109","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766","series":"1.2.826.0.1.3680043.8.498.61195867264290100563921711711852433937","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10262784969531507133353458615303007839","series":"1.2.826.0.1.3680043.8.498.33328952706795214580541249148048792819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10262784969531507133353458615303007839","series":"1.2.826.0.1.3680043.8.498.12304370715601989966187711977702706585","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10273861326753707436672556008991334542","series":"1.2.826.0.1.3680043.8.498.50250788232183504450267442000416776519","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10299618611626190927173051117228647808","series":"1.2.826.0.1.3680043.8.498.92877978003179927499633610841309561294","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10274956886032850418742861294277331382","series":"1.2.826.0.1.3680043.8.498.64895985949862027443293428780380221630","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10287488031206457680758379452936914669","series":"1.2.826.0.1.3680043.8.498.18219182788189827031887928772723544232","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10309477218703786877612946261946268132","series":"1.2.826.0.1.3680043.8.498.23816706223453344020224240407904285751","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10261794102024899290234062909324134903","series":"1.2.826.0.1.3680043.8.498.30264278418771456519734060951186433217","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738","series":"1.2.826.0.1.3680043.8.498.12371741272528045075821788654879530170","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10304242666970626179776896768140580106","series":"1.2.826.0.1.3680043.8.498.21647436319077947464496009531361339719","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10321556388278642027434955207617064548","series":"1.2.826.0.1.3680043.8.498.35789497814262217098604749942496866116","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10322724865012212544754724404226583315","series":"1.2.826.0.1.3680043.8.498.12870978401576394708550583634587068822","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10326352669696598934883805147731461341","series":"1.2.826.0.1.3680043.8.498.11145186758637043518446857366399416749","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10330805760778577501878048995484987687","series":"1.2.826.0.1.3680043.8.498.28843655461443652834287897281876397783","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10318366890716843554246682186865048846","series":"1.2.826.0.1.3680043.8.498.11035294124377312501783856172465496025","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10354576637002647863048798607011636355","series":"1.2.826.0.1.3680043.8.498.77834783911923335736291528227805236444","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179","series":"1.2.826.0.1.3680043.8.498.71481674289694068040563564652813938771","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10348611327224519953673858098392135936","series":"1.2.826.0.1.3680043.8.498.52505020521115481524991285675872814604","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179","series":"1.2.826.0.1.3680043.8.498.11354203021982743106720525294138594122","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801","series":"1.2.826.0.1.3680043.8.498.40621827593274408107813177177150954786","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10374791734880310643587680407133671218","series":"1.2.826.0.1.3680043.8.498.12629505340899704782067958949831592721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10395211177230760565674811878437948933","series":"1.2.826.0.1.3680043.8.498.57543659608406800480575242648236717798","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387227472110643414333498953513810066","series":"1.2.826.0.1.3680043.8.498.11365950413806191354382685836693130801","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10402431429660337753694827208437698007","series":"1.2.826.0.1.3680043.8.498.33060889247587023780117147063392571355","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10407865232593182855530878478701744409","series":"1.2.826.0.1.3680043.8.498.85632429287138635196815844563432536695","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10422536395266111565753575369125069689","series":"1.2.826.0.1.3680043.8.498.21694314023612293511309971884062882170","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387876708310019345913209598612198432","series":"1.2.826.0.1.3680043.8.498.40149591164211868964032347604988945819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10415616591430643294142401482460086713","series":"1.2.826.0.1.3680043.8.498.71168461947837848800140396098584593065","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10442828926117258624043497279351962738","series":"1.2.826.0.1.3680043.8.498.19335685552269112843468091484312101491","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10442828926117258624043497279351962738","series":"1.2.826.0.1.3680043.8.498.49376918330415107224934671356580803207","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10425131723456547056365404016184663035","series":"1.2.826.0.1.3680043.8.498.73218536788406482353311453087220459950","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10447920208680921688438738178539475509","series":"1.2.826.0.1.3680043.8.498.12557256100879734517965569754969876670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10434135978927658622039675788413759069","series":"1.2.826.0.1.3680043.8.498.48078024353132221856402718520085872787","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695","series":"1.2.826.0.1.3680043.8.498.40258877787887134433688592772246515182","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10485026284343436524865922231113748546","series":"1.2.826.0.1.3680043.8.498.66123899265825770149785331924943179670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10459901081962705567398560859229437818","series":"1.2.826.0.1.3680043.8.498.15385913854350179995371121262900351489","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10495855084258410825408867079175646590","series":"1.2.826.0.1.3680043.8.498.38213738767893629034251742799995055366","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10459901081962705567398560859229437818","series":"1.2.826.0.1.3680043.8.498.10617525032286442761923502016480664304","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10505282086293876725410939800054491454","series":"1.2.826.0.1.3680043.8.498.62019145825711385539085785610410456612","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10495855084258410825408867079175646590","series":"1.2.826.0.1.3680043.8.498.18330874241754295767557491645332724371","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10512466806929129734785318932315918522","series":"1.2.826.0.1.3680043.8.498.13273707243929656401793364347035371808","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10522585237725235866013737575867341946","series":"1.2.826.0.1.3680043.8.498.12280532479271882879927457123485279806","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10533568124272062696642461303295694739","series":"1.2.826.0.1.3680043.8.498.10798993094336432647213839804101833378","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10501068619333941198079851978532746117","series":"1.2.826.0.1.3680043.8.498.15380940264024332616592928680039003642","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828","series":"1.2.826.0.1.3680043.8.498.85285866926488618424127672418510763643","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156","series":"1.2.826.0.1.3680043.8.498.10660852664314576512773973710885619647","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10530226787566708645477257565528473018","series":"1.2.826.0.1.3680043.8.498.11752312521368041265933049669258908763","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156","series":"1.2.826.0.1.3680043.8.498.87394329414885499542217779516402336859","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10541215772904358938061268226270989801","series":"1.2.826.0.1.3680043.8.498.10135821375172863699392732809609582781","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10578013533179563448221888658506023084","series":"1.2.826.0.1.3680043.8.498.74377330130728146629703179660346538995","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169","series":"1.2.826.0.1.3680043.8.498.64777809540902121696715362743978786269","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10563016328082326759266280856730347260","series":"1.2.826.0.1.3680043.8.498.21570196298309210291885243085764510163","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10589889055723532336864750992142411145","series":"1.2.826.0.1.3680043.8.498.95337724948868975961823075422731897406","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10542526906255526405613760733936070251","series":"1.2.826.0.1.3680043.8.498.49560781421342916443519696772141175722","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10593694750925047629619718537436931628","series":"1.2.826.0.1.3680043.8.498.70123357669415805656737336925519503721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10601711635138681269270505152909539290","series":"1.2.826.0.1.3680043.8.498.12069473940220088270591721393180140353","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10615816709018712955642546871639043403","series":"1.2.826.0.1.3680043.8.498.63727968281770767444988085873821315427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10619967804774235096151367674093177430","series":"1.2.826.0.1.3680043.8.498.36573843666696173049026764641712105065","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10603952065748622499432940419853732241","series":"1.2.826.0.1.3680043.8.498.11129361903313479398925104395247769084","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593","series":"1.2.826.0.1.3680043.8.498.67147134172383321074115855277711753277","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10631238417904652761254062374051500506","series":"1.2.826.0.1.3680043.8.498.14649173688130704678046626214854302802","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066","series":"1.2.826.0.1.3680043.8.498.33713300229424206979331394596728155281","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10600654987896381983520458496207741798","series":"1.2.826.0.1.3680043.8.498.46335445306861916646378221650123177514","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971","series":"1.2.826.0.1.3680043.8.498.10346088613010110474605983256744286881","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10652764425372125015319160977894047658","series":"1.2.826.0.1.3680043.8.498.20040198002208537788040456403423745793","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10661650555947788912511478165000737040","series":"1.2.826.0.1.3680043.8.498.12024607756616740176927191712472776231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10657584461509809047185519742966035356","series":"1.2.826.0.1.3680043.8.498.89281222299822380540083609011822279163","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10656388982978239967297620214873636996","series":"1.2.826.0.1.3680043.8.498.82529458988694896780725754232146300968","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10674673641872871321890995010616205087","series":"1.2.826.0.1.3680043.8.498.10589773355337960239999538700855853626","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10686407053180376367066321489809395592","series":"1.2.826.0.1.3680043.8.498.59864558861260857063358712480488082714","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10650751856837517181090589171468851374","series":"1.2.826.0.1.3680043.8.498.21785088381805751892490720966845719763","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10676253869668012791900504544106495589","series":"1.2.826.0.1.3680043.8.498.65009092331383208161472396164162032580","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080","series":"1.2.826.0.1.3680043.8.498.12876673442620591022773197355692357418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080","series":"1.2.826.0.1.3680043.8.498.14311318974303035256807030420716595509","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802","series":"1.2.826.0.1.3680043.8.498.10902849749213465346403392331533932201","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802","series":"1.2.826.0.1.3680043.8.498.93430662806389225278588343422834573092","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10748806810865802591313438958975301375","series":"1.2.826.0.1.3680043.8.498.25134828673023620236499920910521636569","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10726624821670417064567481545750543282","series":"1.2.826.0.1.3680043.8.498.48375381448360416658266043083132278745","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10748806810865802591313438958975301375","series":"1.2.826.0.1.3680043.8.498.72798549999446834141956300840701994399","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762","series":"1.2.826.0.1.3680043.8.498.12084361935673759080393343591654668857","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826","series":"1.2.826.0.1.3680043.8.498.12414368360314712251919576458639408508","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826","series":"1.2.826.0.1.3680043.8.498.10116662845773324052026000780995686078","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10751346650129497348457481341700957445","series":"1.2.826.0.1.3680043.8.498.68807755933173332550575330327965707308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762","series":"1.2.826.0.1.3680043.8.498.36139811491548941456829201644194900991","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10768305343887790230989914438483009428","series":"1.2.826.0.1.3680043.8.498.11083518817199897486042464802591730382","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10829579914809383212986717269028170263","series":"1.2.826.0.1.3680043.8.498.36867574525341385553974127784927174613","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10783501127779906316117667790784040770","series":"1.2.826.0.1.3680043.8.498.58721682632661588848074976538618213287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10818980124981334596808438908811623528","series":"1.2.826.0.1.3680043.8.498.27588624354016760018276784265431608591","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10808663384878038371710260386807468588","series":"1.2.826.0.1.3680043.8.498.63420876273169778546266694899637021395","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10843917293035279597622029625809500972","series":"1.2.826.0.1.3680043.8.498.37644664991855014158089945476229092000","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10854223557359336882116168394037581766","series":"1.2.826.0.1.3680043.8.498.12946822191919877819479892246063050616","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263","series":"1.2.826.0.1.3680043.8.498.13129519289813283933699242733137506983","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10896245499713818848442117831482755673","series":"1.2.826.0.1.3680043.8.498.10736132749531087097079811480530448777","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10876177342063734507664188801203192617","series":"1.2.826.0.1.3680043.8.498.34857239444430784503437877200247295510","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10917606677895195837968927716728385881","series":"1.2.826.0.1.3680043.8.498.46293903826829324792025836895014796249","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655","series":"1.2.826.0.1.3680043.8.498.53213148828170026137041940312722718485","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10919665962088003807153039302898262235","series":"1.2.826.0.1.3680043.8.498.95066015931924364454588463904837539966","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359","series":"1.2.826.0.1.3680043.8.498.67070623886557207951623419646045810532","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10903003118372337016190215211789983264","series":"1.2.826.0.1.3680043.8.498.51656157398949785953609134734854724999","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10931111730564885182398163970692599398","series":"1.2.826.0.1.3680043.8.498.26932998385518304169714246705708146332","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359","series":"1.2.826.0.1.3680043.8.498.40564709858339076848788967276637319815","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626","series":"1.2.826.0.1.3680043.8.498.71176993937045984887485178192374216812","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626","series":"1.2.826.0.1.3680043.8.498.19203615667587645395484665964089253197","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10951328287594724738476663200798896670","series":"1.2.826.0.1.3680043.8.498.74185874352645956314216062499329443281","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768","series":"1.2.826.0.1.3680043.8.498.11871893868980567714339311441974902370","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10965573028231658307306828073340279415","series":"1.2.826.0.1.3680043.8.498.28956268014562032138315085108999535249","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10960770430559425421609885784366918575","series":"1.2.826.0.1.3680043.8.498.22331187374917580606556952211545398417","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962","series":"1.2.826.0.1.3680043.8.498.83690178930371005813774190245621000775","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994854354532517072359864903910225174","series":"1.2.826.0.1.3680043.8.498.63009377438116990863303189359667886290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11007527194601989039620198927241696256","series":"1.2.826.0.1.3680043.8.498.12152336407945042070903349786041063784","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10998405409474759941744305665540376339","series":"1.2.826.0.1.3680043.8.498.19550189169403601256315168917698236155","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480","series":"1.2.826.0.1.3680043.8.498.35944846554473368813529371807566252796","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11011061033863592500429481719734134286","series":"1.2.826.0.1.3680043.8.498.53310496205773842965270172191221711943","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11043926695471016727738735328317975282","series":"1.2.826.0.1.3680043.8.498.12827050470113716868799858134974610649","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11070900122260301212131296598983372089","series":"1.2.826.0.1.3680043.8.498.28400819478168221906868411096090648256","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994554560803816407362754025763036100","series":"1.2.826.0.1.3680043.8.498.29414572606361747732764096297287823836","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11084010833221855396433326769153950681","series":"1.2.826.0.1.3680043.8.498.58408971589922585653403754932244436552","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11040296449332764126530525726037541310","series":"1.2.826.0.1.3680043.8.498.71585447033365787912519640179748543872","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11079556243110299049578981210837702995","series":"1.2.826.0.1.3680043.8.498.80173846444587696730843254547598369684","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547","series":"1.2.826.0.1.3680043.8.498.53596599813707844574925843216643752795","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11091958567356981151034727948765010881","series":"1.2.826.0.1.3680043.8.498.12466335535631629397426499102507026078","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11111022632180183697497694737350901427","series":"1.2.826.0.1.3680043.8.498.12428661641561428031510177499938309062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11123446342116837882811834332727990911","series":"1.2.826.0.1.3680043.8.498.51030196291457733484662016944497791969","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11126383719852491587081347886000714259","series":"1.2.826.0.1.3680043.8.498.56277946662012745845533578136442622874","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290","series":"1.2.826.0.1.3680043.8.498.17184381764505093448899692678937741582","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11103894265181149878864749992746236768","series":"1.2.826.0.1.3680043.8.498.99833984019170770899810570361289185437","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11123446342116837882811834332727990911","series":"1.2.826.0.1.3680043.8.498.19718225929787549754525725052233684236","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11135115047200279264914390528987153419","series":"1.2.826.0.1.3680043.8.498.10724674925723039372459249968487720765","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11145632111083286399683467139301564425","series":"1.2.826.0.1.3680043.8.498.34716280168819137214179207362609752642","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11150597567791052354257165699079348456","series":"1.2.826.0.1.3680043.8.498.11231585327163676125168431117858584563","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11149530279799223497509928655010867960","series":"1.2.826.0.1.3680043.8.498.36436960154373810873908916689136180902","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11152585968424897353533125085307104033","series":"1.2.826.0.1.3680043.8.498.14592543121470926536424003831079712400","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11156884483511690263238401598901614096","series":"1.2.826.0.1.3680043.8.498.42544283891981713453189884565904554413","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11135115047200279264914390528987153419","series":"1.2.826.0.1.3680043.8.498.17208998495562885336807599923355830781","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11183915112115196028799391678356464392","series":"1.2.826.0.1.3680043.8.498.81992436255971386174144831524961429097","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11192287547210248515805146368662883131","series":"1.2.826.0.1.3680043.8.498.75821181738985483720264099454678078601","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11201741528059502111642265469347871878","series":"1.2.826.0.1.3680043.8.498.16535880882710155944588846038842258521","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11173951743276817877673759975853741079","series":"1.2.826.0.1.3680043.8.498.18197452534184519780245131207091930342","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258","series":"1.2.826.0.1.3680043.8.498.10401203504595896441928605534503423062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11190292536746235063218740991532759863","series":"1.2.826.0.1.3680043.8.498.77014852110840872909625135283120624780","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258","series":"1.2.826.0.1.3680043.8.498.79018781824745948347460141919169420352","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11229566047228169850473554486278603754","series":"1.2.826.0.1.3680043.8.498.73191925856001710745765340736006595766","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11218351123386367961865331646414911627","series":"1.2.826.0.1.3680043.8.498.99921849181460843543024533935216237206","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11223691171233118900160265287691267884","series":"1.2.826.0.1.3680043.8.498.10492506964015755184073701033720298590","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11229566047228169850473554486278603754","series":"1.2.826.0.1.3680043.8.498.12434317978423172392089367844577460720","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11232674915186872899189440191805868376","series":"1.2.826.0.1.3680043.8.498.12682567067855622828387506871274033447","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11257168136762043265165109052826442974","series":"1.2.826.0.1.3680043.8.498.29523901881212580589386657150318809783","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11232674915186872899189440191805868376","series":"1.2.826.0.1.3680043.8.498.93827564998232296482906708217262150323","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11270845320057097450047912107461684885","series":"1.2.826.0.1.3680043.8.498.30283551753959006425936734452970690319","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11270845320057097450047912107461684885","series":"1.2.826.0.1.3680043.8.498.12441708157585025854872523785138286688","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11260248089108026217929330361529065683","series":"1.2.826.0.1.3680043.8.498.96755556085603353431682778965644039754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11249981834802444119817546155435759007","series":"1.2.826.0.1.3680043.8.498.50349184734969313204902337569597640650","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11276674006815021828943434205349220551","series":"1.2.826.0.1.3680043.8.498.52634596682552762026530520017838606747","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11287173314232027941677698384842764593","series":"1.2.826.0.1.3680043.8.498.70145905462607039582649572508546278983","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11291875173369366854889507729635968509","series":"1.2.826.0.1.3680043.8.498.55873417497300701849234184809457061290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11282162920338602001456704018617788412","series":"1.2.826.0.1.3680043.8.498.42196701384567593766055965791646327665","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11295899132697862632905938037090529644","series":"1.2.826.0.1.3680043.8.498.81739280016559050332394885945464238064","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11295899132697862632905938037090529644","series":"1.2.826.0.1.3680043.8.498.11787809568620979491270258966474553519","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11298099571581989006135838942599921393","series":"1.2.826.0.1.3680043.8.498.32914721834344564941589017719515617486","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11276674006815021828943434205349220551","series":"1.2.826.0.1.3680043.8.498.46165743730422054749973702946995549381","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11307870660160563329996058267938398411","series":"1.2.826.0.1.3680043.8.498.19576891203894197765482352927593102332","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11334486185870324597335039163116965763","series":"1.2.826.0.1.3680043.8.498.10004845943302932961578284887251371028","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11340429352365972769264064439788186260","series":"1.2.826.0.1.3680043.8.498.48470120565256947524371241732822117804","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11369844885381156110080055326139875897","series":"1.2.826.0.1.3680043.8.498.21563620570271797548468978513260466177","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11314687816975146963598226343179928616","series":"1.2.826.0.1.3680043.8.498.10205605656623725155254975716115771457","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11314687816975146963598226343179928616","series":"1.2.826.0.1.3680043.8.498.67765663587327532607624409861928609133","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238","series":"1.2.826.0.1.3680043.8.498.73426276941399852513998542018430402942","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11385845157446557781575804962818981994","series":"1.2.826.0.1.3680043.8.498.24725345037283573415439855171951180329","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11425019165071014217290822852865654127","series":"1.2.826.0.1.3680043.8.498.77918823455977078844797342740627434419","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602","series":"1.2.826.0.1.3680043.8.498.54554148733198937114744652437713268193","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238","series":"1.2.826.0.1.3680043.8.498.54935794411829066097355160457970604724","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11441489276425334980205891153179893797","series":"1.2.826.0.1.3680043.8.498.88886954862051633150097677980318458276","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597","series":"1.2.826.0.1.3680043.8.498.27873771543687985144892825453784598580","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11448326350892044215419380818679679579","series":"1.2.826.0.1.3680043.8.498.34458041690717786320883335753920466232","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11440214477653629899108986507154583794","series":"1.2.826.0.1.3680043.8.498.29475631756297692455126280634450230076","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11451448731901996559834481853733404099","series":"1.2.826.0.1.3680043.8.498.34724967703489117616196448676960784391","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597","series":"1.2.826.0.1.3680043.8.498.33631499421148004539113351641876448498","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197","series":"1.2.826.0.1.3680043.8.498.15535676340856192556666660993915309674","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11467955500991327547315970867313333592","series":"1.2.826.0.1.3680043.8.498.10872919224447546244506598357268222992","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11455388290952699607973723923118848769","series":"1.2.826.0.1.3680043.8.498.40869364922747559066234322429194823042","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11471148332557464549579838595592736047","series":"1.2.826.0.1.3680043.8.498.11644000660829808236023823890685327075","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11489881286208501558797333161632486099","series":"1.2.826.0.1.3680043.8.498.12426161928826971650775878199232288086","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197","series":"1.2.826.0.1.3680043.8.498.17508866413253936000685059712309195981","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11497761063752461132736045914751707046","series":"1.2.826.0.1.3680043.8.498.10853671211430654352514510850554237271","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11482325994550095339373157256174349858","series":"1.2.826.0.1.3680043.8.498.11741428336848269563223885777047437535","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11497761063752461132736045914751707046","series":"1.2.826.0.1.3680043.8.498.56833988135807589908860014554723339127","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11506642732332295874076832453198901115","series":"1.2.826.0.1.3680043.8.498.77344545541838320436689140089725450500","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11516498203727058146656616413699979433","series":"1.2.826.0.1.3680043.8.498.55260027541471623811376892467496123665","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11498505068787084440256890724559761815","series":"1.2.826.0.1.3680043.8.498.10322983986612959352213669104230746047","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11529405479134457001825038259936796388","series":"1.2.826.0.1.3680043.8.498.67592398936915389119012536014081840765","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11508340006652282767645009271305921426","series":"1.2.826.0.1.3680043.8.498.88656406097274000393951494853291689730","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11542298537902846156898976374522470463","series":"1.2.826.0.1.3680043.8.498.12213358282906869208407668260512853996","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11569601172991867969672947467467392425","series":"1.2.826.0.1.3680043.8.498.10712814712790339974164519147987971585","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11548839502674800051973767797346682106","series":"1.2.826.0.1.3680043.8.498.68610375922699818831444330091653967655","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11577630811395858629711985689258647804","series":"1.2.826.0.1.3680043.8.498.57557016066015267782394467181591651529","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11574480940878399452466699512074769686","series":"1.2.826.0.1.3680043.8.498.64198346668888987597635541739933324476","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11561519302314146020052258325459371962","series":"1.2.826.0.1.3680043.8.498.25379697062877044081295161848958893386","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11583098598455610459898849451117846552","series":"1.2.826.0.1.3680043.8.498.20627850319982740263289394922206621366","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11583098598455610459898849451117846552","series":"1.2.826.0.1.3680043.8.498.10815465141351709268596769656034066608","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11561519302314146020052258325459371962","series":"1.2.826.0.1.3680043.8.498.96379684941862716805955569469948325675","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11587989960888698769912899464470116193","series":"1.2.826.0.1.3680043.8.498.13154343279135125204852407710840591054","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11592056376890319271545710970155310824","series":"1.2.826.0.1.3680043.8.498.81154520340729836602336014403805461037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11596866241403598097171345671773992202","series":"1.2.826.0.1.3680043.8.498.10809817385254740786010423449643675579","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11599387556469808488160280438305325860","series":"1.2.826.0.1.3680043.8.498.13194413063577467094566669903740210086","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11595338216297251382188195835520492789","series":"1.2.826.0.1.3680043.8.498.93670395223792512293996254829085321413","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11613637647831801354398273334552399874","series":"1.2.826.0.1.3680043.8.498.80232839691236434379467430789489613339","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11610176134207131408691324676304640501","series":"1.2.826.0.1.3680043.8.498.11076965464675392441205848750435159583","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11625598825018422177242262569319579378","series":"1.2.826.0.1.3680043.8.498.77043320464734359347687094390195619494","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11641274367861998748475337224297076397","series":"1.2.826.0.1.3680043.8.498.11006727654624315522471736547261328631","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11642228346306175407288158335830376197","series":"1.2.826.0.1.3680043.8.498.99398409625332602121579540436010035600","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11669120316759278775792190599036288128","series":"1.2.826.0.1.3680043.8.498.11779081258913722248829282962083655362","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11682243736539231948790371952356165171","series":"1.2.826.0.1.3680043.8.498.11077147645787997851240110358799424970","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11679044101697033975478370934058085847","series":"1.2.826.0.1.3680043.8.498.34879755101703749562723259219503616144","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11679044101697033975478370934058085847","series":"1.2.826.0.1.3680043.8.498.12622307701760037610179730571716201765","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11695097367686121784104505410611821860","series":"1.2.826.0.1.3680043.8.498.11721112507011653246778407510825259986","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11689820638581431730501617718741779835","series":"1.2.826.0.1.3680043.8.498.14297023582134892230230813777307855455","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928","series":"1.2.826.0.1.3680043.8.498.67301050251233600869705844232192953437","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928","series":"1.2.826.0.1.3680043.8.498.55787743694464420631976381966219903287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11723286647024326363349453996818420374","series":"1.2.826.0.1.3680043.8.498.24479085121451893433173430382802701227","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11695097367686121784104505410611821860","series":"1.2.826.0.1.3680043.8.498.57758266410684389014515409351085614067","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11770690655120964721059321709525979950","series":"1.2.826.0.1.3680043.8.498.92548045540265493493728463985272230666","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11775352515236862473778507708524782015","series":"1.2.826.0.1.3680043.8.498.25049781253213576590800269566953468838","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11716605112302071535035323666308656381","series":"1.2.826.0.1.3680043.8.498.12915784540450644523295826000668806596","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11799272823743145498601173910069737855","series":"1.2.826.0.1.3680043.8.498.10674642636063197010456542407948227897","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11799272823743145498601173910069737855","series":"1.2.826.0.1.3680043.8.498.12727886398692612565745457715791649472","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11804799183537166337413388415687675758","series":"1.2.826.0.1.3680043.8.498.12310954864680582903733245836188834811","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11814923363609140180293604689814785969","series":"1.2.826.0.1.3680043.8.498.17299933291401347611614443696214030820","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11779387602988880691008088661075117406","series":"1.2.826.0.1.3680043.8.498.78542777773195468248339797405014654540","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11830915612184936576804778995164042200","series":"1.2.826.0.1.3680043.8.498.11083351605337075770250709857628080234","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11796547950032543541071896156852778697","series":"1.2.826.0.1.3680043.8.498.10451230569720702824303456456406121246","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11863890747360341056211511874563792862","series":"1.2.826.0.1.3680043.8.498.12355604882573707549506011246746173691","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604","series":"1.2.826.0.1.3680043.8.498.38946384864117135171384663677960789880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11868154141292577776796381772987549697","series":"1.2.826.0.1.3680043.8.498.58521247172333872164251603756153410248","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11880385031047315524154761508473409762","series":"1.2.826.0.1.3680043.8.498.12974976071771723439083951988476811466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11887143891500181344364122992066593229","series":"1.2.826.0.1.3680043.8.498.10497958774005576258456056141881811578","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011","series":"1.2.826.0.1.3680043.8.498.12637357087017427078555292164142753511","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11897083656099489353496670273111097794","series":"1.2.826.0.1.3680043.8.498.31337681759520584943781184307144972909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011","series":"1.2.826.0.1.3680043.8.498.11087112225767928247326918517101775781","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11901474252960403779405194385221103673","series":"1.2.826.0.1.3680043.8.498.32441810509454610438277596406552081200","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11912618147674655465071922267060032851","series":"1.2.826.0.1.3680043.8.498.11914354743578042990183571808547572729","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11923218013055841952315996726408812182","series":"1.2.826.0.1.3680043.8.498.10121513054086095412322965074054631327","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11926412946593048423086380882538180028","series":"1.2.826.0.1.3680043.8.498.13019765955406124790508104724063213433","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11934483804567594363179973225449686890","series":"1.2.826.0.1.3680043.8.498.93351769926829755139011186447426321596","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369","series":"1.2.826.0.1.3680043.8.498.13716234635599956014552013895250623576","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11938864543152001817866013174907274014","series":"1.2.826.0.1.3680043.8.498.38582646609524473521042208085471899154","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11934483804567594363179973225449686890","series":"1.2.826.0.1.3680043.8.498.99943316054853328343693184612516474561","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11920026019322352546372153111621676066","series":"1.2.826.0.1.3680043.8.498.14731039237922793213007066391644414535","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11964826300307458382604479841253457304","series":"1.2.826.0.1.3680043.8.498.96840370346485950544813410240155463502","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11980805687949950504930975019161587970","series":"1.2.826.0.1.3680043.8.498.98998064375679835891417951417719491596","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11992172807590757302730513420827032389","series":"1.2.826.0.1.3680043.8.498.71074591215970878844614053209922285344","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11957476321850340968244508976719889277","series":"1.2.826.0.1.3680043.8.498.54365352876095966171558784779554861056","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11988551795896617815207749665102049499","series":"1.2.826.0.1.3680043.8.498.40474937549159218614525338583527956221","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12005327370242618323737652892230673473","series":"1.2.826.0.1.3680043.8.498.21770352444254401700334791770019159657","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12007335335641704192494936946829650792","series":"1.2.826.0.1.3680043.8.498.79720215474764774702548415752693811767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11977465062890611952998635094033472451","series":"1.2.826.0.1.3680043.8.498.11214379029766993573010708654862859536","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12010150865087936209475730225377000084","series":"1.2.826.0.1.3680043.8.498.50823238587101586335237204227580429849","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12015406188790105426257083591807220760","series":"1.2.826.0.1.3680043.8.498.10741453405479600030121266145300300258","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12036404333388388978317335770611404933","series":"1.2.826.0.1.3680043.8.498.81712637977161795523558008311300041525","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12054099347723802635102026371218576198","series":"1.2.826.0.1.3680043.8.498.28741009779146411817875168190881489063","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12041491173612341571223481616114183729","series":"1.2.826.0.1.3680043.8.498.24933948061011119327037275675401586799","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12064822036518837577612019712902340953","series":"1.2.826.0.1.3680043.8.498.11591440518165842045256561646017601624","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12073401429398042797505820525732905336","series":"1.2.826.0.1.3680043.8.498.11646695881688512265758773171744227365","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12100254887364913022980951672127944515","series":"1.2.826.0.1.3680043.8.498.30253901671420957760016451229977123909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267","series":"1.2.826.0.1.3680043.8.498.66214009867810440930296992227457469145","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12135456587022560662140325717612726990","series":"1.2.826.0.1.3680043.8.498.94448077130340841701274896751522434246","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12050415099056648944934039580585859179","series":"1.2.826.0.1.3680043.8.498.50699047078402990941500110970555763384","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12068674252637046261029360607215822017","series":"1.2.826.0.1.3680043.8.498.53842863208242402680471710397069468905","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267","series":"1.2.826.0.1.3680043.8.498.13348101806553877324577421903421844186","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12166081590548185636776183599317715045","series":"1.2.826.0.1.3680043.8.498.12903137383558124548346418807599252300","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12176466551281285838907773521868954581","series":"1.2.826.0.1.3680043.8.498.12731338636677261325511791627759359543","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12154188739734505793699974626991872621","series":"1.2.826.0.1.3680043.8.498.12813703960078945494493530082123341822","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12160065317167766993207036411434801297","series":"1.2.826.0.1.3680043.8.498.96042754572468293593006040122218089259","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12171469604213841534511019872648327851","series":"1.2.826.0.1.3680043.8.498.93157109844013014444779429138579940548","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12178921459424917661210327999672203211","series":"1.2.826.0.1.3680043.8.498.32720389905266006458714955048441139359","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12186494601308291174548814564387764515","series":"1.2.826.0.1.3680043.8.498.44195921250242998302179423736796357537","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12201598444731259697380878112783653796","series":"1.2.826.0.1.3680043.8.498.28705867214486255587531203663675392012","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12207279366369605175255203969558548753","series":"1.2.826.0.1.3680043.8.498.82240758773355695461425525486004944169","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12215902620611618183321984433114460986","series":"1.2.826.0.1.3680043.8.498.57229935060360507512473006989665419797","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12195090472992304712555822541659656360","series":"1.2.826.0.1.3680043.8.498.20624400107035562940776897688360333749","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12229628534316014336864328244217875465","series":"1.2.826.0.1.3680043.8.498.77664171814406878377789127426017865310","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12182309401920763498667366352681074302","series":"1.2.826.0.1.3680043.8.498.63270357751551632749928459904181619295","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12229628534316014336864328244217875465","series":"1.2.826.0.1.3680043.8.498.29727023765439474464302324612664820608","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12238753439953831519247789198174756615","series":"1.2.826.0.1.3680043.8.498.10756811194153083532992026997032203235","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12238753439953831519247789198174756615","series":"1.2.826.0.1.3680043.8.498.34455061265614397342832034148373711045","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12241413500128553133597113709525575494","series":"1.2.826.0.1.3680043.8.498.10336394951722636029707214007144773842","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12243764506921404196245986681613075136","series":"1.2.826.0.1.3680043.8.498.94012746347110951192829574692121650585","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12246086238999179953298261549112671014","series":"1.2.826.0.1.3680043.8.498.48166118273537380990308720990636641339","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12241413500128553133597113709525575494","series":"1.2.826.0.1.3680043.8.498.10135085758541254302051254552798844163","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12256365473385257798561212457071712327","series":"1.2.826.0.1.3680043.8.498.84165265932642387623736332575768614953","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12257484608743859456600273083454447341","series":"1.2.826.0.1.3680043.8.498.71969054802122081627522096376518395605","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12257484608743859456600273083454447341","series":"1.2.826.0.1.3680043.8.498.56769496223753360304389512240564484884","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12256365473385257798561212457071712327","series":"1.2.826.0.1.3680043.8.498.89858881895612879602646374501929112752","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12267944258766605501055560486655358039","series":"1.2.826.0.1.3680043.8.498.24565939130272277844487353863141995552","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12266133889557145601371064833389800065","series":"1.2.826.0.1.3680043.8.498.11840863379283649751884156405229799315","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12274801503555668613400469130773239683","series":"1.2.826.0.1.3680043.8.498.47363125232922539916347796380503447339","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12264607993101717201407955391437731134","series":"1.2.826.0.1.3680043.8.498.56797125809501797997649624361477482211","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12280407492991430179266369843997653205","series":"1.2.826.0.1.3680043.8.498.98547990995738129600352200890305696730","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12280407492991430179266369843997653205","series":"1.2.826.0.1.3680043.8.498.61708957883181496207022279314824901625","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12285177271228289867211553575739205132","series":"1.2.826.0.1.3680043.8.498.11496985612579910123565911554608752040","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12285177271228289867211553575739205132","series":"1.2.826.0.1.3680043.8.498.26843705691144844596009192792253010661","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12273356591019341319494035708698721367","series":"1.2.826.0.1.3680043.8.498.82244179458701501947613155384596080837","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12295284437770380354725533247473017729","series":"1.2.826.0.1.3680043.8.498.12438199441699505482213713357048208815","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12296624595750602443033689366743784387","series":"1.2.826.0.1.3680043.8.498.32845047960008317816572501179807242779","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12295284437770380354725533247473017729","series":"1.2.826.0.1.3680043.8.498.21515306653523763613064783220424514345","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12305281328019552390691110900015627668","series":"1.2.826.0.1.3680043.8.498.54900075915279412841403371799181669854","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12307356432736970350794033171174403632","series":"1.2.826.0.1.3680043.8.498.99960433273557168945332980270758348163","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12301945990161670321760708884248055936","series":"1.2.826.0.1.3680043.8.498.12984412030655226842058048012977672396","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12314729913655626716039376606973474150","series":"1.2.826.0.1.3680043.8.498.35693576675468519447604417889589773848","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12314729913655626716039376606973474150","series":"1.2.826.0.1.3680043.8.498.69006855054819705568511467294061632691","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12326930847104022196329390608780066947","series":"1.2.826.0.1.3680043.8.498.90278443346626867350765162900005017158","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12333996498811456012744466468462396081","series":"1.2.826.0.1.3680043.8.498.33994302602647933128707045029806574972","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12331841096593105748526634055968654914","series":"1.2.826.0.1.3680043.8.498.10039733347836310036939398053348895446","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12355011301785011370279005103251815187","series":"1.2.826.0.1.3680043.8.498.69572127155379835976920233822006197141","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12345064031074704673835101507426902246","series":"1.2.826.0.1.3680043.8.498.61221858151576122853838833339689075338","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12374183329974116022606010915169117790","series":"1.2.826.0.1.3680043.8.498.12002991624729783262465057339310468032","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12348960754791091583201828167607825033","series":"1.2.826.0.1.3680043.8.498.66939916851806381827037754969148387256","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12379570725733005682132316461218878636","series":"1.2.826.0.1.3680043.8.498.10507365085603578123729615425987725247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12386255059403233724886571725482313876","series":"1.2.826.0.1.3680043.8.498.60270109925472586697812455986709798981","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12352132666379687393012255589355771387","series":"1.2.826.0.1.3680043.8.498.64466083433041592504533442646488641824","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12406043058701411628555010402350032739","series":"1.2.826.0.1.3680043.8.498.74702419628045052846195194723775141777","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408314059030096510457440879771212020","series":"1.2.826.0.1.3680043.8.498.50623541619120568480821618308602541860","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12426381920136706157912507136730458010","series":"1.2.826.0.1.3680043.8.498.12476206292683557050070170926527943317","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12462703906850759490077163612851243080","series":"1.2.826.0.1.3680043.8.498.11498680943128516681590603508560718657","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12438775479300781065075835270622538230","series":"1.2.826.0.1.3680043.8.498.22188541879379387356577552068102528511","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910","series":"1.2.826.0.1.3680043.8.498.12012846866132739640873958327520867072","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408945963789246646229147558387793270","series":"1.2.826.0.1.3680043.8.498.34975249118332179190340489759967248596","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12473533314363654795650919604551685896","series":"1.2.826.0.1.3680043.8.498.87881650287385497204741353923109414830","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12481913908649439189738319460342358360","series":"1.2.826.0.1.3680043.8.498.11587596693855513566699314135653151395","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12484700311993345120015691857089576898","series":"1.2.826.0.1.3680043.8.498.37652801049726327088077975804750515169","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12489752598467506094826904655029553699","series":"1.2.826.0.1.3680043.8.498.12707440311028479852734571117562305287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491395125580387847235487841392723348","series":"1.2.826.0.1.3680043.8.498.77657134406610723985453108562447745069","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12489752598467506094826904655029553699","series":"1.2.826.0.1.3680043.8.498.22204020867787235387913012086919880209","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491796764641306046176352066215240074","series":"1.2.826.0.1.3680043.8.498.33118459143370246234610331614793196300","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12497144849433082154223897356471317072","series":"1.2.826.0.1.3680043.8.498.43349730538110110071531484201970400089","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12501925425699863373513819814520995919","series":"1.2.826.0.1.3680043.8.498.11257197106770766228713602585969068308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491796764641306046176352066215240074","series":"1.2.826.0.1.3680043.8.498.59233436666055342259713041291708777766","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12507957850103916926719601413279798696","series":"1.2.826.0.1.3680043.8.498.26436136520404763932959480889339873131","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12526876471494728147635430816990398219","series":"1.2.826.0.1.3680043.8.498.11020283634771050365517944354105849855","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12533105585687352326867082376911240942","series":"1.2.826.0.1.3680043.8.498.77738882407207531314197472772164625107","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12509944331769453135724043962410129357","series":"1.2.826.0.1.3680043.8.498.12411158088598982086937616514309407442","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12553455350653672209409408595948190982","series":"1.2.826.0.1.3680043.8.498.11970003939626181119744013678530304379","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12559687473583953266143897736224215680","series":"1.2.826.0.1.3680043.8.498.32743599831047365616976996511348749847","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12568337973304376222940287737691898395","series":"1.2.826.0.1.3680043.8.498.24014121320642853623111841740566509924","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12552701911592685137269544109687472988","series":"1.2.826.0.1.3680043.8.498.53814263143818363025637262218935485767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12570576337098834311171883034999327454","series":"1.2.826.0.1.3680043.8.498.11890117362879911392838632929938487214","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12574270269124650675934073761741661292","series":"1.2.826.0.1.3680043.8.498.10336976938665370705925677004162348783","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12577828789206467631166875572423539298","series":"1.2.826.0.1.3680043.8.498.11533938700541278622422694860439869461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12570576337098834311171883034999327454","series":"1.2.826.0.1.3680043.8.498.97349071835893908339382328969818985158","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12580232188757225504172709417466515184","series":"1.2.826.0.1.3680043.8.498.60132801491926083582491700083257138231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12598627296066406747921546933391681918","series":"1.2.826.0.1.3680043.8.498.14370125155144367669397352426783244294","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973","series":"1.2.826.0.1.3680043.8.498.27146062832035348929206890381154881403","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973","series":"1.2.826.0.1.3680043.8.498.37032826821035854589423026056104771845","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12614097867016983076979065551985251584","series":"1.2.826.0.1.3680043.8.498.76698423091696603308977632822987281874","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12629372332421954368597330267888483345","series":"1.2.826.0.1.3680043.8.498.34472924470178467348339361895972309628","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12607659092357834912029375690348886159","series":"1.2.826.0.1.3680043.8.498.12803037699082480306633482790012748409","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12637591861595747322913829625646935491","series":"1.2.826.0.1.3680043.8.498.11154885276488973751714409289440343361","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12644400073937544865441360809610886635","series":"1.2.826.0.1.3680043.8.498.89393107145588398732148135928712089957","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172","series":"1.2.826.0.1.3680043.8.498.11653596402418616231044114346669158320","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172","series":"1.2.826.0.1.3680043.8.498.17759260281473746379249610110375053985","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12640439197386154384526090653248054582","series":"1.2.826.0.1.3680043.8.498.52865871330972672365946425900496715852","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12668708646283106269005674386248135220","series":"1.2.826.0.1.3680043.8.498.29554687101728108233947237265383503753","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12658572494577556884626613972035441625","series":"1.2.826.0.1.3680043.8.498.59261179613172600683863119655142813082","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12674069717230126880288353271062347095","series":"1.2.826.0.1.3680043.8.498.51881879771586296768304161516669165175","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12676359576477452816896379524598090410","series":"1.2.826.0.1.3680043.8.498.99395882585639470663192212630582899128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12690803821768205116702487219908820630","series":"1.2.826.0.1.3680043.8.498.43475452295441929303058000527885521475","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12690803821768205116702487219908820630","series":"1.2.826.0.1.3680043.8.498.11118195156228185257091646709817623422","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12697859103448296208701081683487338111","series":"1.2.826.0.1.3680043.8.498.67644112337807971365514928309118426499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12717728703860353055063551162640833849","series":"1.2.826.0.1.3680043.8.498.99742552647138207772944162520167143655","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12719397512102459853075227679498459688","series":"1.2.826.0.1.3680043.8.498.99239177064704463645613574120857503193","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12715130507546263968763238702574191950","series":"1.2.826.0.1.3680043.8.498.11108003381611882194332132192906362168","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12727488524631885827798129418558343386","series":"1.2.826.0.1.3680043.8.498.28038508695501263241847349494597143012","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12724250000565012779853884787450109956","series":"1.2.826.0.1.3680043.8.498.48343167332098408635254196198402167770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12754381146006839341912662615844567067","series":"1.2.826.0.1.3680043.8.498.96154335942332055089378317225227749496","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12762900657007480138202359874575403361","series":"1.2.826.0.1.3680043.8.498.12689825173506806036971620703093931768","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12775105605170844918620305197829216665","series":"1.2.826.0.1.3680043.8.498.46736650111491199405491732454419787960","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12771448321084229688831202344007889535","series":"1.2.826.0.1.3680043.8.498.51020481064310704933530210972010327785","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12790542912560396272981117250277568532","series":"1.2.826.0.1.3680043.8.498.98831565753675240739609409848860987107","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12785620442256422697216192462703273802","series":"1.2.826.0.1.3680043.8.498.36231521173847206377664978637799986925","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12800130397185143653180509040746675358","series":"1.2.826.0.1.3680043.8.498.11887297483714221287034895637968479422","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12761906443846433581503942530016459978","series":"1.2.826.0.1.3680043.8.498.14291780066865918861778131501245637967","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12801643268939133682891906705186571010","series":"1.2.826.0.1.3680043.8.498.11467138572238808015835936247474612292","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12798419553759200632295966180228523212","series":"1.2.826.0.1.3680043.8.498.11547561702788322971316624452361789020","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12810074901642984174782853829149619043","series":"1.2.826.0.1.3680043.8.498.12028015624082793300229800710890100004","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12815132582382825980088023336277934996","series":"1.2.826.0.1.3680043.8.498.72591720011372335977352290412127385233","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12818010040786942613541254450332754046","series":"1.2.826.0.1.3680043.8.498.27963413412379765180472770218392990688","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12820051821104182459960036809430047569","series":"1.2.826.0.1.3680043.8.498.71025802642727941731802898978385344707","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12904190317996738774518258536997285832","series":"1.2.826.0.1.3680043.8.498.10108090931013822790901496624547993503","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12831769902826114242555323443809541052","series":"1.2.826.0.1.3680043.8.498.85903345310002600935342530209597142696","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12862782592303338109631225504988756108","series":"1.2.826.0.1.3680043.8.498.80497338217356011526281188664963830642","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12912436233176216254115594260643600155","series":"1.2.826.0.1.3680043.8.498.38688701975173494850421439714152699275","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12912436233176216254115594260643600155","series":"1.2.826.0.1.3680043.8.498.62024837984566674745838214105863635337","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664","series":"1.2.826.0.1.3680043.8.498.68608662210414660456623573930613845952","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664","series":"1.2.826.0.1.3680043.8.498.42728894453720330458191403252985663444","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265","series":"1.2.826.0.1.3680043.8.498.12425365367662767797065142147538604226","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12954163390367261226110152166469167935","series":"1.2.826.0.1.3680043.8.498.13012202730436801630565933011131395933","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12949322114105642048256996295428966924","series":"1.2.826.0.1.3680043.8.498.86925822068788896797977531849168666363","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265","series":"1.2.826.0.1.3680043.8.498.99794601079093793611579856041752546448","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12957752106697316015996777476239762111","series":"1.2.826.0.1.3680043.8.498.11744743084974014118553088601233168009","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12978172997053500029791786614906570285","series":"1.2.826.0.1.3680043.8.498.12063538788496538836920957959230503607","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12980448935316870673341494036873313299","series":"1.2.826.0.1.3680043.8.498.10577210597786937162496073412182909868","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12980448935316870673341494036873313299","series":"1.2.826.0.1.3680043.8.498.25121430353021414639747047958789120952","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12961116776122838195555132309258963046","series":"1.2.826.0.1.3680043.8.498.10012053860908951630034517323361929409","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12986028147682493711912242081052787346","series":"1.2.826.0.1.3680043.8.498.78796094223525918081758501420836735128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12982826264059547618124186257414818172","series":"1.2.826.0.1.3680043.8.498.70756384117522607813842731571254640433","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13016684730504329799460698013153308088","series":"1.2.826.0.1.3680043.8.498.90527762083320810616016835345058142499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13003520962744192762179786723724282013","series":"1.2.826.0.1.3680043.8.498.72347493306797938728236440444714462700","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13009271274636013067662786774946029285","series":"1.2.826.0.1.3680043.8.498.93231024109273104527730286606347716797","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13024250150960117282590397343184827466","series":"1.2.826.0.1.3680043.8.498.50608858775560107633339329024568924801","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13027024577729659065187419038453181852","series":"1.2.826.0.1.3680043.8.498.62882889127229871686192780838291152599","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13026024768908635083636086393662414870","series":"1.2.826.0.1.3680043.8.498.12871036502342638811688417538955451310","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13034169434311535671044673510644711229","series":"1.2.826.0.1.3680043.8.498.62256727612139688251154001572262469660","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13062438493893996542058869514390908492","series":"1.2.826.0.1.3680043.8.498.11261653077426031332625731979391724111","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13026024768908635083636086393662414870","series":"1.2.826.0.1.3680043.8.498.10585757307102148506582411797574357185","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13042847643086819857036458569632034999","series":"1.2.826.0.1.3680043.8.498.57020178429856025451416068617066550431","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13069081594164738319108129224400751974","series":"1.2.826.0.1.3680043.8.498.68900241887686256293534234556194881207","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13073387272228394257811861905056479438","series":"1.2.826.0.1.3680043.8.498.11072522293527069439770688811820020741","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13078834627392793401892111941627273421","series":"1.2.826.0.1.3680043.8.498.13419898958792828572608370029750057298","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13078009378379325641560321462188164769","series":"1.2.826.0.1.3680043.8.498.84048555440710188609230339719630180341","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13088550794087945662322609097092458487","series":"1.2.826.0.1.3680043.8.498.77680823601804170220833949558459546075","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13086260989634665810181105278930088749","series":"1.2.826.0.1.3680043.8.498.19460007553265267217738558325800531328","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13096288495834275184930201615129174987","series":"1.2.826.0.1.3680043.8.498.79864978602552541511404752012649787731","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13118532243906060645114391749986421707","series":"1.2.826.0.1.3680043.8.498.92502737523192082435615200592591097475","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13114228294359876639109410575915394712","series":"1.2.826.0.1.3680043.8.498.25631418567290041262873690097185653987","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13093528938282977263747792690249036456","series":"1.2.826.0.1.3680043.8.498.86488633500999378546669003020200053177","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13116941594523500503401720636978356127","series":"1.2.826.0.1.3680043.8.498.12585434086335603762934219031878074111","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13126070481622692281046898445918832841","series":"1.2.826.0.1.3680043.8.498.10654657658027922821506862732324254320","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13131420451380633781538717117200087258","series":"1.2.826.0.1.3680043.8.498.10283456805442863947708694339090396231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13126070481622692281046898445918832841","series":"1.2.826.0.1.3680043.8.498.10446332118650617353720605002660282199","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13135567399913958681613600650440761032","series":"1.2.826.0.1.3680043.8.498.10894573580356338747662386185667831714","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13136392756016978765204420377500607545","series":"1.2.826.0.1.3680043.8.498.96965711659153543373280925591603523770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13135567399913958681613600650440761032","series":"1.2.826.0.1.3680043.8.498.44177755196535822847248610561942355841","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13129258242739880102472095242703169773","series":"1.2.826.0.1.3680043.8.498.11778992497524987974226903180298823379","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13143446880514535170343243531129864116","series":"1.2.826.0.1.3680043.8.498.19512071316334566149090652023514279688","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13147249459968468500804358644901642270","series":"1.2.826.0.1.3680043.8.498.83149498469258321819240383673190917220","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13147249459968468500804358644901642270","series":"1.2.826.0.1.3680043.8.498.43980687171532504774720685920111009403","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13151845892175420505816340845155354956","series":"1.2.826.0.1.3680043.8.498.11723417050732618048241721015605082969","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13137549927896953291645459480590055925","series":"1.2.826.0.1.3680043.8.498.95941217263098579502456561765065955187","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032","series":"1.2.826.0.1.3680043.8.498.10622847405565705403945786287990550954","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032","series":"1.2.826.0.1.3680043.8.498.27199121110110763503785074807190041199","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13163727085668231782866823012535268245","series":"1.2.826.0.1.3680043.8.498.59760620017954521062433966722919625217","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13173926977743814814616127342102020710","series":"1.2.826.0.1.3680043.8.498.11319595832817212604878811769434721742","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13173050135651736408720687361834045815","series":"1.2.826.0.1.3680043.8.498.74344784889991148956271118832389450998","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13167727300538353052482338786183678684","series":"1.2.826.0.1.3680043.8.498.12341092394857002356914031491426229558","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13162284470388627411596358366248494046","series":"1.2.826.0.1.3680043.8.498.29764529726408539666984756693084546524","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13179169600887013946657470830682778717","series":"1.2.826.0.1.3680043.8.498.12832668589575052468039888415165990574","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13211838518836535713919214396985811434","series":"1.2.826.0.1.3680043.8.498.10408989185029972401995366241754102474","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13207489724395276840990928372610055495","series":"1.2.826.0.1.3680043.8.498.12118741478214766284539566795612575843","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13214267416673058581197714011913363750","series":"1.2.826.0.1.3680043.8.498.64762305466729951252689566796366428040","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13224933005220329091792052916905776921","series":"1.2.826.0.1.3680043.8.498.59480750415179500184169313802076584420","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13211838518836535713919214396985811434","series":"1.2.826.0.1.3680043.8.498.56963537577799545856692198966359902994","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13234278816788324761159326624155958655","series":"1.2.826.0.1.3680043.8.498.25674611753338662533530472182915944359","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13234278816788324761159326624155958655","series":"1.2.826.0.1.3680043.8.498.57191162647593192902676660927764700297","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13229563888378977350299319159748823477","series":"1.2.826.0.1.3680043.8.498.27869674715378862224829808342924745879","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13200225436827306542001129042739578441","series":"1.2.826.0.1.3680043.8.498.10260035364406876815719322958772444042","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13261595972961358158990410740461308054","series":"1.2.826.0.1.3680043.8.498.55878900350178600868889123979213620444","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13262473718294425031790270527413368245","series":"1.2.826.0.1.3680043.8.498.61562118046114157246083271150852286123","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13242536312535334908597493894251783308","series":"1.2.826.0.1.3680043.8.498.52425995352921691791654659570002187778","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13263128568157735797391439005687112733","series":"1.2.826.0.1.3680043.8.498.54496517056330373078905986891937509337","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13273890409523605456488117581706115075","series":"1.2.826.0.1.3680043.8.498.36582540871128035077919243531502484119","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13295231176137292542631413614945667314","series":"1.2.826.0.1.3680043.8.498.11438505509556639800345695025420947309","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982","series":"1.2.826.0.1.3680043.8.498.11322695511401983536538565837577264736","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13286048014183811835847662577023535662","series":"1.2.826.0.1.3680043.8.498.24667191054129740156202925503860325560","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875","series":"1.2.826.0.1.3680043.8.498.11815694482487553001949474160944631615","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13295231176137292542631413614945667314","series":"1.2.826.0.1.3680043.8.498.30092602730022089766861149300903773661","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875","series":"1.2.826.0.1.3680043.8.498.11400782415751700171066250191715503242","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13323187196249612936800308722675884644","series":"1.2.826.0.1.3680043.8.498.35373079750417088735206850985213830685","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982","series":"1.2.826.0.1.3680043.8.498.21645253967107068689450581612579844855","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13297973188397662905314777604089528518","series":"1.2.826.0.1.3680043.8.498.87414588343784914417212011848281585458","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903","series":"1.2.826.0.1.3680043.8.498.68216362009231374856176479984344037705","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13357923062946813820390731946535198170","series":"1.2.826.0.1.3680043.8.498.18334568037433683791485863369299078962","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13353409340862821090624127486679097099","series":"1.2.826.0.1.3680043.8.498.85610759384836070208803930679827278017","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13341507176140340924941654608476073030","series":"1.2.826.0.1.3680043.8.498.10564954105710350113083345807064415065","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13363856251607349805111249441972460575","series":"1.2.826.0.1.3680043.8.498.61718286142737796659784907062776621983","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13367740087756379568826046059252678700","series":"1.2.826.0.1.3680043.8.498.43539042322278850379152656352376288247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13377426362348493372112241208607685717","series":"1.2.826.0.1.3680043.8.498.80917273700700432825726526156869574714","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13377426362348493372112241208607685717","series":"1.2.826.0.1.3680043.8.498.17547497673960695811629174516009936531","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13383972749798909973472489456126521194","series":"1.2.826.0.1.3680043.8.498.17656711803599248377594417280838418867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13405120554981098042415743645998258385","series":"1.2.826.0.1.3680043.8.498.11622250359196650502537044172308464771","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13460268079835549295338279322112693785","series":"1.2.826.0.1.3680043.8.498.47123395599170166434705820763512813026","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13459473184684263663596121399737033933","series":"1.2.826.0.1.3680043.8.498.45533153391210942518670736344378740118","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13460268079835549295338279322112693785","series":"1.2.826.0.1.3680043.8.498.10215254483285389434699913098277254104","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13405120554981098042415743645998258385","series":"1.2.826.0.1.3680043.8.498.17276488279720011745912583346912917358","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13575953449958751965232580597585303226","series":"1.2.826.0.1.3680043.8.498.13062879795509306416292689425592101714","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13463672277039459152171414002273376185","series":"1.2.826.0.1.3680043.8.498.49098601323390422851145639246753988536","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13762716088334302039143651507177887969","series":"1.2.826.0.1.3680043.8.498.97986291329456633657104469386786421259","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13616982533395291590655841043722813973","series":"1.2.826.0.1.3680043.8.498.13164331664225578396963600067456858402","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13915367834258386094001181500560177470","series":"1.2.826.0.1.3680043.8.498.94019237224647542389507155291218110188","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13695989890112408005495264068064110179","series":"1.2.826.0.1.3680043.8.498.46994041934285903958945317843925646630","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13972325309264527611989585341636140156","series":"1.2.826.0.1.3680043.8.498.10885838708233481956985079426822149078","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14117402902169664329719927863268363596","series":"1.2.826.0.1.3680043.8.498.48023464407504409225343537792943296339","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14188540532926067083914263496492255704","series":"1.2.826.0.1.3680043.8.498.91920928412195731000272906737605823695","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14229598279617168024212382336641104878","series":"1.2.826.0.1.3680043.8.498.35850260097964453137320183425471746639","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14024129019390041106306062001538710978","series":"1.2.826.0.1.3680043.8.498.10652747667138737825221820590139327506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14300107110014472682373589824643789093","series":"1.2.826.0.1.3680043.8.498.11947578595198143326999774651689900672","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14400292895187181989838122627394761532","series":"1.2.826.0.1.3680043.8.498.64683873754721161320182343190417653716","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14472261871417480346162193127520918127","series":"1.2.826.0.1.3680043.8.498.11914414021060839385347519330555677886","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14347036698014475959973071477777915575","series":"1.2.826.0.1.3680043.8.498.45741040243669138742623013453601454750","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14479913217228604289584726096114526268","series":"1.2.826.0.1.3680043.8.498.61985151738410034547318763908318691314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14728068688166107022558957899468489735","series":"1.2.826.0.1.3680043.8.498.38558604628961050502187561748184601425","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14806659119591146961784818305031900863","series":"1.2.826.0.1.3680043.8.498.11674243102119280114796004543157521685","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14495028403937132710814780922642512791","series":"1.2.826.0.1.3680043.8.498.76360421277415163214911201560328593365","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14963472809950919852176661973255059069","series":"1.2.826.0.1.3680043.8.498.37042945038334407747602238241975774203","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15025957400098336605043184930980421418","series":"1.2.826.0.1.3680043.8.498.55119271123212202820221640749400352065","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14472261871417480346162193127520918127","series":"1.2.826.0.1.3680043.8.498.63296658633482703173315697924995518270","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.15090991392324602559982474708414427621","series":"1.2.826.0.1.3680043.8.498.87446640076048533838687516880056283555","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15025957400098336605043184930980421418","series":"1.2.826.0.1.3680043.8.498.10274525526979095809579714083485380730","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.15133818248365103660456294775831248835","series":"1.2.826.0.1.3680043.8.498.92888655827648731540938657011863480608","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14850534390650184292244380816212794996","series":"1.2.826.0.1.3680043.8.498.50466301514188270757926622338695634764","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.15219824433491621460467738389345697247","series":"1.2.826.0.1.3680043.8.498.62442469813000649442734664254118050701","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15360150088598749206650025637914821738","series":"1.2.826.0.1.3680043.8.498.95598720853886926259051375667170114612","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15486463042236628073590077209793828847","series":"1.2.826.0.1.3680043.8.498.92030109539871470580497188123313237762","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15399233309269178192583032682669258885","series":"1.2.826.0.1.3680043.8.498.79812752664946333316235005213307604576","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15630146025588580188363070220779173742","series":"1.2.826.0.1.3680043.8.498.12638340815947156821011822215922579484","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15595595490910196952568459534537629865","series":"1.2.826.0.1.3680043.8.498.97743734282230195911148946870890904383","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15744573360087157139672078937679122840","series":"1.2.826.0.1.3680043.8.498.11234742990571523936612291006495444865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15863325224755490561125835019647303556","series":"1.2.826.0.1.3680043.8.498.16544345985341256132013484669618139566","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15744573360087157139672078937679122840","series":"1.2.826.0.1.3680043.8.498.11341508609893767655069464599987930631","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.15539098084847502447238233198010208924","series":"1.2.826.0.1.3680043.8.498.68107273020870930268497329233186999377","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.16189833935983604051983090678102023832","series":"1.2.826.0.1.3680043.8.498.34915825240271993888031987711670161589","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192","series":"1.2.826.0.1.3680043.8.498.82999681476036202881857678022765229979","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943","series":"1.2.826.0.1.3680043.8.498.11766454514088375278654972147039106579","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192","series":"1.2.826.0.1.3680043.8.498.75266919254774703903096233357689465742","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943","series":"1.2.826.0.1.3680043.8.498.78043287320720592978158748691677617840","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.16318078762749420171833135145622117452","series":"1.2.826.0.1.3680043.8.498.35234113749316087053218419663685116117","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16483470188406187000599644150600279596","series":"1.2.826.0.1.3680043.8.498.32717063457742403860326898610544301721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16534078612126477228267765508485297367","series":"1.2.826.0.1.3680043.8.498.26243198023987984037282459989058511588","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835","series":"1.2.826.0.1.3680043.8.498.10248087719476589017843109082789164432","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16709696133689803501344038926512501779","series":"1.2.826.0.1.3680043.8.498.13363390236927506021792822866837808222","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16534078612126477228267765508485297367","series":"1.2.826.0.1.3680043.8.498.36992640194531095139827034743817098896","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.16666020201961191623151386398626511849","series":"1.2.826.0.1.3680043.8.498.98632081059271591828164635680299864288","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16845451648760595922758166878121325861","series":"1.2.826.0.1.3680043.8.498.48886479838052842673696951759389139315","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16864649493021092944977766326505053153","series":"1.2.826.0.1.3680043.8.498.11150151270793694543000471670755998026","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16812524557513419866531695739014681414","series":"1.2.826.0.1.3680043.8.498.15615904519288998367423966302608190516","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17070139245390396445395414066427036835","series":"1.2.826.0.1.3680043.8.498.11082797874142477458289099228682643158","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16724013341359693140331158148551346369","series":"1.2.826.0.1.3680043.8.498.37449406013095113070569697645165360107","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.16864649493021092944977766326505053153","series":"1.2.826.0.1.3680043.8.498.77587893353987852937710224810902634180","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17107696939614797679012793547913769707","series":"1.2.826.0.1.3680043.8.498.16127201577091531597314566930840651776","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119","series":"1.2.826.0.1.3680043.8.498.79186759514951470066801190764713246607","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17241550277273207658303011467222594165","series":"1.2.826.0.1.3680043.8.498.76159443837356724396339192103965802743","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119","series":"1.2.826.0.1.3680043.8.498.53713765237196658026322945543142111312","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17389120886685633448235711042721669823","series":"1.2.826.0.1.3680043.8.498.11350330383955415028526030673123341678","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17274490471295558668613733667390457509","series":"1.2.826.0.1.3680043.8.498.20305377716969141454548673873550176504","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17420673628956367425739018933968443620","series":"1.2.826.0.1.3680043.8.498.10515248563472553827070311412588159787","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17241550277273207658303011467222594165","series":"1.2.826.0.1.3680043.8.498.56375615879889358609259967031325979401","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17517950899674150719043097786728121608","series":"1.2.826.0.1.3680043.8.498.51964761684309178143134372850968161774","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17768582402712001917772688462080201650","series":"1.2.826.0.1.3680043.8.498.66328090334478063137101654819070723636","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17662880041629379035049367252086125592","series":"1.2.826.0.1.3680043.8.498.50611480059730944056486009629726633792","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17768582402712001917772688462080201650","series":"1.2.826.0.1.3680043.8.498.19404812819230864885303098706606274688","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17596657287050550465463272509976106854","series":"1.2.826.0.1.3680043.8.498.50595036998349381288845152045974640874","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17691941055331862184525411493110144433","series":"1.2.826.0.1.3680043.8.498.89054571801921632637258156983078091295","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711","series":"1.2.826.0.1.3680043.8.498.11261251202884490916489531421908521865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17985756708438762670829959560049370730","series":"1.2.826.0.1.3680043.8.498.81001123812469341354147972738734177063","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18090147030578547920423840630147682826","series":"1.2.826.0.1.3680043.8.498.12943020891195253245429903353410759499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17931021655186901394796168474942485728","series":"1.2.826.0.1.3680043.8.498.10003074389531472338522226501097852466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18113123512696698441392990570166201548","series":"1.2.826.0.1.3680043.8.498.10745143988150473282468918272152675881","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17865613300681542407687000381594067366","series":"1.2.826.0.1.3680043.8.498.12643413171994340255424796706769211717","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18199465156572252531439880765733639584","series":"1.2.826.0.1.3680043.8.498.10298812622325318054519069066426733792","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18245500498843512203508664874652003124","series":"1.2.826.0.1.3680043.8.498.30833481587663821286813888438734074657","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18344563362014614046776037921812502569","series":"1.2.826.0.1.3680043.8.498.41815374979401517806123190969815775193","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18592535981451766048952107991674227570","series":"1.2.826.0.1.3680043.8.498.69606524653319303861014449647495607428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18671006890719350655167441375718348829","series":"1.2.826.0.1.3680043.8.498.45320501572776065377387620144112010826","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18175450083338316799857698898702333009","series":"1.2.826.0.1.3680043.8.498.27663991349576346857906529584056970961","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.18758199166360428760124471581842191566","series":"1.2.826.0.1.3680043.8.498.20879478117965970989038957036168486753","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18403627643718880914843209837672242908","series":"1.2.826.0.1.3680043.8.498.42298905803013548293596982981586411347","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18807952338149288346040972005636698334","series":"1.2.826.0.1.3680043.8.498.15830344183643737530026147823587622074","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18848481788456572234982139368351538716","series":"1.2.826.0.1.3680043.8.498.10687371503063218654909370634022749028","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18968967345606818331567480514437075646","series":"1.2.826.0.1.3680043.8.498.82622707974423831915785815880478114537","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18695727379446620258431748063749176639","series":"1.2.826.0.1.3680043.8.498.75144393291445383484891391817231709242","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.18807952338149288346040972005636698334","series":"1.2.826.0.1.3680043.8.498.43359237872829902347639069091429046406","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.19034825848229751701363461935217290295","series":"1.2.826.0.1.3680043.8.498.83099100581317914207590941885683834094","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19124879088354547836118593585646705198","series":"1.2.826.0.1.3680043.8.498.55703598257738230989484505389159878506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19291375342005405073321673638971145777","series":"1.2.826.0.1.3680043.8.498.25822706776264559233980587028689051936","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19374331384306309057311171252852774841","series":"1.2.826.0.1.3680043.8.498.11583795425245372284693065236891308586","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19516545000256238042886469152307943182","series":"1.2.826.0.1.3680043.8.498.83894398980609129965979863390598300709","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19489875090732464985703624680756212359","series":"1.2.826.0.1.3680043.8.498.35254854011115494145906572961159895859","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19624674714110931897302481276137067638","series":"1.2.826.0.1.3680043.8.498.10634106926603826527096532987600256308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19183011851352129901149443549646899548","series":"1.2.826.0.1.3680043.8.498.10101237742481135035557344187439703337","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.19813267030974842239752491935957206955","series":"1.2.826.0.1.3680043.8.498.99610843995157325466722641496744853850","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19830663206351645385520268848711462246","series":"1.2.826.0.1.3680043.8.498.43239088394215642405340338158380018670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19845711038895023426012115550237130530","series":"1.2.826.0.1.3680043.8.498.28038675817674240765745221444376624604","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19722687746450343001601428638894755031","series":"1.2.826.0.1.3680043.8.498.32164418214373306270279753783932960899","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19925337578871911327225897926650188274","series":"1.2.826.0.1.3680043.8.498.69905618915951856850020075201168200463","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20063306448698185569553477478719533390","series":"1.2.826.0.1.3680043.8.498.12957807429037643884499493892596548170","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20134168618929769962391688049013971830","series":"1.2.826.0.1.3680043.8.498.13030802083996377030875521008757539425","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20196165276643741937915346646793735100","series":"1.2.826.0.1.3680043.8.498.10590644275062294508767253018760514744","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19887850741193535685529812263489892001","series":"1.2.826.0.1.3680043.8.498.78366110185732494999906925021098706057","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20375739629244657719385962474248679850","series":"1.2.826.0.1.3680043.8.498.22013217704457487557735926603260083187","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465","series":"1.2.826.0.1.3680043.8.498.44039841310940439537779360150118988245","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20488337421252584047930242186072775007","series":"1.2.826.0.1.3680043.8.498.80705798358487192889737969810958439032","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20488337421252584047930242186072775007","series":"1.2.826.0.1.3680043.8.498.38732057308647421985491915016016128470","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.20232953944759707220316171133595554284","series":"1.2.826.0.1.3680043.8.498.11363309414439675180176323191754729462","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465","series":"1.2.826.0.1.3680043.8.498.38332139832025121381291298672987640480","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.20587514285427671071934631922521946816","series":"1.2.826.0.1.3680043.8.498.44032355019019578078051122155154587705","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20953536352252551462901259297974956430","series":"1.2.826.0.1.3680043.8.498.10621633968174075869953411885289061506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20733151481921261415295204209910573639","series":"1.2.826.0.1.3680043.8.498.11942113926536909533659029037260679308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20933151653103548000061448393771614270","series":"1.2.826.0.1.3680043.8.498.23635045597451958088492882101934120565","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20801986011906346655458394198951999413","series":"1.2.826.0.1.3680043.8.498.29212683099610864330770071711943239851","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21239630702461934662855021164119114203","series":"1.2.826.0.1.3680043.8.498.10675440579383180515456206609457714791","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21312254534068992054745674608909722454","series":"1.2.826.0.1.3680043.8.498.12169592222988048509907969985357359146","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21135201439483157418374033420156525658","series":"1.2.826.0.1.3680043.8.498.10088722197008341424800966901103293591","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.21012635480218712816069264326295410954","series":"1.2.826.0.1.3680043.8.498.12210838826196346475090170966302604203","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21389336408905524316915532623385847283","series":"1.2.826.0.1.3680043.8.498.67091124691782552281462784500257160092","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21413285047211074156709106352750752275","series":"1.2.826.0.1.3680043.8.498.23726324591928859106092149394325106376","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21584859831565112159503971937497384030","series":"1.2.826.0.1.3680043.8.498.58248619180494662361876037929600896223","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871","series":"1.2.826.0.1.3680043.8.498.12159773988930868854146694845458972150","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21606196343708804451390109412118036330","series":"1.2.826.0.1.3680043.8.498.41685366825233985624102409916297431455","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871","series":"1.2.826.0.1.3680043.8.498.19944191560206457855272502954921085214","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.21636337451527049130190928852193267826","series":"1.2.826.0.1.3680043.8.498.47317638526018857985269804251260538965","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21862609534273524003153225756948913437","series":"1.2.826.0.1.3680043.8.498.11884836226472210399035529235445864726","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21937000442536121262190458446148124910","series":"1.2.826.0.1.3680043.8.498.96450439742561973116177317430227436266","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21937000442536121262190458446148124910","series":"1.2.826.0.1.3680043.8.498.20541770403691678577267364860621717118","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.21874562538764236819894456417694074673","series":"1.2.826.0.1.3680043.8.498.68814524570679968520673929316106451854","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22088734384869383587217556574382319654","series":"1.2.826.0.1.3680043.8.498.57302123765465276236665607684861658747","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22025729757015485203756049440674914537","series":"1.2.826.0.1.3680043.8.498.69243020966477058749918373065264345062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21991252201684723900045301917670383228","series":"1.2.826.0.1.3680043.8.498.72547433735522012013339746856698368878","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22249763791584161324425922920269052844","series":"1.2.826.0.1.3680043.8.498.26568157629512677976650654959351694741","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965","series":"1.2.826.0.1.3680043.8.498.11518347023788527266354862352010423284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965","series":"1.2.826.0.1.3680043.8.498.73770285204847709873243435883019482639","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.22594249996322566888700984133307337128","series":"1.2.826.0.1.3680043.8.498.36300665521074994078164442160997227279","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22407175411826914385667795929161924645","series":"1.2.826.0.1.3680043.8.498.48531891317577362079185282914670007862","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22456525904575663955604077506103354060","series":"1.2.826.0.1.3680043.8.498.93911039105181212909279423580008096076","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22596004572382994883130705870731193046","series":"1.2.826.0.1.3680043.8.498.13232411309655372210540775323491942095","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22632919018724241015271844108954804020","series":"1.2.826.0.1.3680043.8.498.24196107334076803498288469665124595772","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22275537949387641831010030150398063643","series":"1.2.826.0.1.3680043.8.498.96168357149595703576006734166420684850","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22678957404909201599340396039237525550","series":"1.2.826.0.1.3680043.8.498.25311274562753934792730851770540853002","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22803020644232321685575897661362804423","series":"1.2.826.0.1.3680043.8.498.12423083258874298696639022964496927018","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23043120269252616884307662769766701449","series":"1.2.826.0.1.3680043.8.498.84793901856599026181086541233505568686","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23063675880057833741455102729816364091","series":"1.2.826.0.1.3680043.8.498.40914537247644284816573138937906978927","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22985467719058223198642920273233975053","series":"1.2.826.0.1.3680043.8.498.69567259975931509709221756626102710558","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22886251629508830700774506697670139848","series":"1.2.826.0.1.3680043.8.498.96095102561995593180134493479542132372","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.23073694267101759922738121732233014980","series":"1.2.826.0.1.3680043.8.498.12781349281590949585384212688766791229","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23094733099955217022059584418468076048","series":"1.2.826.0.1.3680043.8.498.59755796796418869838454051956985710402","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22985467719058223198642920273233975053","series":"1.2.826.0.1.3680043.8.498.48635335020683727895056146044605234993","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.23409201792139701921731647470152180932","series":"1.2.826.0.1.3680043.8.498.99016392436099887338494530286934459754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23523176367321534630723586470190430254","series":"1.2.826.0.1.3680043.8.498.47346738682912959275839868823084469561","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23686732555936573712275826587678502899","series":"1.2.826.0.1.3680043.8.498.33329196772356826937296787326648963856","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23427198594727947883953289189443006288","series":"1.2.826.0.1.3680043.8.498.10261722270486719227047013261409695867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23733147518259581649562651915997449088","series":"1.2.826.0.1.3680043.8.498.10222138187662296716685243217835676138","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23939313916974849526459717576344063277","series":"1.2.826.0.1.3680043.8.498.11675742812263873810174068316154454848","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23849672490426589132734390202638283865","series":"1.2.826.0.1.3680043.8.498.63102193243767382652350096289896364260","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24267153760943721146187182369810590631","series":"1.2.826.0.1.3680043.8.498.20681459897877022440731006873212098042","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23822313132245448173452915228147497501","series":"1.2.826.0.1.3680043.8.498.32094803877681859554780887271212683064","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24068239310155798723127213793623172530","series":"1.2.826.0.1.3680043.8.498.28052059980725273256124923535511760652","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24313647347273496316728514161733307557","series":"1.2.826.0.1.3680043.8.498.37070949198663775470439128628775461099","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24291465113360632954006568021942978765","series":"1.2.826.0.1.3680043.8.498.99500485107281028636575065459834119705","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.24661137747502154216858169258339731263","series":"1.2.826.0.1.3680043.8.498.95075207181639818778426395476757700754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24588524807799377432185260702152371503","series":"1.2.826.0.1.3680043.8.498.66684786824185545955915013074491513800","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24502332819381293694020055836255028614","series":"1.2.826.0.1.3680043.8.498.87530650957540118335573725943162093868","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24680736086205072406257935876858402593","series":"1.2.826.0.1.3680043.8.498.55350139277152985054131652622537552199","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038","series":"1.2.826.0.1.3680043.8.498.15682388138867287606013641869935336686","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983","series":"1.2.826.0.1.3680043.8.498.70421260894040790671949910132471710773","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24826041178550854190310664677274404684","series":"1.2.826.0.1.3680043.8.498.59908021763865724301573574942804650740","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983","series":"1.2.826.0.1.3680043.8.498.28276525488167036824932862380860225736","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.25010661650105533302140829168731585935","series":"1.2.826.0.1.3680043.8.498.26422782538212168973224039348895059691","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25103708168569906591706406794385334861","series":"1.2.826.0.1.3680043.8.498.29757090984929419949146881558446820857","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25252553477605568387898480443616118724","series":"1.2.826.0.1.3680043.8.498.10070738928814618247259166613683629125","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25266231062774974303723359806535863810","series":"1.2.826.0.1.3680043.8.498.12863631741719721083855811519874625449","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25359770839263995354733835365510047109","series":"1.2.826.0.1.3680043.8.498.10891962874374730014804867587369713595","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25145588904873072866960435474159832047","series":"1.2.826.0.1.3680043.8.498.94145392694842484435104970385732489765","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25431207706481141354761246438973384670","series":"1.2.826.0.1.3680043.8.498.22383898251093197220584303756235552047","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25616615589791001678067099907823838797","series":"1.2.826.0.1.3680043.8.498.56260707343451606348681430979175583118","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25663557443141366332431300959089922567","series":"1.2.826.0.1.3680043.8.498.11198993717636973438753007486704457875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25480883220477368504575353972328124891","series":"1.2.826.0.1.3680043.8.498.20326306144251834051574362839496883854","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25480883220477368504575353972328124891","series":"1.2.826.0.1.3680043.8.498.47015694008705117816192116911552403228","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487","series":"1.2.826.0.1.3680043.8.498.10431359907759302112904561594331019770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25765711614507755527825654507920391096","series":"1.2.826.0.1.3680043.8.498.32301181256639371325684258201297067957","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25697539332996211770844878649053316629","series":"1.2.826.0.1.3680043.8.498.34526159429912223542060883648741564643","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25815857653380672532983223088817283976","series":"1.2.826.0.1.3680043.8.498.11142741420634722418843176875718366964","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487","series":"1.2.826.0.1.3680043.8.498.34260834550457679073635287817513014352","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.25904473139226718487560978103077541725","series":"1.2.826.0.1.3680043.8.498.33258788924077222303652041119370307607","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25928779639851562853500186715690156268","series":"1.2.826.0.1.3680043.8.498.10194484310009277513385604468657453298","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26004572902429209855865822977387283547","series":"1.2.826.0.1.3680043.8.498.16445928890789414256607241654946184497","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25977532612504997807641730439567378227","series":"1.2.826.0.1.3680043.8.498.10683901231951245862082511437641140449","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26023120405608635089544039694812003604","series":"1.2.826.0.1.3680043.8.498.79779651997225151907548873078335295861","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25856396695552621816673203639618612020","series":"1.2.826.0.1.3680043.8.498.51693813682292308736611044319384266177","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26072622476167889001255150030772906081","series":"1.2.826.0.1.3680043.8.498.67814767012570828693756172301396776830","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25766673238950703352855821505938941878","series":"1.2.826.0.1.3680043.8.498.10881662665068917651675360868198721819","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26373657290121957625263322898744712730","series":"1.2.826.0.1.3680043.8.498.34569269094198269865160385324493649357","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26431974442245797095978351976773354669","series":"1.2.826.0.1.3680043.8.498.11012848063166174188089485397679094837","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26266119962108543565575361589492142091","series":"1.2.826.0.1.3680043.8.498.97551944790896343748584722955899009912","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26485916396240515006042297316506505324","series":"1.2.826.0.1.3680043.8.498.11541503034413153412715510361253358130","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26536992666456596813876340849399448897","series":"1.2.826.0.1.3680043.8.498.11087053584217547712226323035250837652","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26431974442245797095978351976773354669","series":"1.2.826.0.1.3680043.8.498.40683889381819586665217492814108147867","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26201626427712146573573952548179938304","series":"1.2.826.0.1.3680043.8.498.85978827534777287764049173132048893274","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26536992666456596813876340849399448897","series":"1.2.826.0.1.3680043.8.498.34406775285617319479872630245083692371","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26698126402727061400851465377574886112","series":"1.2.826.0.1.3680043.8.498.13373872543997584166033689392932893516","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26842258746390750314059176647398369242","series":"1.2.826.0.1.3680043.8.498.51878073793849258679080519971762533948","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26880562799522930458745136318439622601","series":"1.2.826.0.1.3680043.8.498.86691612756565808808636503474651517998","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26619973716086136503589916868066348957","series":"1.2.826.0.1.3680043.8.498.64500545578767086860018520484288614195","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27020884753639915245324093232218617967","series":"1.2.826.0.1.3680043.8.498.60655339980250867789624964257952928666","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27025989808002468335294585261291982661","series":"1.2.826.0.1.3680043.8.498.64065389598762387303480867295493288502","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26978027400525885904983020900996228004","series":"1.2.826.0.1.3680043.8.498.72580786429084863941402332620616945441","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27161418589407291713644384596541786246","series":"1.2.826.0.1.3680043.8.498.78700444344419223048745612092665344515","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27161418589407291713644384596541786246","series":"1.2.826.0.1.3680043.8.498.24464822604368868885944643725135678972","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27192678730060203972082511503480566620","series":"1.2.826.0.1.3680043.8.498.50983458669832053488744331263235119104","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27192678730060203972082511503480566620","series":"1.2.826.0.1.3680043.8.498.13239156871442516367090172880688779977","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27224327142578332905227939015479767285","series":"1.2.826.0.1.3680043.8.498.17571517783315627802511219237369999830","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27319908980978184820438244981035567211","series":"1.2.826.0.1.3680043.8.498.81756564566876084705240449374884767561","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27224327142578332905227939015479767285","series":"1.2.826.0.1.3680043.8.498.40929553596323036867666332656445327531","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27339999385861467858422021400023533486","series":"1.2.826.0.1.3680043.8.498.23253025209189662371265024877116033468","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27456622705361831116491505615162380604","series":"1.2.826.0.1.3680043.8.498.41457892961284245673169390278604488595","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27653320597564308144389724321115208375","series":"1.2.826.0.1.3680043.8.498.11229247696285308722909597631050020097","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27627190971637735537007595480396264063","series":"1.2.826.0.1.3680043.8.498.11367835684275642321335302251160883505","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27319908980978184820438244981035567211","series":"1.2.826.0.1.3680043.8.498.12715211551947485275927848748971469433","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27656830051511853680197276412365020780","series":"1.2.826.0.1.3680043.8.498.36863650637382162014788829203970778683","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924","series":"1.2.826.0.1.3680043.8.498.42472038773573179227871679105768097718","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721","series":"1.2.826.0.1.3680043.8.498.12994612592612102757879623625750784814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27474440682441854303485403899867525864","series":"1.2.826.0.1.3680043.8.498.17004059915837135670226080262204876420","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721","series":"1.2.826.0.1.3680043.8.498.22204007224362001929442255779127976641","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351","series":"1.2.826.0.1.3680043.8.498.19029778683081496978920507239333155008","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351","series":"1.2.826.0.1.3680043.8.498.11375401144507611025734098969206135440","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27796604203156439069742212039368371002","series":"1.2.826.0.1.3680043.8.498.16690436884198907202593933430098533310","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27890860883826681731748268459485259935","series":"1.2.826.0.1.3680043.8.498.45813213851002737858434908177590046316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924","series":"1.2.826.0.1.3680043.8.498.10430108675976725353539977125906268009","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27925145307825795804342879375835424449","series":"1.2.826.0.1.3680043.8.498.36626459220617142964026031217168287330","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28045606887691550130031344985830975325","series":"1.2.826.0.1.3680043.8.498.48080982933567067752071146881843478310","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28134711422142706763867845558971485241","series":"1.2.826.0.1.3680043.8.498.34489201244882064911139549609495894421","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27946605663086243142356841956045514227","series":"1.2.826.0.1.3680043.8.498.12473518633714511441790738005244133734","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28011057282528460306257420499560926354","series":"1.2.826.0.1.3680043.8.498.83743655849565367650519982106676963410","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795","series":"1.2.826.0.1.3680043.8.498.94741796065001006160244007433885481283","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795","series":"1.2.826.0.1.3680043.8.498.10693624851940967543683689560480179102","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.28289838818003822677994912735532050038","series":"1.2.826.0.1.3680043.8.498.21588904273746116982175558201078733308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28309626953500461628112343997868050028","series":"1.2.826.0.1.3680043.8.498.11239762079815561645892732387757510115","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28488216314048990109149780234417422948","series":"1.2.826.0.1.3680043.8.498.10813360787614236517394609143558721500","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.28425280718252632810092859612928047098","series":"1.2.826.0.1.3680043.8.498.11710204764388899404815734499128270284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28351884945918062351436698411322252525","series":"1.2.826.0.1.3680043.8.498.10637579596771376929524912301212076402","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28488216314048990109149780234417422948","series":"1.2.826.0.1.3680043.8.498.80843077291347862224714423628942124958","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28595729988867300278425411189173775382","series":"1.2.826.0.1.3680043.8.498.95964013576352506407269531192760186639","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28628472249424790112957421215774036156","series":"1.2.826.0.1.3680043.8.498.96568345107544290465944673938523009150","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28638490997780173004029312240334483157","series":"1.2.826.0.1.3680043.8.498.96809673382535983344686466371345362763","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28779021651975021236045277212517629154","series":"1.2.826.0.1.3680043.8.498.72301646335633555659554148534487195336","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28746954364434312909368235768936980607","series":"1.2.826.0.1.3680043.8.498.58202947869066069382410295858502974930","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608","series":"1.2.826.0.1.3680043.8.498.11438210485488424020029704142703549114","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28873180831637763124652074775631752918","series":"1.2.826.0.1.3680043.8.498.40092580142341416105733323831504439656","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28539130184243787339297053654373783542","series":"1.2.826.0.1.3680043.8.498.90901496223099642463487949009913592769","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.28686498221359515429184385385077600687","series":"1.2.826.0.1.3680043.8.498.99133438631356576124046130048556136901","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.29227028592609694700827329867343028944","series":"1.2.826.0.1.3680043.8.498.10170545250088782952611958629261159931","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29364640434226980751405780979697632001","series":"1.2.826.0.1.3680043.8.498.28853278261544404817675802353463547199","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28954361146470748868500417103935640742","series":"1.2.826.0.1.3680043.8.498.74344212998010426323644806229859251189","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29375052467745327893128894062568944496","series":"1.2.826.0.1.3680043.8.498.12705522794762501087995976767134608589","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29364640434226980751405780979697632001","series":"1.2.826.0.1.3680043.8.498.29020082251654848193638272552653467980","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.29473278891551292295080887778714088878","series":"1.2.826.0.1.3680043.8.498.60469371069242846744039037659375675190","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29375052467745327893128894062568944496","series":"1.2.826.0.1.3680043.8.498.89476881159227843391524015370517386418","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.29418130858182890216867847991592582370","series":"1.2.826.0.1.3680043.8.498.65385355809992998610281138370778499348","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29758331841591711520514486913006094054","series":"1.2.826.0.1.3680043.8.498.43011631140094154635984244419508656891","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29739171278960083313048266628194182569","series":"1.2.826.0.1.3680043.8.498.61165994969752684769936158555993773868","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29579560656789178333629609998252952856","series":"1.2.826.0.1.3680043.8.498.87655002688607398104414949254891574237","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922","series":"1.2.826.0.1.3680043.8.498.12153095665930544560501292120515504483","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29866827083314645739444078187995496049","series":"1.2.826.0.1.3680043.8.498.49182001084882911047213797117242618411","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922","series":"1.2.826.0.1.3680043.8.498.18893444986254332630190214700602412073","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30174064884443577177838026250637902388","series":"1.2.826.0.1.3680043.8.498.58445936259190390291885179531198837880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29794301133358685635777478470865282350","series":"1.2.826.0.1.3680043.8.498.63435793266716751389828885446589098487","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911","series":"1.2.826.0.1.3680043.8.498.75078392068446053207082784619084087737","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30222189977108980260768952691472020722","series":"1.2.826.0.1.3680043.8.498.12279501514754075749189809797794868260","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30381005430261460472880465705496199540","series":"1.2.826.0.1.3680043.8.498.17658861121119089844526401412496983852","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762","series":"1.2.826.0.1.3680043.8.498.27556438001949963501805247955142999375","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989","series":"1.2.826.0.1.3680043.8.498.10611618947763489886507677940119186847","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30252501257451878373370333329266264057","series":"1.2.826.0.1.3680043.8.498.12819990572695764639571555444272449369","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30520444048368652235940346583890702451","series":"1.2.826.0.1.3680043.8.498.48802388941925394284370677328106114375","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989","series":"1.2.826.0.1.3680043.8.498.44026285139411551036922554108283279057","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30662407196303880392459287553277761223","series":"1.2.826.0.1.3680043.8.498.52537238122873942697674876707778513126","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30673411238679998084995204090753297949","series":"1.2.826.0.1.3680043.8.498.11417693418637809272983496139532620507","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30520444048368652235940346583890702451","series":"1.2.826.0.1.3680043.8.498.35122246082562707132968400320779162866","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30557810085867943803354292590079995622","series":"1.2.826.0.1.3680043.8.498.20818516745343010089236838934182058662","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30746167840159843733412534591988948270","series":"1.2.826.0.1.3680043.8.498.33231707118602307177817721945696602758","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30763696628968291934469221453461209579","series":"1.2.826.0.1.3680043.8.498.75393325317316365528253444802895350897","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30838586189496669675465778955893367497","series":"1.2.826.0.1.3680043.8.498.84728877396463898626420862246288300451","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30763696628968291934469221453461209579","series":"1.2.826.0.1.3680043.8.498.63848465555926722566325755462258488378","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30947630416867960170317316514422933066","series":"1.2.826.0.1.3680043.8.498.11823130856919630622164172969779714622","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31080452089421812812019413302222437434","series":"1.2.826.0.1.3680043.8.498.22617645152631299626024454472364257544","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30832771419882354941430609179589560374","series":"1.2.826.0.1.3680043.8.498.52098575488812247212094796944434462982","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30888379005655662168586864728687777425","series":"1.2.826.0.1.3680043.8.498.62049710655335968997316093679389232040","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.31175171249406459042393597291141275669","series":"1.2.826.0.1.3680043.8.498.10144475926277948957560313410381417146","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31099878224304681441631176766255029271","series":"1.2.826.0.1.3680043.8.498.11881835185926960237555976926976118104","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30998941785989496591079894962766360976","series":"1.2.826.0.1.3680043.8.498.10722130763966226643727120662626974772","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090","series":"1.2.826.0.1.3680043.8.498.12919496990401970668406752551441415420","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31280487369922340170695453174997912050","series":"1.2.826.0.1.3680043.8.498.95880070161932590048171686176102051703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090","series":"1.2.826.0.1.3680043.8.498.11527291024991421518430070398708707006","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.31745644522310786426860265012373331979","series":"1.2.826.0.1.3680043.8.498.30443546411966746904073412057311110618","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31193772347652083986607183406944630716","series":"1.2.826.0.1.3680043.8.498.21955096597342108247751465199911811390","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.31373064397225665483187291274194812486","series":"1.2.826.0.1.3680043.8.498.31764207635008544220862712568103464418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31555333355990829772911515362502000209","series":"1.2.826.0.1.3680043.8.498.11604237841435081697274002640287867539","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31827987476038668074615045841102711882","series":"1.2.826.0.1.3680043.8.498.10087867987021022925083019994606229192","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31851627014639252866084081561126592250","series":"1.2.826.0.1.3680043.8.498.13285004461442883381456163818511324524","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32043012602421532910420313983803503778","series":"1.2.826.0.1.3680043.8.498.81828628697254427213831470696864960260","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31899121807187535405063131351086765296","series":"1.2.826.0.1.3680043.8.498.74207662530741940110019193222188952100","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32226253011052584155674401943276961907","series":"1.2.826.0.1.3680043.8.498.10317728914742475483658015459425868033","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32172672595181417103935593872811182816","series":"1.2.826.0.1.3680043.8.498.28029026915035454501271902854630607244","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32329374180424300100441619205766536205","series":"1.2.826.0.1.3680043.8.498.11162392691058125540854137224771917967","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945","series":"1.2.826.0.1.3680043.8.498.81084171603553369816257504401663206753","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32265622493980668732317541941196894795","series":"1.2.826.0.1.3680043.8.498.62527780329511441173933929856226618507","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32521102053110469817073126425955562934","series":"1.2.826.0.1.3680043.8.498.98802492489268048026454604754427719933","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32109377199577933060124357239881959602","series":"1.2.826.0.1.3680043.8.498.48862022446753678690722306948284116495","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664","series":"1.2.826.0.1.3680043.8.498.13362193928962445360197527975426180601","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461","series":"1.2.826.0.1.3680043.8.498.51805278630137177032424656421198512504","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32504785212530110880087745488871181111","series":"1.2.826.0.1.3680043.8.498.12287634077080248808200810596779473345","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.32417506901165787885027994386927321654","series":"1.2.826.0.1.3680043.8.498.11467473148378400081600010197605394037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32835080036019877751472792740274297358","series":"1.2.826.0.1.3680043.8.498.43907976029565138410241950183282713097","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32910372391370203605482772577821971133","series":"1.2.826.0.1.3680043.8.498.88561260305368825310438766403707101403","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32996271790302034744448837505729856569","series":"1.2.826.0.1.3680043.8.498.10302982703653341408044392338431306652","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461","series":"1.2.826.0.1.3680043.8.498.26756759643520026517849937163874615115","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33085434860684110795966817803985794300","series":"1.2.826.0.1.3680043.8.498.11113069029145600227507412228423573354","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33324606756355240783076971858231422510","series":"1.2.826.0.1.3680043.8.498.37179110279453919873822823278880900150","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071","series":"1.2.826.0.1.3680043.8.498.23209448596594632671652497349254865301","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071","series":"1.2.826.0.1.3680043.8.498.12621145057217394629541457313774661072","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443","series":"1.2.826.0.1.3680043.8.498.13387836131520411057370415743491815035","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33324606756355240783076971858231422510","series":"1.2.826.0.1.3680043.8.498.91746505878781150856808389543505527532","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33273874534046658504935877940537027702","series":"1.2.826.0.1.3680043.8.498.86660623961749410755729062401011621267","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33427080769740419588952225424281396684","series":"1.2.826.0.1.3680043.8.498.61981342699629821686124780528647413880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33393833442961030521286257650328223086","series":"1.2.826.0.1.3680043.8.498.42731359218158626502001838354491392349","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33709759248055631658322056320849494088","series":"1.2.826.0.1.3680043.8.498.35976811383765806907456122153209654308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33479806419966566464727310013949313521","series":"1.2.826.0.1.3680043.8.498.99929355029926997192734120610018004035","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33834227140201599591317512187502720859","series":"1.2.826.0.1.3680043.8.498.10841805699218611762079818273077113637","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33880013318702285028756482543453984598","series":"1.2.826.0.1.3680043.8.498.15911466032527485071308994755174971570","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33614319611050071354692274770073371314","series":"1.2.826.0.1.3680043.8.498.60616819542595947238291371983206368840","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33657688895130083119886800524324572941","series":"1.2.826.0.1.3680043.8.498.12883999866503742498230235501754494715","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33899056854391617633929258302459140830","series":"1.2.826.0.1.3680043.8.498.47079586257669333957913235534568933698","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33957165685646407808982253731894148028","series":"1.2.826.0.1.3680043.8.498.53103134093058753781757240214643488836","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34076627216941578548083498328399240999","series":"1.2.826.0.1.3680043.8.498.13352819128040464915458464463537668510","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34206043272866138622475050809785944941","series":"1.2.826.0.1.3680043.8.498.66805076343767847361789966344873057335","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34076627216941578548083498328399240999","series":"1.2.826.0.1.3680043.8.498.28252437031393470549191037048332786822","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745","series":"1.2.826.0.1.3680043.8.498.11551454000378799855667969590315637616","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34379812274299658721362253287768041613","series":"1.2.826.0.1.3680043.8.498.39321650168095989597368765336620087186","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34109091435857467600499949506940613037","series":"1.2.826.0.1.3680043.8.498.50137498973842924730145522944480524765","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34461302710296907289543656738174044465","series":"1.2.826.0.1.3680043.8.498.30384101658003999716759869077284095656","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34433343440016591348979084714300062954","series":"1.2.826.0.1.3680043.8.498.62340569231437070237096313069605054666","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34461302710296907289543656738174044465","series":"1.2.826.0.1.3680043.8.498.10300349669508922546329856632372275862","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34546106114963066554367298449299214677","series":"1.2.826.0.1.3680043.8.498.11431889471052410381932318693825860122","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34654275504962399799861007974465776328","series":"1.2.826.0.1.3680043.8.498.65060460847849920403483685281776368767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542","series":"1.2.826.0.1.3680043.8.498.24646993722416206798391255467240811420","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34690162612328981644511154212357885455","series":"1.2.826.0.1.3680043.8.498.10367162593660233997239753176698867967","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34654275504962399799861007974465776328","series":"1.2.826.0.1.3680043.8.498.52937311016280536011972932945867384026","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34740783941206992242367609204835771599","series":"1.2.826.0.1.3680043.8.498.10697325614403149079670323973202708032","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542","series":"1.2.826.0.1.3680043.8.498.11611209382647481793503209866576787588","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34791780913322886484171421115861604853","series":"1.2.826.0.1.3680043.8.498.13026261312571984114129508754731641941","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34861721301001052975626513770908147451","series":"1.2.826.0.1.3680043.8.498.21435673481175530020427614468186976199","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34791780913322886484171421115861604853","series":"1.2.826.0.1.3680043.8.498.46726962731665696181756166670054400185","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35015675197378142698488944856288804597","series":"1.2.826.0.1.3680043.8.498.13079139454873403083239070637129019769","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35026052487843947230089350615730580521","series":"1.2.826.0.1.3680043.8.498.40268286263640491282726968405310939909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35026052487843947230089350615730580521","series":"1.2.826.0.1.3680043.8.498.60629926680822400533897165818323113312","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35121262802959257590413582180924083425","series":"1.2.826.0.1.3680043.8.498.79209940972336397773509749834772154290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35210451107308788915675910467662426761","series":"1.2.826.0.1.3680043.8.498.19913218922439257245986953710565831795","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35257158249961136373696255071228322165","series":"1.2.826.0.1.3680043.8.498.10151175395030773776410110426248004517","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35064504053970754752690367203246822139","series":"1.2.826.0.1.3680043.8.498.67912360974638448125259602914413521926","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35330802038595377723999341992570259909","series":"1.2.826.0.1.3680043.8.498.10284170232040357709690232967198339759","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35015675197378142698488944856288804597","series":"1.2.826.0.1.3680043.8.498.26190161195183929804521180373336032297","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35549628666222168741574331855838621990","series":"1.2.826.0.1.3680043.8.498.13127904396756695943180814076874936581","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35330802038595377723999341992570259909","series":"1.2.826.0.1.3680043.8.498.12494801806538052677947963008136129205","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35664741425496814074030858561751287211","series":"1.2.826.0.1.3680043.8.498.20102003258506404796308076440177281499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35633121799529901420381085923067525212","series":"1.2.826.0.1.3680043.8.498.86630329079425427984807052759302248720","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35549628666222168741574331855838621990","series":"1.2.826.0.1.3680043.8.498.13205897929859487014781305918501057776","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35664741425496814074030858561751287211","series":"1.2.826.0.1.3680043.8.498.13248300747862274862000577323022935809","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35777075794856195065665011443485613681","series":"1.2.826.0.1.3680043.8.498.48553145476506237522507906012612308953","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35835033603115368194824573270370351403","series":"1.2.826.0.1.3680043.8.498.62498808279136811636961669438431179963","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720","series":"1.2.826.0.1.3680043.8.498.12453746892583420475527008103932256335","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720","series":"1.2.826.0.1.3680043.8.498.22291853273059584228244582877183628082","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35717294165699019392845830077937768611","series":"1.2.826.0.1.3680043.8.498.13131035763326979335962824953325448927","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35892665011493087075493127252825546314","series":"1.2.826.0.1.3680043.8.498.86471144737866460329054427352932157832","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923146679164443588381195543533271939","series":"1.2.826.0.1.3680043.8.498.52098266318294833015149541557239193479","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35853404993418331599021266761280794201","series":"1.2.826.0.1.3680043.8.498.45906930536658758912293363502243541288","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35942316720546803304976641081186518384","series":"1.2.826.0.1.3680043.8.498.99152339365490494633794166643034588134","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923753382244056317223397206343376734","series":"1.2.826.0.1.3680043.8.498.12861766882487493186021415178535239036","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36090989845682961399757203739039806565","series":"1.2.826.0.1.3680043.8.498.71440054374202727041480230214267282997","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36158956431673597472934120620368656246","series":"1.2.826.0.1.3680043.8.498.78449084500782587145810298446977821554","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36221680997794815863088530960610069278","series":"1.2.826.0.1.3680043.8.498.48220433806595051377220673578358704290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35942316720546803304976641081186518384","series":"1.2.826.0.1.3680043.8.498.29030680715650317729254047387690582341","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36293583160636884934622362743062825202","series":"1.2.826.0.1.3680043.8.498.10194490009016522432775872277833185247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36221680997794815863088530960610069278","series":"1.2.826.0.1.3680043.8.498.12257028330298148888976279402376323759","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36293583160636884934622362743062825202","series":"1.2.826.0.1.3680043.8.498.59484989975387876296577388802720297194","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36470828078108352859868482351276699300","series":"1.2.826.0.1.3680043.8.498.21073491679441834876351531642056546450","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36648377536792309029975432419163533533","series":"1.2.826.0.1.3680043.8.498.12837935739917159712035257892208218631","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36521162836904095186654423648490092035","series":"1.2.826.0.1.3680043.8.498.10826022044822518210876638314024787783","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36470828078108352859868482351276699300","series":"1.2.826.0.1.3680043.8.498.67192855711597648369142475353275528690","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36648377536792309029975432419163533533","series":"1.2.826.0.1.3680043.8.498.64080185336643312972931785071033706023","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36891467514776207809384804066253854193","series":"1.2.826.0.1.3680043.8.498.10515654514775180323274473927670327875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36757476214794170118868510630678488185","series":"1.2.826.0.1.3680043.8.498.55549672144185995632224345824066384192","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36757476214794170118868510630678488185","series":"1.2.826.0.1.3680043.8.498.12172964188106376339752363126935670613","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36995206854684735704099881847485073991","series":"1.2.826.0.1.3680043.8.498.30255464995802375700835393032706878605","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36688959138536319644194458211515765203","series":"1.2.826.0.1.3680043.8.498.52288261927805100961829875311057749764","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.37057548998853865829402084168010676727","series":"1.2.826.0.1.3680043.8.498.45315597525118513085678448335589591904","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37147839676713817130306478240352601535","series":"1.2.826.0.1.3680043.8.498.50647735796453585269462405188559610283","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37332768072749723117839591872457536315","series":"1.2.826.0.1.3680043.8.498.13027009759689039364751272858104344083","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37147839676713817130306478240352601535","series":"1.2.826.0.1.3680043.8.498.84350927740255633749568334887759860909","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683","series":"1.2.826.0.1.3680043.8.498.43759038523990610121401841788711871477","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37427993203943111265113924891058049338","series":"1.2.826.0.1.3680043.8.498.86553169044556705797084140530436430628","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37519350121906786941281770770693979271","series":"1.2.826.0.1.3680043.8.498.19197230525517498392923916609236675088","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37508036912473928281276902883575142611","series":"1.2.826.0.1.3680043.8.498.55133612857752858610577653955500061182","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37371365833775442014007234936409739360","series":"1.2.826.0.1.3680043.8.498.88685392398763004257871586836415489482","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.37463970179664003092832463732043124908","series":"1.2.826.0.1.3680043.8.498.95183130408974467926726419286583035340","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37591186611460576939009950935820113903","series":"1.2.826.0.1.3680043.8.498.20293459293337315733171566852773780812","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37591186611460576939009950935820113903","series":"1.2.826.0.1.3680043.8.498.11329805913101523892431962684400122138","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.37572710131396388050286115883363501023","series":"1.2.826.0.1.3680043.8.498.28925465731473425673301837129782358696","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37633035024065331400434438202028794597","series":"1.2.826.0.1.3680043.8.498.80980942647345992049813780515725818794","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37729566534337521443213426693965453255","series":"1.2.826.0.1.3680043.8.498.13090249907355796199513887560916584250","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37729566534337521443213426693965453255","series":"1.2.826.0.1.3680043.8.498.12957847083868470102811350999007750093","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.37833587429731221455928642963031995680","series":"1.2.826.0.1.3680043.8.498.44371997195817413982467477951770244117","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37846940236952007502147453013489359606","series":"1.2.826.0.1.3680043.8.498.89561984729430965399670803292961875837","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37921127120678939078868474340027123335","series":"1.2.826.0.1.3680043.8.498.40989213970605997221312954531190601747","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37846940236952007502147453013489359606","series":"1.2.826.0.1.3680043.8.498.12636596976608716593982287715619359699","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.38069035744880456154810472748123499221","series":"1.2.826.0.1.3680043.8.498.37257449146749863634073393992932768285","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38272345519163551901865251730721530169","series":"1.2.826.0.1.3680043.8.498.10467656696119828251048737256295640263","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38233845434092134418342290639986261345","series":"1.2.826.0.1.3680043.8.498.18916587549319171583244754760178104574","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38303307720827621886420004896097661262","series":"1.2.826.0.1.3680043.8.498.12078866720450613335707816258980606593","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38373204312077453422262347769111605003","series":"1.2.826.0.1.3680043.8.498.10461090507523938547581889269479032612","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38325494489236992123219618004392575979","series":"1.2.826.0.1.3680043.8.498.34458013667678984862347585998167554361","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38361762550587464310603057450403132552","series":"1.2.826.0.1.3680043.8.498.69915263316256545126962163675379858146","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38373204312077453422262347769111605003","series":"1.2.826.0.1.3680043.8.498.78420932581348415752928030840662649246","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.38528511947818361078957998546327084024","series":"1.2.826.0.1.3680043.8.498.51904981937493121832107983110645288297","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38427680930662301114332563903957471851","series":"1.2.826.0.1.3680043.8.498.37391485278543206690377671600104314540","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38485168845116671268554752030254772359","series":"1.2.826.0.1.3680043.8.498.61061785190029658981331842669148186481","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38485168845116671268554752030254772359","series":"1.2.826.0.1.3680043.8.498.65247768487922537957265841250892704409","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.38655145317957160440692625317823254736","series":"1.2.826.0.1.3680043.8.498.58191595288441834384948068226260200277","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38591915802992557518846862828618375715","series":"1.2.826.0.1.3680043.8.498.70068550396991489145269382748433924031","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38617606118499305781729905815407382451","series":"1.2.826.0.1.3680043.8.498.58748294474703368686796496941523650493","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.38750761816114659458563557493234655604","series":"1.2.826.0.1.3680043.8.498.93675543862357330699121890350351418184","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38844149478205202385909887748058004820","series":"1.2.826.0.1.3680043.8.498.76717884054436998308622331137493746493","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38567342184301320850258918666303107813","series":"1.2.826.0.1.3680043.8.498.14731208371591629149586494005888074665","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38795933423959135237935740989825843893","series":"1.2.826.0.1.3680043.8.498.12014289839998860304003111799350846066","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38898732985465285166803609252960353405","series":"1.2.826.0.1.3680043.8.498.13019574564676419783518800136058434014","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38981318408220047366318316620365040799","series":"1.2.826.0.1.3680043.8.498.77112196631835587983185505979931897675","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38928323954764175259962816410233336718","series":"1.2.826.0.1.3680043.8.498.38401507330085691510769885694159694427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39020232951771672617852805188845504161","series":"1.2.826.0.1.3680043.8.498.54231242265283192098203073478271464003","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38912715505731269998129663552068223373","series":"1.2.826.0.1.3680043.8.498.10267205694072647155177732409599139499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38870467957974446108337293912930572060","series":"1.2.826.0.1.3680043.8.498.42514228267696892333304363983816094403","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.39190090804401525573012570218955054016","series":"1.2.826.0.1.3680043.8.498.56951807221431865845625895727886096174","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39064406065534097226924475518005522427","series":"1.2.826.0.1.3680043.8.498.11312757813418473346163359511013609768","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39275838957651450909114440721012584183","series":"1.2.826.0.1.3680043.8.498.51359341937435171167910228036245217049","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39070169128115370701189151138307035053","series":"1.2.826.0.1.3680043.8.498.58542643826756717410079464478563773719","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39375796600811714870169173423682043709","series":"1.2.826.0.1.3680043.8.498.15594001162536018135542477247821207272","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39250844287482885227700969308333354206","series":"1.2.826.0.1.3680043.8.498.42438160585196023644048824512994800776","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39399668753650041254826892657562722764","series":"1.2.826.0.1.3680043.8.498.24743741364794692522821760652709663036","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500","series":"1.2.826.0.1.3680043.8.498.48864168281440917690854528188298055620","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39642276506855996181597769215091644995","series":"1.2.826.0.1.3680043.8.498.12727917933149553212258642779741081033","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611","series":"1.2.826.0.1.3680043.8.498.12027773600357564116111019614467338762","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611","series":"1.2.826.0.1.3680043.8.498.12242058484533296580820974319390000631","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.39547752108509751520306190293259313911","series":"1.2.826.0.1.3680043.8.498.12095294204842532984053650863296581178","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917","series":"1.2.826.0.1.3680043.8.498.95123370568078933068598148297101944590","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917","series":"1.2.826.0.1.3680043.8.498.29174522500069339662546892377483550632","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.39875636232483792306109110284167439592","series":"1.2.826.0.1.3680043.8.498.85343616597492362090427915275961040809","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39932424305167611414527482641898907759","series":"1.2.826.0.1.3680043.8.498.11796644959424495274832770345937138364","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39959632509842689058626880477779808473","series":"1.2.826.0.1.3680043.8.498.73790099724267285099858362938023525773","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40036526071307024601256787605725348112","series":"1.2.826.0.1.3680043.8.498.12311329262116919401988858008972147460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39932424305167611414527482641898907759","series":"1.2.826.0.1.3680043.8.498.10665320886287104594024035177097169850","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40223239401390462911823193489179244754","series":"1.2.826.0.1.3680043.8.498.25699036223051184901506138676410521298","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40086005762552976451418579375198597540","series":"1.2.826.0.1.3680043.8.498.22043086626821401470667205457654082073","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40254123984175499564145728084464192446","series":"1.2.826.0.1.3680043.8.498.12620105724267803874725621269102668036","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40223239401390462911823193489179244754","series":"1.2.826.0.1.3680043.8.498.22391554814811908695246991616586331261","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818","series":"1.2.826.0.1.3680043.8.498.13187353452718568819540507069482635419","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40254123984175499564145728084464192446","series":"1.2.826.0.1.3680043.8.498.16630648036487765071129101314165034223","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818","series":"1.2.826.0.1.3680043.8.498.10450593028127771881719976733382103216","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40317527233615709136328116805254969837","series":"1.2.826.0.1.3680043.8.498.91525498300552939027444889121542605409","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40441879276927383122480339472406105572","series":"1.2.826.0.1.3680043.8.498.27294174211356776050861559456065656121","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40533893516114346078820881178450524201","series":"1.2.826.0.1.3680043.8.498.10662793534753485076077010396621042547","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40576954651310602807432132614823906315","series":"1.2.826.0.1.3680043.8.498.21303084861773297057861017549695063719","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40627043074325458726412346974457616938","series":"1.2.826.0.1.3680043.8.498.12138397179588503855766490119442272781","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40703974473944295875248011843170371394","series":"1.2.826.0.1.3680043.8.498.10074907724122468767765060075160612194","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40703974473944295875248011843170371394","series":"1.2.826.0.1.3680043.8.498.10353507183645747509706957254539263059","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40627043074325458726412346974457616938","series":"1.2.826.0.1.3680043.8.498.99597812831959900025117464688036602654","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40823842819404259265489417470859248075","series":"1.2.826.0.1.3680043.8.498.73296859407051378347799753820705446477","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40837296112545776586904280088902000849","series":"1.2.826.0.1.3680043.8.498.53941661154071494000755299226727514439","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40910368031948275476880475208016058754","series":"1.2.826.0.1.3680043.8.498.98851469407755075764228692225929253557","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40723835064354784133289444204866484476","series":"1.2.826.0.1.3680043.8.498.72420586450147923490112998045713805936","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40994616246264625946610553907227975893","series":"1.2.826.0.1.3680043.8.498.88052307116668483076272506647589925461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41326575785634778198143222323166756210","series":"1.2.826.0.1.3680043.8.498.11168811957095003572885516032087944829","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41124244151692842048993594448992046260","series":"1.2.826.0.1.3680043.8.498.57070135958020527642815178799773392449","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41366889271483886167284378157366992074","series":"1.2.826.0.1.3680043.8.498.67078227894895472847691474252832658175","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40967435550546093348221485942169879116","series":"1.2.826.0.1.3680043.8.498.77758425902021536767545254456154887289","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41254472410580125134596211010943842213","series":"1.2.826.0.1.3680043.8.498.78596650630081320539763268867378351054","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41439889720274233955076963938741061358","series":"1.2.826.0.1.3680043.8.498.22634464298408837640098568819053290080","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41454762422897656673710145732036115764","series":"1.2.826.0.1.3680043.8.498.13343422264705961073426159010672178259","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41623512571866040054960002151226127775","series":"1.2.826.0.1.3680043.8.498.63874667259007594752093398739442601391","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41587640208769641326416798167574152049","series":"1.2.826.0.1.3680043.8.498.11603029539347581365578224668056371647","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41439889720274233955076963938741061358","series":"1.2.826.0.1.3680043.8.498.82434297805034574489661243417957940925","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41657361598083583047831423768034379414","series":"1.2.826.0.1.3680043.8.498.30628867572934275922246887202581160037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41698150452794914665945455397334605778","series":"1.2.826.0.1.3680043.8.498.38329406043709448668212560654205243506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41672145969790132883562432403529293545","series":"1.2.826.0.1.3680043.8.498.34571140402609869401976242046269907560","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41751049067043481681879071436091069601","series":"1.2.826.0.1.3680043.8.498.12058426034630231359706906833074811916","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41672145969790132883562432403529293545","series":"1.2.826.0.1.3680043.8.498.70160273359870384482914018667728370583","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41751049067043481681879071436091069601","series":"1.2.826.0.1.3680043.8.498.46009729729396517437336430395185721517","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41698150452794914665945455397334605778","series":"1.2.826.0.1.3680043.8.498.14873488589197746123622123880575171303","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41776233288569201013095816839442161537","series":"1.2.826.0.1.3680043.8.498.67749178612035403129309652546970593263","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42018410640565840140895012243840256810","series":"1.2.826.0.1.3680043.8.498.11245376206826128345779960757099192392","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42023788139360998656558751921918860808","series":"1.2.826.0.1.3680043.8.498.68067007265633660982839074818031065136","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42057349781448469704191384429694247725","series":"1.2.826.0.1.3680043.8.498.20740847771867953537505885946507272400","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42130173624033946465704266507948122088","series":"1.2.826.0.1.3680043.8.498.81402924067996488156290134217662940423","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41854610297797849683195403204758363476","series":"1.2.826.0.1.3680043.8.498.20936634473430741209081247528866059681","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42221373360893091454504422964928099406","series":"1.2.826.0.1.3680043.8.498.12893773981280446555468960296036387242","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41939824564209194493855981805593285528","series":"1.2.826.0.1.3680043.8.498.39806870171371896357001835537089668883","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42287188181305579531212818123200473947","series":"1.2.826.0.1.3680043.8.498.11086930334636678882408024434403988126","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42396216793489016293032290339618185275","series":"1.2.826.0.1.3680043.8.498.23837845959000429243461967134083788003","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42476443240785038044326483794941265936","series":"1.2.826.0.1.3680043.8.498.94393548735122602965905439826018870121","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42580107838226807862930203257414265276","series":"1.2.826.0.1.3680043.8.498.73188318763407156908108687791517535531","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42778524243264061817587398263842942394","series":"1.2.826.0.1.3680043.8.498.11891773528717921891487382385858601290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42634709955067416383713194799683605006","series":"1.2.826.0.1.3680043.8.498.83820996157790773592812265427659253488","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42829490507052740062127365747846030214","series":"1.2.826.0.1.3680043.8.498.89838823994627868761100743419235250625","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42887028634948156598417204377884297339","series":"1.2.826.0.1.3680043.8.498.13349825914649819002018713598087154247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42921593713351589642747633505679170597","series":"1.2.826.0.1.3680043.8.498.91898995662726217961501325384850073741","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42990542863216044717944418712887800295","series":"1.2.826.0.1.3680043.8.498.55890907476705739521308623110400970240","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43117007553940146763245418710268028741","series":"1.2.826.0.1.3680043.8.498.14096229508332475990590957407195431088","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43256905750043156063416845850825787417","series":"1.2.826.0.1.3680043.8.498.12572819196288470999492444894478411505","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42887028634948156598417204377884297339","series":"1.2.826.0.1.3680043.8.498.36795784643307178990315145588772482158","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.43325226037168674077736336919956196684","series":"1.2.826.0.1.3680043.8.498.25116955717101495883697051757061764095","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43412154740072739455825163326814980838","series":"1.2.826.0.1.3680043.8.498.11268625730549819341310509952031699316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43779228281849457650343989788070421822","series":"1.2.826.0.1.3680043.8.498.28591651952993216156787162544138862044","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052","series":"1.2.826.0.1.3680043.8.498.11993054309809493783302927678663946256","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44350638600163778456163342454087669278","series":"1.2.826.0.1.3680043.8.498.12391877115092131385322812052323174098","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44323131330689365651417881352766725366","series":"1.2.826.0.1.3680043.8.498.28184689114707581367242660725227695010","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727","series":"1.2.826.0.1.3680043.8.498.11728351354289094921571420352039943970","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44405554266141576380633014711246118209","series":"1.2.826.0.1.3680043.8.498.13397030373252644818149333051783513670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052","series":"1.2.826.0.1.3680043.8.498.72155128776079009546916411090424123371","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44629238055006732101555855930071163428","series":"1.2.826.0.1.3680043.8.498.51287701793167814309669476089775146858","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44733200650377282012534389271385687852","series":"1.2.826.0.1.3680043.8.498.15899299663188983350517520841365616252","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44733200650377282012534389271385687852","series":"1.2.826.0.1.3680043.8.498.13105218226124748991249652507461103517","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727","series":"1.2.826.0.1.3680043.8.498.53198365843090451265560068409274553630","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44875593707781066742286566388388035791","series":"1.2.826.0.1.3680043.8.498.76940591971074951576017283196794889698","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44629238055006732101555855930071163428","series":"1.2.826.0.1.3680043.8.498.10451520973595108643625974789544423445","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010","series":"1.2.826.0.1.3680043.8.498.44616733692555217636792984144739838589","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44875593707781066742286566388388035791","series":"1.2.826.0.1.3680043.8.498.10642122502633339343657470076124623359","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.45118217412917282413100913670807141634","series":"1.2.826.0.1.3680043.8.498.90913860843583326358367260708548832936","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45033326048521572717771708818514772861","series":"1.2.826.0.1.3680043.8.498.70104329926015293210436950231511931021","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45273393220765922863913956738614783632","series":"1.2.826.0.1.3680043.8.498.51873372323793318811325897752737944800","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45207812477654339877341538582090218293","series":"1.2.826.0.1.3680043.8.498.68509673080193666342035143580977107775","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641","series":"1.2.826.0.1.3680043.8.498.91584118067125423350030115618486512531","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45304570280714567513866386380356988876","series":"1.2.826.0.1.3680043.8.498.12344400982862140457496895755769957742","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45326662718293919909188368237039419949","series":"1.2.826.0.1.3680043.8.498.22265759881506524041031882598516894705","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775","series":"1.2.826.0.1.3680043.8.498.11054658209652840680024404180667300679","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45630051398709774279703407401383545642","series":"1.2.826.0.1.3680043.8.498.13398048817436588629564251223949438038","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45768777378108699271714894225826640121","series":"1.2.826.0.1.3680043.8.498.75481644998610072376207050394615031838","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45678969592219293243366784683191701674","series":"1.2.826.0.1.3680043.8.498.97708895203050291227611283011978830920","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45868806544202339567395643235910857549","series":"1.2.826.0.1.3680043.8.498.16342114870882716713253823912683259184","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45862157701704991290669186265816463426","series":"1.2.826.0.1.3680043.8.498.11350045455674602617294171278395881922","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.45380312679924593883892686293576260616","series":"1.2.826.0.1.3680043.8.498.56668210276652084147572204176450619403","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.46016776602328665980436110567854612595","series":"1.2.826.0.1.3680043.8.498.35167121674366803804301834914653737955","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46123761840464615814604226041496278995","series":"1.2.826.0.1.3680043.8.498.19816614034514555991444111881799163045","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46222691720198967161306520058359099856","series":"1.2.826.0.1.3680043.8.498.49814091991066216153477592673774831229","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46305694398420300356121078124030126469","series":"1.2.826.0.1.3680043.8.498.52843717883742913291611627837183232466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46051119885630602872050329412008545893","series":"1.2.826.0.1.3680043.8.498.13193887684795951337152256963472393219","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45968957856865486437256472516713628614","series":"1.2.826.0.1.3680043.8.498.31348287605696677946704651618605568032","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.46259494895672269475940484605821837768","series":"1.2.826.0.1.3680043.8.498.54832276293459416264596647665819051162","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46427879286435198611282055405541286766","series":"1.2.826.0.1.3680043.8.498.81358212941052617035277035145128506817","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46658713832425741743740258795376270265","series":"1.2.826.0.1.3680043.8.498.62101545680432739314428055095579746495","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46745172905542117506863575411125924241","series":"1.2.826.0.1.3680043.8.498.59648979593276993488419677734021663784","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46491051379327985199309365945879841064","series":"1.2.826.0.1.3680043.8.498.72241430192346346908971801196663198315","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46335849264221717416382765091672040855","series":"1.2.826.0.1.3680043.8.498.75837938664125195260779715681446174654","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.46909729013880708074310736136355969956","series":"1.2.826.0.1.3680043.8.498.12051710579978310193151285953055952018","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431","series":"1.2.826.0.1.3680043.8.498.10758761211319401814577924762509425419","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47061252152535181371772020997295000029","series":"1.2.826.0.1.3680043.8.498.47963401083152584918523226090823115662","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46825768767498441994570613185981335949","series":"1.2.826.0.1.3680043.8.498.82031261657776553210121022141251619208","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431","series":"1.2.826.0.1.3680043.8.498.26109988953413552297262786877156977660","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.47080537724499166054609508776778189902","series":"1.2.826.0.1.3680043.8.498.51833828646584338541340942010906127834","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46884541791777826238103426426267571657","series":"1.2.826.0.1.3680043.8.498.49147497449602192288530551486564650327","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47339093185444044578247608022398494955","series":"1.2.826.0.1.3680043.8.498.49333474409006287578960938625470965266","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47449893776425045020340604662487159684","series":"1.2.826.0.1.3680043.8.498.19997203892050294233531992776470585515","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47255564202539432049700324486197024855","series":"1.2.826.0.1.3680043.8.498.89841607589559196716600277015317417947","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47130872195336642107576758658144949813","series":"1.2.826.0.1.3680043.8.498.12377610240395415081977273649387056760","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47495385306095610608327984787779011133","series":"1.2.826.0.1.3680043.8.498.53267513935604426819345085791912274744","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47651715105501347255071941051553742767","series":"1.2.826.0.1.3680043.8.498.11435012018203283374373062459158528205","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47666607805434428477774854287399691400","series":"1.2.826.0.1.3680043.8.498.73002514475551845526732965991300384006","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710","series":"1.2.826.0.1.3680043.8.498.12910413013171964410637092383453724559","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47864423097056282241181943425451597067","series":"1.2.826.0.1.3680043.8.498.35890258285076351688871116158567875934","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47784065888386216103165400551121771174","series":"1.2.826.0.1.3680043.8.498.49524895932570954854395033043587071028","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187","series":"1.2.826.0.1.3680043.8.498.47044501123729808256032287793135859318","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47949251060660360953300340380324578835","series":"1.2.826.0.1.3680043.8.498.48261868502210520025020789077348738969","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48435973333068748044485658920315789799","series":"1.2.826.0.1.3680043.8.498.11787554472851309464917664227236836981","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274","series":"1.2.826.0.1.3680043.8.498.11341617802848274596509422018595940673","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48068031587855513429324313927339963541","series":"1.2.826.0.1.3680043.8.498.59802343928383391921450135737418059667","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48780598610511011488578963416186905560","series":"1.2.826.0.1.3680043.8.498.33205498022946058917568962416237535830","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48658459018915173972212724570862424208","series":"1.2.826.0.1.3680043.8.498.43384149213026621796699365889038368768","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619","series":"1.2.826.0.1.3680043.8.498.27201030582291459356578500931782480892","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48818931767017230678539144135307491171","series":"1.2.826.0.1.3680043.8.498.89743232203268345532079159153504018234","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48520238688122533624681796433258211236","series":"1.2.826.0.1.3680043.8.498.28752766479505763489028537924415760461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49025249926859976858200443939208207124","series":"1.2.826.0.1.3680043.8.498.34659433684565983190207755433532211221","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49048739684287632681242054218807611262","series":"1.2.826.0.1.3680043.8.498.11629853075179398674953974303770214695","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619","series":"1.2.826.0.1.3680043.8.498.57927024705912721772953222869307556059","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49094276335794811195117175603556929507","series":"1.2.826.0.1.3680043.8.498.11325743321113612770057272713539646005","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49287143478474218875661109488367501329","series":"1.2.826.0.1.3680043.8.498.80509723849803836095750074208160243368","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48918090379630186340047006216221526186","series":"1.2.826.0.1.3680043.8.498.56151097499757691328268797167085457747","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49379034023506626029648394840207360925","series":"1.2.826.0.1.3680043.8.498.20786914732519424890585661663953150188","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49441835811239898312243678447796777305","series":"1.2.826.0.1.3680043.8.498.13030619801715782427296805015936762203","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49186859342644783675130322288585477727","series":"1.2.826.0.1.3680043.8.498.82792977516051877599283612851392105341","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49481811111224238761880576486796603139","series":"1.2.826.0.1.3680043.8.498.32672605252067578296137987228081726416","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49495644703047055149965226028097259207","series":"1.2.826.0.1.3680043.8.498.70496800664874258458925055435998463049","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49379034023506626029648394840207360925","series":"1.2.826.0.1.3680043.8.498.92669247109314784934272630802927647138","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49603554452147117475589797389254733654","series":"1.2.826.0.1.3680043.8.498.11328762447625421037316745236438486973","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49818874332345742217581267858748819921","series":"1.2.826.0.1.3680043.8.498.30041758532043603631248543482816802354","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49679916686449433962263937489400526112","series":"1.2.826.0.1.3680043.8.498.60371035535156546218600232184614764640","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49679916686449433962263937489400526112","series":"1.2.826.0.1.3680043.8.498.10539400540971218905707537952670911117","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49866252311077397175807461727885923657","series":"1.2.826.0.1.3680043.8.498.66250585727951274254158656452575435481","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49575149159425506211571562732929634662","series":"1.2.826.0.1.3680043.8.498.11714204221642893797262713556622239643","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49850672111501136862025523789202033809","series":"1.2.826.0.1.3680043.8.498.47067099186349764932537900992441032740","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913","series":"1.2.826.0.1.3680043.8.498.43701102984744197796704672973595059652","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116","series":"1.2.826.0.1.3680043.8.498.43369870036306270545644658015786022966","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50182378358892377789975570268512056553","series":"1.2.826.0.1.3680043.8.498.69142471195977839055166456389343546273","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913","series":"1.2.826.0.1.3680043.8.498.26566443917372254382268727900398751892","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50261788831539215224639813005933285052","series":"1.2.826.0.1.3680043.8.498.38547761243193886045238530423010925743","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116","series":"1.2.826.0.1.3680043.8.498.79421000550649637471681545010690132021","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50458745240631577290185672298591447264","series":"1.2.826.0.1.3680043.8.498.13217222805239364368672945153034319288","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50458745240631577290185672298591447264","series":"1.2.826.0.1.3680043.8.498.10665756076346231271025693656340246551","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50501020658943435430736717390350999395","series":"1.2.826.0.1.3680043.8.498.11362648711608348406048475555209180304","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50392110746914984873009735588879049321","series":"1.2.826.0.1.3680043.8.498.23364970215057399280655303691211042973","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536","series":"1.2.826.0.1.3680043.8.498.10200060645286054663301728558173793668","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50680622091377404463269575858175673158","series":"1.2.826.0.1.3680043.8.498.71327909016438723938935336549313144927","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50666048236278074641860077012751071758","series":"1.2.826.0.1.3680043.8.498.11491991922654590224439184638227350008","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536","series":"1.2.826.0.1.3680043.8.498.98707693361303800071068801481777979583","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50689256431939553935971505794681061679","series":"1.2.826.0.1.3680043.8.498.10645227211415406602523257848600065751","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50823279853002320074080586693604976784","series":"1.2.826.0.1.3680043.8.498.10778767889505509210876556877908131463","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50848799900564667487859253004538870622","series":"1.2.826.0.1.3680043.8.498.78286642687972842198525782467639444356","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50707407856055788475528375847872674241","series":"1.2.826.0.1.3680043.8.498.67829658131238489090847661534729877166","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50780438941220579448947609376330661130","series":"1.2.826.0.1.3680043.8.498.87186935634170489725900810205939860486","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51059575642903646008860869691477305022","series":"1.2.826.0.1.3680043.8.498.68562172258124436840451630211011357198","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51143994849141407980459861759302509053","series":"1.2.826.0.1.3680043.8.498.58506750828096451884073325917270568150","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367","series":"1.2.826.0.1.3680043.8.498.34996526446392366396914567878488395967","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367","series":"1.2.826.0.1.3680043.8.498.74126991582256179961291017712460391529","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.51509214978951980397275258374070843967","series":"1.2.826.0.1.3680043.8.498.35819143472360457670552786902395926141","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51581188134109925758379838961317006116","series":"1.2.826.0.1.3680043.8.498.17759272266179354975388739101946887448","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51351386796317851725461803790273870995","series":"1.2.826.0.1.3680043.8.498.43934573577208116853503209422563770247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51684117251092766010524179062007524257","series":"1.2.826.0.1.3680043.8.498.65921469492228813546390647050426064428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51768733026033198941231586462443503827","series":"1.2.826.0.1.3680043.8.498.51923508030530023462927219697439332466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643","series":"1.2.826.0.1.3680043.8.498.50880585216242029849414147924857258221","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643","series":"1.2.826.0.1.3680043.8.498.18252978352352553012843974492886796309","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.51006813205605862503064447831381009425","series":"1.2.826.0.1.3680043.8.498.87495159805968657421810229321485619083","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.51782185934115000642557689300687723099","series":"1.2.826.0.1.3680043.8.498.11073369656333594529313240645394526704","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.51867133375333949773527521703402111601","series":"1.2.826.0.1.3680043.8.498.65751042235229116816640129759366012876","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51970198810758997428998886534742243215","series":"1.2.826.0.1.3680043.8.498.87918729980231637897787163933708419654","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52010826440192227660457107121818389761","series":"1.2.826.0.1.3680043.8.498.90386194879954722911746489831209925284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52059522418267239652374290810994711032","series":"1.2.826.0.1.3680043.8.498.58675891289035922454691478103039840759","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52098249444968803324423421624756966294","series":"1.2.826.0.1.3680043.8.498.11978173470409208127514384487455889131","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52132258638267172087240021957310219977","series":"1.2.826.0.1.3680043.8.498.52079337482948602738649817105640520777","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52098249444968803324423421624756966294","series":"1.2.826.0.1.3680043.8.498.10432374481448479492962601299241349851","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.51985313438970133264846259111827877590","series":"1.2.826.0.1.3680043.8.498.10096196214412855751569933718655124475","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.52140911874747534807033188343849537207","series":"1.2.826.0.1.3680043.8.498.30428228981312562877934442002595694759","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52330656126848153803758928167944040055","series":"1.2.826.0.1.3680043.8.498.11048488857242713495125116732708152365","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52390908285903728334239836699232920065","series":"1.2.826.0.1.3680043.8.498.25684844482798737066123514533591868120","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152","series":"1.2.826.0.1.3680043.8.498.66507966678059791526049517258658984330","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52140911874747534807033188343849537207","series":"1.2.826.0.1.3680043.8.498.75041817585102669497684078349042603760","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.52745530063404662086111315503240395535","series":"1.2.826.0.1.3680043.8.498.10433063857589847102604929200580755236","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52846236545327527967419125989598675101","series":"1.2.826.0.1.3680043.8.498.69647589636205866034368210153911169875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52499379042723926379822784411787551701","series":"1.2.826.0.1.3680043.8.498.39746801909137860431876215696558512461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52636981391578580436180193211113340410","series":"1.2.826.0.1.3680043.8.498.19338764122237791977310487514582982878","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53025489296610571044246428509154950855","series":"1.2.826.0.1.3680043.8.498.12488663346238643077505670979147495144","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53131430415327479511358970615140982847","series":"1.2.826.0.1.3680043.8.498.48043335153211096733741254751989468844","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53156118232564605912649705207438612782","series":"1.2.826.0.1.3680043.8.498.38909250269800336335703389189602053199","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53094450151502907325041400000717725711","series":"1.2.826.0.1.3680043.8.498.91668841018066234554533914267605155301","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53174810138171412044410050984734412146","series":"1.2.826.0.1.3680043.8.498.11989968352167300691974272635907635981","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53078461307849490495536549806091931824","series":"1.2.826.0.1.3680043.8.498.44968860697313761246446065802848740349","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.53331626867618182966740022261110760133","series":"1.2.826.0.1.3680043.8.498.98424760642661901160887003459105226000","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53343379040681745197301575417064236501","series":"1.2.826.0.1.3680043.8.498.11543292450071950655908719863629942307","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53490080963614272234072703760363052476","series":"1.2.826.0.1.3680043.8.498.18654559453568022065201362295054382181","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53490080963614272234072703760363052476","series":"1.2.826.0.1.3680043.8.498.11294289843723911767319012232478314962","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.53579416035647672067965892020505016454","series":"1.2.826.0.1.3680043.8.498.88329491824702062150385993467207269750","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53262958748075737242820629127001253146","series":"1.2.826.0.1.3680043.8.498.10137254015924957886233054309763173499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53884715696186989798696544661396487883","series":"1.2.826.0.1.3680043.8.498.74396420144585983527029914302186287022","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53792159922324410857755054939229546555","series":"1.2.826.0.1.3680043.8.498.30949861515304839918995580690338838155","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53901077733518036755308692304268815965","series":"1.2.826.0.1.3680043.8.498.44511076693656880530689549350669488171","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53968943793571323467397128102531786394","series":"1.2.826.0.1.3680043.8.498.52017392530791763063272568292447256721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53628897373416863107623485630264501760","series":"1.2.826.0.1.3680043.8.498.14540854081092695486250662819028313481","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092","series":"1.2.826.0.1.3680043.8.498.11514573965312681026370137143912454812","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54099199514076260519579096546600505763","series":"1.2.826.0.1.3680043.8.498.34870220861875012727868849092973899316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54270039578867689669466339980682684126","series":"1.2.826.0.1.3680043.8.498.13193761973759463498302688820761300319","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383","series":"1.2.826.0.1.3680043.8.498.44614487987239346637987619302014373898","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383","series":"1.2.826.0.1.3680043.8.498.33192477889995457167180846735443441537","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.54101830029742852876183926998159899424","series":"1.2.826.0.1.3680043.8.498.19847390363475334877162361980839805011","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54336962625483046366337626878609850219","series":"1.2.826.0.1.3680043.8.498.44019736021047619802589592640262664284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54138747504377266740838999528964336002","series":"1.2.826.0.1.3680043.8.498.35730224875749928340254895793969613762","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.54394066317295451515247748115888976860","series":"1.2.826.0.1.3680043.8.498.54893115327846913774331816413746267056","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.54482848766921329667469114743547203929","series":"1.2.826.0.1.3680043.8.498.41845437902929449485042445000886859120","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54539703415639765348239790227512479583","series":"1.2.826.0.1.3680043.8.498.43080382950641429877727618567314149107","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54689086194769265957947122616025330642","series":"1.2.826.0.1.3680043.8.498.57154476210021066119132254470183548911","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54689086194769265957947122616025330642","series":"1.2.826.0.1.3680043.8.498.45237431716415062469567127307248696719","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.54699196872730350880681018674893216245","series":"1.2.826.0.1.3680043.8.498.78355229406720743461008964261130495198","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665","series":"1.2.826.0.1.3680043.8.498.12377068076244001084971882161256618192","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.54619696938940618755334224550745735815","series":"1.2.826.0.1.3680043.8.498.70267881713400611912041743121142227374","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665","series":"1.2.826.0.1.3680043.8.498.59846291883505263555598454701461620766","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55008913229055942530957507434366806883","series":"1.2.826.0.1.3680043.8.498.65338804688658931923600697615492278408","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55022631713888448062201243978634363406","series":"1.2.826.0.1.3680043.8.498.88630720389848193582528157861324367746","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54792838438079861454532183998999249213","series":"1.2.826.0.1.3680043.8.498.14703118547242381638987914333502227720","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55105303802677863700162285680213496441","series":"1.2.826.0.1.3680043.8.498.80243485926890145564736482806862433135","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55202459816190607876303419067810637245","series":"1.2.826.0.1.3680043.8.498.72213256200317885472156633629686003417","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55332521421041708438748125666498042763","series":"1.2.826.0.1.3680043.8.498.78456007341293859282294166076225528867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55397602116058381866334455456729470963","series":"1.2.826.0.1.3680043.8.498.30592403905614031804643277321575230776","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55586358833903787781826406120068237870","series":"1.2.826.0.1.3680043.8.498.64245558369175615284996172022556450489","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55105303802677863700162285680213496441","series":"1.2.826.0.1.3680043.8.498.71517052427829006256300643565153173913","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55687452977471862155042703116192295932","series":"1.2.826.0.1.3680043.8.498.11365543330464820848028216770245947881","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55461938012492393262172182405932212246","series":"1.2.826.0.1.3680043.8.498.41854872679426100380968440342191688570","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55713753582074871470943388423906540792","series":"1.2.826.0.1.3680043.8.498.70290331533473067085813676251940404012","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55771181380353653474896884649540028856","series":"1.2.826.0.1.3680043.8.498.11036914294036234900721729777868767368","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55771181380353653474896884649540028856","series":"1.2.826.0.1.3680043.8.498.28231384208661622954034581654261648232","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55713753582074871470943388423906540792","series":"1.2.826.0.1.3680043.8.498.11619616976146383592178609956570979768","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55850385284038070407841702672827533728","series":"1.2.826.0.1.3680043.8.498.51653489828105901957527277063098951201","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55927741980493661714408715886641034285","series":"1.2.826.0.1.3680043.8.498.80697993227069291188536673258695786137","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55991293001848799160041967098772087659","series":"1.2.826.0.1.3680043.8.498.21869235733835244812784241747370679125","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55971152627433130850649668964887611410","series":"1.2.826.0.1.3680043.8.498.10032400772311267984878238000480301502","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56045687004558569005240775213510192805","series":"1.2.826.0.1.3680043.8.498.77926510113017967767861991552347728985","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55991293001848799160041967098772087659","series":"1.2.826.0.1.3680043.8.498.68301641174868634080488853815207939362","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55861544093014499478739864837675161978","series":"1.2.826.0.1.3680043.8.498.11738153680443515182493007500657918018","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391","series":"1.2.826.0.1.3680043.8.498.56248775220010251451334692923881943226","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56146582420561901778012523946835033628","series":"1.2.826.0.1.3680043.8.498.52098492412008198092854974660293039366","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391","series":"1.2.826.0.1.3680043.8.498.99876245666912456361098678800165100212","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56045687004558569005240775213510192805","series":"1.2.826.0.1.3680043.8.498.10409164003604402487727621523310819731","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56170570792418244380186761574367557535","series":"1.2.826.0.1.3680043.8.498.45990745567996907975648917308308750739","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265","series":"1.2.826.0.1.3680043.8.498.47733863137654577393750479312116091574","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56269695574661037525483516531194514456","series":"1.2.826.0.1.3680043.8.498.54163703347661988113333013449268088080","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56216873861595017403772627709751840588","series":"1.2.826.0.1.3680043.8.498.59845525838309370490882224194835453551","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56686055146646210144964207684044815006","series":"1.2.826.0.1.3680043.8.498.10204046559911104551511093511434055862","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56506415277488978301157490514745400410","series":"1.2.826.0.1.3680043.8.498.64704999575263864342371255879009837904","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265","series":"1.2.826.0.1.3680043.8.498.35748041848415859300931494010189369583","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56751453017492477106384704540803963879","series":"1.2.826.0.1.3680043.8.498.84973598197597980187835695256338066047","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56786556618312090924315296993756843070","series":"1.2.826.0.1.3680043.8.498.57294570755055588877920038017141993050","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56798070549054124854685511360777559033","series":"1.2.826.0.1.3680043.8.498.85851063548292299849789033230618280275","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56839374698050163738039760889156066445","series":"1.2.826.0.1.3680043.8.498.13786886845166051968125886233911349060","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56939860329243258146102948466697018543","series":"1.2.826.0.1.3680043.8.498.86233790345510345962049261646316525422","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56894990686683651282564584582335346350","series":"1.2.826.0.1.3680043.8.498.15354535590187023121610734146623792251","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56989113744241846544086409786820246754","series":"1.2.826.0.1.3680043.8.498.51857578458801712329538789944790614371","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57064618560116981041722203028857007300","series":"1.2.826.0.1.3680043.8.498.11807352063964066453507732088856171137","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56882015310689536014061315110969323429","series":"1.2.826.0.1.3680043.8.498.13286503379265371033774285205964550322","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57348281565196196401055821640494904475","series":"1.2.826.0.1.3680043.8.498.41883004558025425452945674528901544446","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57064618560116981041722203028857007300","series":"1.2.826.0.1.3680043.8.498.69439258318746977798517141743638329241","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.57084445429567494093960261980941917499","series":"1.2.826.0.1.3680043.8.498.91796943579838140236006886094701175553","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57491375928585600499323136123872536597","series":"1.2.826.0.1.3680043.8.498.10152926793192573096907395500887922500","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57768539239475064336346600805236336044","series":"1.2.826.0.1.3680043.8.498.10847605398821380490189123082429923907","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57160448024145680660335079050450776852","series":"1.2.826.0.1.3680043.8.498.10790783356264073999992185501529367279","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57789603189240866041812385320399435864","series":"1.2.826.0.1.3680043.8.498.58996715070760154170165789158013415429","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57836524675365700674244122764523161699","series":"1.2.826.0.1.3680043.8.498.25837117313310872675556128111534360313","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57789603189240866041812385320399435864","series":"1.2.826.0.1.3680043.8.498.10482953523378544223931005096908522838","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.57836524675365700674244122764523161699","series":"1.2.826.0.1.3680043.8.498.67682700353087702953950044841225227742","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.57917634151851433037090858498504057826","series":"1.2.826.0.1.3680043.8.498.11604033470282092525291733783263294344","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57859543314667538612269642333387175763","series":"1.2.826.0.1.3680043.8.498.12568965992658599728424445923543928764","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57917634151851433037090858498504057826","series":"1.2.826.0.1.3680043.8.498.12993161009741911762118825521333327673","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58143227319316521737029358930629086887","series":"1.2.826.0.1.3680043.8.498.60694424501282146497298781452156482215","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348","series":"1.2.826.0.1.3680043.8.498.19246634040253279997620827049388393497","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58280111882249056715877316026191844548","series":"1.2.826.0.1.3680043.8.498.11514628179713640198471497644264977383","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348","series":"1.2.826.0.1.3680043.8.498.75049072243943345940551817540951392655","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58250779958866595505585700517927225230","series":"1.2.826.0.1.3680043.8.498.19247869399882732240753599346888684727","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58280111882249056715877316026191844548","series":"1.2.826.0.1.3680043.8.498.11600907113450964774847431470526752314","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58475208045272914292874190001248518687","series":"1.2.826.0.1.3680043.8.498.26519712675749212244979136690926091679","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58482779112389764165599701941310919660","series":"1.2.826.0.1.3680043.8.498.11639756753726637175784913703790439298","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58550899000434548218916895907938175917","series":"1.2.826.0.1.3680043.8.498.11523148178674144351678846426443849316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58475208045272914292874190001248518687","series":"1.2.826.0.1.3680043.8.498.12126625334360898113709585790199269447","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58604600600114899516493668210902251434","series":"1.2.826.0.1.3680043.8.498.18144847193950237124047986941233880482","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58634494571833713296574414948455471101","series":"1.2.826.0.1.3680043.8.498.53640391237684225690037234682347967669","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58685941646495891824559391997783302656","series":"1.2.826.0.1.3680043.8.498.46506504890013522948759620373158669941","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58728013757221328590898166513995249334","series":"1.2.826.0.1.3680043.8.498.58636095365373256174581443865479055981","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58749835013162544157291112101941849646","series":"1.2.826.0.1.3680043.8.498.27364352362269583489383781702544250304","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58776844157027040145262759105410792687","series":"1.2.826.0.1.3680043.8.498.91574050285902522689150219451871736996","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58776844157027040145262759105410792687","series":"1.2.826.0.1.3680043.8.498.53002782204747239861178191234867874084","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58835863392949696556026907211309159182","series":"1.2.826.0.1.3680043.8.498.10802953193391984612776898939511703023","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58888406752590566453001353089284177456","series":"1.2.826.0.1.3680043.8.498.10032031932490473788325421579189340980","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58835863392949696556026907211309159182","series":"1.2.826.0.1.3680043.8.498.57012793738440425781659432525375709673","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58888406752590566453001353089284177456","series":"1.2.826.0.1.3680043.8.498.13027003838443163651042163135200882836","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58965370475364247222892582428665460530","series":"1.2.826.0.1.3680043.8.498.10782681376151847032847258265887255185","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59047534248122071167153110802170658177","series":"1.2.826.0.1.3680043.8.498.22504952822492787894327142791367812227","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58965370475364247222892582428665460530","series":"1.2.826.0.1.3680043.8.498.85325771287281028194524945087418229955","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59073122184069676919857712290159511305","series":"1.2.826.0.1.3680043.8.498.17568228210008974949465689824961363281","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59126324546257448818397467078224944768","series":"1.2.826.0.1.3680043.8.498.76454022460265932700789766974049210896","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59047534248122071167153110802170658177","series":"1.2.826.0.1.3680043.8.498.55202818606662673412875524904706813750","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59254379036462832374262221063106167294","series":"1.2.826.0.1.3680043.8.498.22344028675730518820569867493064717145","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59420153636832938628088507776889305823","series":"1.2.826.0.1.3680043.8.498.29910923725402038567519721566926623297","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59449702487184611096113620550399616423","series":"1.2.826.0.1.3680043.8.498.75339102783200406483599868336206590192","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59533384841491000088839469462294817273","series":"1.2.826.0.1.3680043.8.498.79537031139222138930452905467939389413","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59476657510212258499047356518058209126","series":"1.2.826.0.1.3680043.8.498.13949403750334376276606794122232557324","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59533384841491000088839469462294817273","series":"1.2.826.0.1.3680043.8.498.13263119737797114507578490477710465998","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59680220055968160813801840287391339651","series":"1.2.826.0.1.3680043.8.498.12831377263102999999361056439687265008","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59932425615255727459402684084295081504","series":"1.2.826.0.1.3680043.8.498.12556941192548691519972162168414423518","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59758232061190668535098343218115599228","series":"1.2.826.0.1.3680043.8.498.73586374324306968107469048229525512801","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59758232061190668535098343218115599228","series":"1.2.826.0.1.3680043.8.498.12043394595451318727517027428487616506","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59932425615255727459402684084295081504","series":"1.2.826.0.1.3680043.8.498.54039885251747185965995391144531491053","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59864697179957009967756900038846073439","series":"1.2.826.0.1.3680043.8.498.61884933630516822310593153104675128503","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59939977812549570931951749553467050226","series":"1.2.826.0.1.3680043.8.498.10883585244506773525002187708928084863","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197","series":"1.2.826.0.1.3680043.8.498.62600607422323790851475180038903444709","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60094182309575937331152278560900767484","series":"1.2.826.0.1.3680043.8.498.57578482403598925460696271831065851337","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60189166269847256202786309096136242746","series":"1.2.826.0.1.3680043.8.498.12117055260331355728324225349126553579","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60064080350033586611530748685370040101","series":"1.2.826.0.1.3680043.8.498.63000861685769444367937811934795700703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60240929196665639608259748424475337443","series":"1.2.826.0.1.3680043.8.498.39376041165374446601324127572528791268","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60240929196665639608259748424475337443","series":"1.2.826.0.1.3680043.8.498.73710131077833183904079350220560159889","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.60339775486026531260673620094004244044","series":"1.2.826.0.1.3680043.8.498.12247005486278323975267197657529907423","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60446085094710623303973373214794179184","series":"1.2.826.0.1.3680043.8.498.18579242503683294541152406438995893380","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60435128564448668334758790827765960420","series":"1.2.826.0.1.3680043.8.498.14194501726680239105974840596428876802","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60473167624534682552272469123806503959","series":"1.2.826.0.1.3680043.8.498.51975478343830714021292488048706235470","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60064080350033586611530748685370040101","series":"1.2.826.0.1.3680043.8.498.12809283540172962133123712580181674293","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.60492772672040498730937577391401159109","series":"1.2.826.0.1.3680043.8.498.69741618820975847132645061843301953410","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60582762894748063612925320003677861384","series":"1.2.826.0.1.3680043.8.498.12083740916880129509111590574831857251","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60736996513214411991331917490659995815","series":"1.2.826.0.1.3680043.8.498.38902323997851936023867728296016535263","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331","series":"1.2.826.0.1.3680043.8.498.10047997472951043536409926572347675423","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60661676782543347731000874708337132276","series":"1.2.826.0.1.3680043.8.498.86695289144158958722573567457982250071","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60924385750386872880015870524131415592","series":"1.2.826.0.1.3680043.8.498.11716692397603814589105338507768118071","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61043690805522768840878574610198420558","series":"1.2.826.0.1.3680043.8.498.75493381247022181788777852006894063922","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60965286164497194740320124904525623873","series":"1.2.826.0.1.3680043.8.498.69670870235549888607634451612530780214","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61148667330113929729568621580194520169","series":"1.2.826.0.1.3680043.8.498.54820253112529008593130410503634348508","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61140619849833544411463750382539885141","series":"1.2.826.0.1.3680043.8.498.99775423944799325036101933152542684303","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61313243050348988917987796330691448770","series":"1.2.826.0.1.3680043.8.498.86953556057249889928982957932390501252","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61354742313056287432707914716598361679","series":"1.2.826.0.1.3680043.8.498.31655217979000264258284655333173698015","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61227644940421894951950605627016308160","series":"1.2.826.0.1.3680043.8.498.28903461005319159111125181268927310611","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61518377417979517706237284901970943470","series":"1.2.826.0.1.3680043.8.498.10824313044060519376945112910546901326","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61227644940421894951950605627016308160","series":"1.2.826.0.1.3680043.8.498.83648230259028003660273293853761641840","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.61715630583919056026446344483871058799","series":"1.2.826.0.1.3680043.8.498.51601096887628152744284220684290052944","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61554384977916135091630433680780524386","series":"1.2.826.0.1.3680043.8.498.10803965868662859696394825319541107371","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61518377417979517706237284901970943470","series":"1.2.826.0.1.3680043.8.498.49775188754784417277292296218006201750","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.61581513083114643407028382481113097015","series":"1.2.826.0.1.3680043.8.498.58848273563927895425324779708468508580","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61956863718716027369068406125965902569","series":"1.2.826.0.1.3680043.8.498.17734253084415029583618464171214058322","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62025036385410820041698409408092924893","series":"1.2.826.0.1.3680043.8.498.23297939218150978097866585850104510290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950","series":"1.2.826.0.1.3680043.8.498.22286969299489251893553259451400397598","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62192347880785177732202565414591408086","series":"1.2.826.0.1.3680043.8.498.46666737775301654692257137363353206906","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950","series":"1.2.826.0.1.3680043.8.498.39314494064031642789612465101728623803","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62251470468640645490157592046417355534","series":"1.2.826.0.1.3680043.8.498.32446494160670291371172728822249794058","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62192347880785177732202565414591408086","series":"1.2.826.0.1.3680043.8.498.68962848030855897951079533053744183575","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62251470468640645490157592046417355534","series":"1.2.826.0.1.3680043.8.498.12036690169154659079064932380973744533","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62437671010172828772205139640126159272","series":"1.2.826.0.1.3680043.8.498.79496280106518839939751726174623793865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236","series":"1.2.826.0.1.3680043.8.498.18194661389225439596589076323105764635","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62693151030819598514729637644804213983","series":"1.2.826.0.1.3680043.8.498.32842275987423162568824927135486587721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62329283530383340787096585492217632179","series":"1.2.826.0.1.3680043.8.498.91067726617483342696198688693545985329","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62400371770962739002393364782172386752","series":"1.2.826.0.1.3680043.8.498.12392385874808941511732045782506116231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62760659205261492447481830774781950875","series":"1.2.826.0.1.3680043.8.498.32069314590728962271901022995013642196","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62983291359021407345388012674614537807","series":"1.2.826.0.1.3680043.8.498.85697690517592080880464119117432562491","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62928513795250694194589804784951257414","series":"1.2.826.0.1.3680043.8.498.10765316083295720734095967085237842637","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63069333313670617776227577192591993067","series":"1.2.826.0.1.3680043.8.498.91058825215190683163362418222250170314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63069333313670617776227577192591993067","series":"1.2.826.0.1.3680043.8.498.58438023206344872937339343511656311444","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63151733210715530368323422018984705316","series":"1.2.826.0.1.3680043.8.498.24593307024714701964697112334745815293","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63005973438879239920248535292375481294","series":"1.2.826.0.1.3680043.8.498.34360659692856146755153460097635945939","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63105151375349194807502162946823966410","series":"1.2.826.0.1.3680043.8.498.92865247620345064195957269277932570621","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63197503121546251937003511751049515360","series":"1.2.826.0.1.3680043.8.498.31796825740197385330279476670276101060","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113","series":"1.2.826.0.1.3680043.8.498.23533450479904603295103108892705845321","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63279205896049798083339413931561428540","series":"1.2.826.0.1.3680043.8.498.11557056941512403901098528253592757460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113","series":"1.2.826.0.1.3680043.8.498.12063463984056465813173810026789019105","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63377901538320536934403969965624202519","series":"1.2.826.0.1.3680043.8.498.10945401932329783147606712514253414847","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63320397279666280420359587173167195895","series":"1.2.826.0.1.3680043.8.498.93474800335529789313147356306439348558","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63426442500798189991842988966227275166","series":"1.2.826.0.1.3680043.8.498.14351129593649633208706544647724867373","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63521033783841675104013677430088400343","series":"1.2.826.0.1.3680043.8.498.38535917330460769346947304478781299462","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63510378689078892579575716369639643386","series":"1.2.826.0.1.3680043.8.498.10288622525641624692432373179101443000","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63510378689078892579575716369639643386","series":"1.2.826.0.1.3680043.8.498.85261893366682958419263339836362342714","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63539809431911516912857504094611479480","series":"1.2.826.0.1.3680043.8.498.69563895284921360753339319081415669566","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63426442500798189991842988966227275166","series":"1.2.826.0.1.3680043.8.498.60224196496701258110882101863675023242","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63565089119380386830816381560603756258","series":"1.2.826.0.1.3680043.8.498.91769100149163118586875381282823957006","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63726512375511246975989322174096818969","series":"1.2.826.0.1.3680043.8.498.11647492351994738026192781180536503463","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63556690547679257097888724096967682731","series":"1.2.826.0.1.3680043.8.498.96823207454494403729486280178860252072","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63849051301265505051181245915656194652","series":"1.2.826.0.1.3680043.8.498.60027487693633240313067815852171034885","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63894147741855851067411708837856142566","series":"1.2.826.0.1.3680043.8.498.48744821012265682092411622489093098125","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64029768638431636860053484202080135316","series":"1.2.826.0.1.3680043.8.498.41843332463858550290023830687132498365","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64066465191710322298540005681534170036","series":"1.2.826.0.1.3680043.8.498.61440899752885841992342211519564310574","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453","series":"1.2.826.0.1.3680043.8.498.55787414140753925274618680298630905599","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63673733836218886397484265807806548473","series":"1.2.826.0.1.3680043.8.498.22066632036163267774309140075064501393","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.64186424577324542519674308830472181373","series":"1.2.826.0.1.3680043.8.498.10325229758829749556646655546011774415","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64283618749539713756220539395944025878","series":"1.2.826.0.1.3680043.8.498.11047432462337301703400216886201611671","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64408775259838960958354835366765941676","series":"1.2.826.0.1.3680043.8.498.54830105120608560737425666759105156780","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453","series":"1.2.826.0.1.3680043.8.498.10192777225050747611729749702310348730","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.64443149121726771900576109439951701105","series":"1.2.826.0.1.3680043.8.498.56516361927217933027899907427539946491","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64397747170052970974000650185716672477","series":"1.2.826.0.1.3680043.8.498.11512974439057827025206859411800435638","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64590392444259942263746282444188079724","series":"1.2.826.0.1.3680043.8.498.92229289034200301718543240685935640241","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64663285742626599570990862360902141188","series":"1.2.826.0.1.3680043.8.498.36732567656181545758023489459848729705","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64640435232894033317418319985727526223","series":"1.2.826.0.1.3680043.8.498.74520305644373731111005396813044256512","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64950600385134699009390197520158979183","series":"1.2.826.0.1.3680043.8.498.10829518572080462116197141483348107931","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64865300853258390816475799272221318005","series":"1.2.826.0.1.3680043.8.498.10714041891756060264625726242555745977","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65123055741109319778634264103224173252","series":"1.2.826.0.1.3680043.8.498.25422141867340988792367097030607378438","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65160241567146620406193593170843451582","series":"1.2.826.0.1.3680043.8.498.45397582526841338041527384119357112069","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65077085081642080527129904737510276643","series":"1.2.826.0.1.3680043.8.498.11834876061248800329822935614773457837","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64526757783166918696475501252110447010","series":"1.2.826.0.1.3680043.8.498.11694149834541607458384610387335824604","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65322445533737673246312151513913460816","series":"1.2.826.0.1.3680043.8.498.16231392397773664120365961190529518915","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65403984120528711726317048984986599537","series":"1.2.826.0.1.3680043.8.498.23434287304727560740322237013451173706","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65222333228514862688331581862087607325","series":"1.2.826.0.1.3680043.8.498.11851035075301709990846507707824990897","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65244223225926362862970496735749098549","series":"1.2.826.0.1.3680043.8.498.38497707940468039422233351382057604976","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65607807464576681752987746458209389847","series":"1.2.826.0.1.3680043.8.498.12072748722758310488145211632755788874","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65715827585629489888518528736848969678","series":"1.2.826.0.1.3680043.8.498.11689288357535251359518942626947329517","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65515828911930515002046151253057175178","series":"1.2.826.0.1.3680043.8.498.53255323769229916695493568812601016434","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65855247249608663104822603172047250057","series":"1.2.826.0.1.3680043.8.498.46369061732464976869459226202082060256","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65959182350599581687336889990076028593","series":"1.2.826.0.1.3680043.8.498.82418341380517912098084905365302434466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65855247249608663104822603172047250057","series":"1.2.826.0.1.3680043.8.498.35948238332865869484294532812156815102","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65788206282522750507559682036750692505","series":"1.2.826.0.1.3680043.8.498.12027151130034168067578359372062991521","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66020213859389864645118554257523478921","series":"1.2.826.0.1.3680043.8.498.12733723715713373999710461270514380443","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66031142975529812579758652269360365398","series":"1.2.826.0.1.3680043.8.498.98245099402587417357529873580269242442","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65909865035783831607773206653044421345","series":"1.2.826.0.1.3680043.8.498.55215746734310754677097851619815704801","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66156015559325316834458423557757995415","series":"1.2.826.0.1.3680043.8.498.22725777903896729880944666205750042625","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66305771884507759587182077473863996177","series":"1.2.826.0.1.3680043.8.498.12031989382361785419422348400784064963","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66334686028272068369345540785320291695","series":"1.2.826.0.1.3680043.8.498.54690476002007218895568660117966336039","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66210455934011506415244876770725282769","series":"1.2.826.0.1.3680043.8.498.11748905424369737346368155484679308322","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66321903250295261063309947910189882952","series":"1.2.826.0.1.3680043.8.498.12678771913225391981433670996393544627","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66401771232064170830621215861092532421","series":"1.2.826.0.1.3680043.8.498.24496945471271037714947220567229014285","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66489634638847834550694493585147280754","series":"1.2.826.0.1.3680043.8.498.27687384959962687349497502709413456509","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66690100241274246406145618641481787709","series":"1.2.826.0.1.3680043.8.498.13162175282834269775211667628036437708","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66993076561387452114335652021593805727","series":"1.2.826.0.1.3680043.8.498.33906482998534877335491535754365571225","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66841103065389483583108279900721522084","series":"1.2.826.0.1.3680043.8.498.27384506294408130972778777855061184749","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269","series":"1.2.826.0.1.3680043.8.498.11170540158517120100023339094532389979","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269","series":"1.2.826.0.1.3680043.8.498.54396157650197156521172294293317780349","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67189969532330535369638450306601659967","series":"1.2.826.0.1.3680043.8.498.73302220637808324933295048679967824546","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67181088492292316990397177612918195713","series":"1.2.826.0.1.3680043.8.498.83514004643308074711007986871790140824","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66960891947495601371173426277430301127","series":"1.2.826.0.1.3680043.8.498.11619637202488605590430668710697737794","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67386780309737226410965627751074124678","series":"1.2.826.0.1.3680043.8.498.42828426573670756088395070488398151672","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67404282038417089622852480023919627784","series":"1.2.826.0.1.3680043.8.498.39973792761088709719090440988006074633","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67468397709097947104298147948850473142","series":"1.2.826.0.1.3680043.8.498.12303640761135542651539013526370733030","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67550262192397377344258276170992188234","series":"1.2.826.0.1.3680043.8.498.22384759470759817168056761210883675476","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67468397709097947104298147948850473142","series":"1.2.826.0.1.3680043.8.498.36789025804716126939252928439200498676","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67550262192397377344258276170992188234","series":"1.2.826.0.1.3680043.8.498.11585053900441817668830529397638832855","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67613388743126519966672593774754849330","series":"1.2.826.0.1.3680043.8.498.41485319245058030787095252395270550935","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67705068860320494485379949506807098172","series":"1.2.826.0.1.3680043.8.498.59561184260870500602842090886683063128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67739091465243228820766982459725849814","series":"1.2.826.0.1.3680043.8.498.10623657413008423494228198631193755922","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67678576373934022409084801948795250361","series":"1.2.826.0.1.3680043.8.498.72570114200462886091232291283612901411","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67788047649994083591931245362169694306","series":"1.2.826.0.1.3680043.8.498.69839990227932433660133957390706484314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67678576373934022409084801948795250361","series":"1.2.826.0.1.3680043.8.498.73926143085093235845407144850145500557","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67931702364673809781455566584550075686","series":"1.2.826.0.1.3680043.8.498.70917587410073221467214440250170042637","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67860057199678546706175336855949799228","series":"1.2.826.0.1.3680043.8.498.91523870132991027373659450681205922135","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67958551704692248657133951054036903879","series":"1.2.826.0.1.3680043.8.498.11272033137183498155471435765152767060","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68079818232244983808025467536454890669","series":"1.2.826.0.1.3680043.8.498.95071697530905425576952881499834935707","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67931702364673809781455566584550075686","series":"1.2.826.0.1.3680043.8.498.93063230799480820390464183661578432761","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666","series":"1.2.826.0.1.3680043.8.498.52258335052499678116050605291067195012","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666","series":"1.2.826.0.1.3680043.8.498.11504656323318458047204951956423360619","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606","series":"1.2.826.0.1.3680043.8.498.10370798628018548953580678824040812459","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68168197665046786209369894696801397179","series":"1.2.826.0.1.3680043.8.498.51045261746984269783924810704105846339","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67981539513814867568631217960473023750","series":"1.2.826.0.1.3680043.8.498.13361687376091369197490868415260869980","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606","series":"1.2.826.0.1.3680043.8.498.58434302343002978719224489580240432515","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68256141451764547710152696919981893952","series":"1.2.826.0.1.3680043.8.498.94922822833015308733576586216186637058","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68219942161081583831276769978472014317","series":"1.2.826.0.1.3680043.8.498.12336322998272578155884657888277451471","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68219942161081583831276769978472014317","series":"1.2.826.0.1.3680043.8.498.12132970385324729980191759600759816530","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68264559730756309944466784364591509088","series":"1.2.826.0.1.3680043.8.498.43623687623502374602127114461392455236","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68328371080087298795966933114044350359","series":"1.2.826.0.1.3680043.8.498.84856935910543376412603608882157552351","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68435927323644457286635872824563629385","series":"1.2.826.0.1.3680043.8.498.41338536392435643174761142873262628724","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68524920268469128133836299591793117756","series":"1.2.826.0.1.3680043.8.498.12266552908429747682251232311026317189","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68576465611266620513305434082656807028","series":"1.2.826.0.1.3680043.8.498.31045894876327339115584519350495108030","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68593524008570334427931338225473487643","series":"1.2.826.0.1.3680043.8.498.12973302737091268453301710188951510591","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68524920268469128133836299591793117756","series":"1.2.826.0.1.3680043.8.498.31725279893720226999009822804215429007","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68740466085414970518655922465908223436","series":"1.2.826.0.1.3680043.8.498.33518812944269434546229569906509899739","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157","series":"1.2.826.0.1.3680043.8.498.10161381035240214551499258258065290947","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68931510756844060683643298583132884960","series":"1.2.826.0.1.3680043.8.498.10946241467695410024086842966081374479","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68668620295929607459355154952035421306","series":"1.2.826.0.1.3680043.8.498.18671963842004643963211420397081168457","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157","series":"1.2.826.0.1.3680043.8.498.10033301774034198032502564349466321155","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68971603226593154050514800137550171537","series":"1.2.826.0.1.3680043.8.498.27696358745480866284938849982138632037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002","series":"1.2.826.0.1.3680043.8.498.29181677329774151528489913812825347708","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69267821440055811776827497076481587198","series":"1.2.826.0.1.3680043.8.498.10061845342080774594468251271422501411","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69411539430142524612470544041263824359","series":"1.2.826.0.1.3680043.8.498.12629423041849186473119818542688479555","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69156800026549034619316278954442147703","series":"1.2.826.0.1.3680043.8.498.35154789773882438895752037627964905064","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69544085715595825083308172629360899339","series":"1.2.826.0.1.3680043.8.498.11196343239270276807411581914910806616","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69654898267139720912940475772303404772","series":"1.2.826.0.1.3680043.8.498.13304560153473432627374597991175525873","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69846596572083911408018130524077120494","series":"1.2.826.0.1.3680043.8.498.12619565303347867263478305304684733860","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69729903409900053570304330127687235813","series":"1.2.826.0.1.3680043.8.498.13302003063447406229374047585488123192","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69632034454628807663252084651278084839","series":"1.2.826.0.1.3680043.8.498.10466880668297225259169983524928305243","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.70291739279544255146458190540233023337","series":"1.2.826.0.1.3680043.8.498.99362076637708614371418704287390871497","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69673283369917182409597968479300552579","series":"1.2.826.0.1.3680043.8.498.26354114305010111679386328546793761484","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.70124572708965486791612654433965515063","series":"1.2.826.0.1.3680043.8.498.40553489248617812582177463076241702791","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70310491987092072540988486213408328865","series":"1.2.826.0.1.3680043.8.498.28502197910923189379217871513048732477","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70464717849928808075676221329034929056","series":"1.2.826.0.1.3680043.8.498.10598636614346629504940669196528760067","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70478068758081824347762712452252763894","series":"1.2.826.0.1.3680043.8.498.65430565948415332501391510475984061540","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70609649448269244608860767754947143395","series":"1.2.826.0.1.3680043.8.498.11861686649330003827484619730109918867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70464717849928808075676221329034929056","series":"1.2.826.0.1.3680043.8.498.35397962767713394583344716641672115386","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.70931245733460964524244528230239226104","series":"1.2.826.0.1.3680043.8.498.11219633757056708186545223498604920934","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70873330457307886117949106432465978541","series":"1.2.826.0.1.3680043.8.498.80832994079284592208092526120319740952","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70796360440320543674384511859375295769","series":"1.2.826.0.1.3680043.8.498.84964706707614046751499049842735893115","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.70966951020892708303866968206211604316","series":"1.2.826.0.1.3680043.8.498.25518277760841348555423979768630748089","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70948162965271404678418299669321564765","series":"1.2.826.0.1.3680043.8.498.60964603000189189106826710100466592080","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522","series":"1.2.826.0.1.3680043.8.498.86569833234152366143443820388448409891","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71262022207903748073532915013888861999","series":"1.2.826.0.1.3680043.8.498.13352052718078637996900953951044931338","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71063550263564056105844586167361750567","series":"1.2.826.0.1.3680043.8.498.96130267576375844096054483146813346156","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71355457409240695394093233905149541215","series":"1.2.826.0.1.3680043.8.498.56259320743913591654286419763033002512","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71405492944866967669069984757332849184","series":"1.2.826.0.1.3680043.8.498.42959568419872895819129270619585492857","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71622107620007706778976946784787365505","series":"1.2.826.0.1.3680043.8.498.42831976851723896252231188230257381738","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71405492944866967669069984757332849184","series":"1.2.826.0.1.3680043.8.498.10625483737156461198922750615764930315","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.71660070729774772658965427776586988585","series":"1.2.826.0.1.3680043.8.498.54027099055984580203288261970980580088","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71686717249431292334159336816742076478","series":"1.2.826.0.1.3680043.8.498.12849359094645625361882703939722682571","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71709477515434076105531625950744135656","series":"1.2.826.0.1.3680043.8.498.37919564258066401435626724982350214278","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71753603955749189961638324806586359888","series":"1.2.826.0.1.3680043.8.498.24329565745226215835696326258945983959","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71830371861099586573789550797712053914","series":"1.2.826.0.1.3680043.8.498.11044032401135442718762365489035576029","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71792468450491064417953341293663396190","series":"1.2.826.0.1.3680043.8.498.13109692608135055362186142277268480460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71701484984624141610445774249629295610","series":"1.2.826.0.1.3680043.8.498.11979871641763410370600780440617637861","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525","series":"1.2.826.0.1.3680043.8.498.19499151113559485426805029708289659899","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525","series":"1.2.826.0.1.3680043.8.498.35830257538009789937607500814904876370","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.71899370657875960028369229847153071577","series":"1.2.826.0.1.3680043.8.498.60672898004145481782818295018782799694","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.72017096768214637927798648334301077145","series":"1.2.826.0.1.3680043.8.498.22120938045266163726614937846426701814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72023797276837219152365428999317131110","series":"1.2.826.0.1.3680043.8.498.95625193648692870245605268762927823788","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71985417286226511458068110832784023627","series":"1.2.826.0.1.3680043.8.498.93029383265993951006674736578557105140","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72017096768214637927798648334301077145","series":"1.2.826.0.1.3680043.8.498.52187124996077575634489730247223310972","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.72159028826938094045309841541333724597","series":"1.2.826.0.1.3680043.8.498.41987659697326880209452278140176892549","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72191353854496806844040272237700346774","series":"1.2.826.0.1.3680043.8.498.87026161474275179389102010309270639563","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72182847112910012489603688436912469641","series":"1.2.826.0.1.3680043.8.498.76240945693310477871976969778147465919","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72276631472244076773626607623783737687","series":"1.2.826.0.1.3680043.8.498.10483567027444360172241675960613984189","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72248728319398595146365363062799556637","series":"1.2.826.0.1.3680043.8.498.79015991358090694441124180370103546434","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72255181548595299889516400134053439401","series":"1.2.826.0.1.3680043.8.498.74529313647742425100980825549653181202","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72375103774239787721371778221019402834","series":"1.2.826.0.1.3680043.8.498.80384241949196238416728201739034918910","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72342425990274562150756501476989691572","series":"1.2.826.0.1.3680043.8.498.72097936822787395766390722594710943022","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72453426317225885428514595399496580896","series":"1.2.826.0.1.3680043.8.498.88878690142615467712708924289514130529","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72547312490317214326791189007411860203","series":"1.2.826.0.1.3680043.8.498.12331574199648102538128587655428879484","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72495590491484814220060275823761767516","series":"1.2.826.0.1.3680043.8.498.14784314868477253785798100825118197709","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72375103774239787721371778221019402834","series":"1.2.826.0.1.3680043.8.498.77337675224309273961892971688419996488","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.72680485820462987585370556994612687540","series":"1.2.826.0.1.3680043.8.498.44445740528337295585904380493414791210","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72740499664060575780267947543695605585","series":"1.2.826.0.1.3680043.8.498.16041400130546567411185979528690125122","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921","series":"1.2.826.0.1.3680043.8.498.57215472262185507167106208233273188371","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72896348955280307080916504120667484813","series":"1.2.826.0.1.3680043.8.498.10183002382366122196721283076964117231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72587440202779699193620339987034971895","series":"1.2.826.0.1.3680043.8.498.10975979156342844176072029132403136773","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72781608662300877077691853670862414309","series":"1.2.826.0.1.3680043.8.498.40903114975609029357805140860099570851","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72931900540220826595003388066347326492","series":"1.2.826.0.1.3680043.8.498.46444221873048847952170938721582633928","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72800659596099468282135738543971261904","series":"1.2.826.0.1.3680043.8.498.88208105502029063326875243616289560989","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.72943191222961132416678221885831944435","series":"1.2.826.0.1.3680043.8.498.13040212818534732735439762770324014783","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73037642285055457162691486258363101398","series":"1.2.826.0.1.3680043.8.498.29965821567949062977426906118996592372","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73037642285055457162691486258363101398","series":"1.2.826.0.1.3680043.8.498.24168888324252380403473219324103275057","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.73077380963620788952030569132380549826","series":"1.2.826.0.1.3680043.8.498.38045873288836588191909402251884936144","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73101997284755549124715401557235981642","series":"1.2.826.0.1.3680043.8.498.11652604130171918724530053163631122299","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73077380963620788952030569132380549826","series":"1.2.826.0.1.3680043.8.498.99235435137725131221181665085247298501","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.73205916973275504240156555900460720571","series":"1.2.826.0.1.3680043.8.498.26925243648239229819963083722405549166","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73257122462858353041066181004123253930","series":"1.2.826.0.1.3680043.8.498.31054636126570881375178752209438074369","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73314900301724532218347713512381622074","series":"1.2.826.0.1.3680043.8.498.60626044696085585755403022958872427134","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73448420644251136304786298254801920284","series":"1.2.826.0.1.3680043.8.498.13068472100121293288675082114540048664","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73682608760611536443754406204042210115","series":"1.2.826.0.1.3680043.8.498.34425117703828161069141876665526630766","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73194579460722181989151410198309642266","series":"1.2.826.0.1.3680043.8.498.42954787679788576701850405246412133278","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.73511703934549662392765328175356230615","series":"1.2.826.0.1.3680043.8.498.24185627522498470160664893715676328329","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73825252186234755453650759159564715478","series":"1.2.826.0.1.3680043.8.498.50990784220316390809469217007715514676","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73602922823389605007702521335136347605","series":"1.2.826.0.1.3680043.8.498.11232152515030548730978852729127839380","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73891117356545433634964184054304516063","series":"1.2.826.0.1.3680043.8.498.82046558525392297116647101018253711295","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73983081622010008601849687650985260980","series":"1.2.826.0.1.3680043.8.498.12006434320147316220803012996097487703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74095402538454760179339842746572802563","series":"1.2.826.0.1.3680043.8.498.38962366787947856862772567948167087176","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74140806279409854828032496578072554031","series":"1.2.826.0.1.3680043.8.498.42468047346176646806723383776568822544","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74247325306307303036016759070403973520","series":"1.2.826.0.1.3680043.8.498.12674768684671530384696381269236983043","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74243110959141459268838217468465880820","series":"1.2.826.0.1.3680043.8.498.10706904119157612096816580739784317161","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74282280145383357786409764912398881272","series":"1.2.826.0.1.3680043.8.498.73204999765566785248033524498053377492","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73748007182569068263280098473557128226","series":"1.2.826.0.1.3680043.8.498.36915359446980120495099374944779228012","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74394784271003789564066563841806954326","series":"1.2.826.0.1.3680043.8.498.99072092540318807442839474329430118815","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74362364640190722036251935797260325860","series":"1.2.826.0.1.3680043.8.498.76995936838348964315256433350844648814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204","series":"1.2.826.0.1.3680043.8.498.82000133706494980144637539357651413316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74471724874857406578316075456414902954","series":"1.2.826.0.1.3680043.8.498.94720769982902124833463096257256761914","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74714018385278253151175853998945654373","series":"1.2.826.0.1.3680043.8.498.87645552318874307353920489616595827087","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74864145321761783409935288797327597738","series":"1.2.826.0.1.3680043.8.498.65900635762254123805885675950502114903","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74914646210033032631540802996162814268","series":"1.2.826.0.1.3680043.8.498.14600188276380252737041609765700390202","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75168480371573015758963049782687952716","series":"1.2.826.0.1.3680043.8.498.33175037533438804952879631219403198406","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74940598515597814600837126537274199312","series":"1.2.826.0.1.3680043.8.498.18204311483922757099337685198153728008","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75378323910171368568360074903297446681","series":"1.2.826.0.1.3680043.8.498.13307729860672197867066293071167936266","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75518253718111135068488417046278011303","series":"1.2.826.0.1.3680043.8.498.21450191861413509861211568935024698089","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75276563505367629340309025030148269450","series":"1.2.826.0.1.3680043.8.498.24083787154979455025820530689092893083","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190","series":"1.2.826.0.1.3680043.8.498.93551484318140274266459267320072605992","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75523541256169382504121567002441615798","series":"1.2.826.0.1.3680043.8.498.11194931413038294839277930348278802468","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75605038901688937188489854019478082759","series":"1.2.826.0.1.3680043.8.498.10106675448892620006752329358004847280","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75639423607035582240174986752280730347","series":"1.2.826.0.1.3680043.8.498.18046295966421427847641861448646209253","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75518253718111135068488417046278011303","series":"1.2.826.0.1.3680043.8.498.11098571561481946608797473584574827015","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75548650589349966221875964542604002971","series":"1.2.826.0.1.3680043.8.498.93380513379297637770982274418857310425","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75605038901688937188489854019478082759","series":"1.2.826.0.1.3680043.8.498.95148968774435651663271018788600080505","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75688484058613633923943882730405145535","series":"1.2.826.0.1.3680043.8.498.10855836314769251601288863065686541431","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75639423607035582240174986752280730347","series":"1.2.826.0.1.3680043.8.498.57022160575648381582689792095286920060","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75893915698353338311947268382839152877","series":"1.2.826.0.1.3680043.8.498.62437227160527252902858229407047757742","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75742195884147990442829552992649204078","series":"1.2.826.0.1.3680043.8.498.29526295681061650310181811505369966009","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75773447415943537938555952303103557864","series":"1.2.826.0.1.3680043.8.498.51617475805298849002445007648174109604","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76093766204928582819565636658977147523","series":"1.2.826.0.1.3680043.8.498.10765186065678532160249417183955148851","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76027799502777251766328820405966979386","series":"1.2.826.0.1.3680043.8.498.10107178824288925541943025533677616645","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75751784078102748563815298069799859542","series":"1.2.826.0.1.3680043.8.498.23979390851836866765961007179193688079","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.76122127303562588545128053661609574837","series":"1.2.826.0.1.3680043.8.498.98291179156152245398289998234662621165","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76222770668660274805993655800271653712","series":"1.2.826.0.1.3680043.8.498.77072244963833729976409350390605844287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76122127303562588545128053661609574837","series":"1.2.826.0.1.3680043.8.498.20089779260780487923500296703080005851","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75980374267002793040832157122311396174","series":"1.2.826.0.1.3680043.8.498.41966663344817964262517489160178828985","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76350662767360412185091180512006419379","series":"1.2.826.0.1.3680043.8.498.11496388328656249236457113897484779959","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76281930374588438789344170829280438697","series":"1.2.826.0.1.3680043.8.498.11316051418338834504545273651337232152","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76514504104121634569792846980451896030","series":"1.2.826.0.1.3680043.8.498.83459826600516305978207961132246739691","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76651080491383110752977286124044311636","series":"1.2.826.0.1.3680043.8.498.83784700912299442521264658857968151914","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76680092665218514711300086707222109323","series":"1.2.826.0.1.3680043.8.498.12430027056839893949837688130167884582","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76903562636417038907113894538863784737","series":"1.2.826.0.1.3680043.8.498.63112731921231190680399618306510314804","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76993922254007307143291254174266755701","series":"1.2.826.0.1.3680043.8.498.69821789247282628496611675579208063469","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76841077743272729415666709641073096909","series":"1.2.826.0.1.3680043.8.498.12175740666985223056107607671355876372","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77037121474756969001683126693104350509","series":"1.2.826.0.1.3680043.8.498.12761501371002041139663943225109517505","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76914908131203373740345988718674850939","series":"1.2.826.0.1.3680043.8.498.73919372051804568358688467137260265719","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77060858802024387619459645469871885412","series":"1.2.826.0.1.3680043.8.498.74280316017614256150976533990811874040","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77164129615096222757740198026718214733","series":"1.2.826.0.1.3680043.8.498.13149883948772538380701678222279639612","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77037121474756969001683126693104350509","series":"1.2.826.0.1.3680043.8.498.42817014427462541055603085262215678501","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77215476384973989055525592556183934827","series":"1.2.826.0.1.3680043.8.498.88843957487586326950967944854999975727","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77215476384973989055525592556183934827","series":"1.2.826.0.1.3680043.8.498.10617153966080303206956655554595198888","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77261014467268552758826551926128217025","series":"1.2.826.0.1.3680043.8.498.23127779601923449366206637244676359990","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77283213969268676660453937180363826917","series":"1.2.826.0.1.3680043.8.498.32508264269837329551097887281787091281","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77283213969268676660453937180363826917","series":"1.2.826.0.1.3680043.8.498.13104165986411687843343175579690913727","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77290477043075828139630734968573461136","series":"1.2.826.0.1.3680043.8.498.40214039967300108887636757058357283874","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77306755058630855443644908572157134062","series":"1.2.826.0.1.3680043.8.498.47639961151070940652874471671349010116","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003","series":"1.2.826.0.1.3680043.8.498.78258700914724339615095082290322844598","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003","series":"1.2.826.0.1.3680043.8.498.11789987658206151165631502413155474798","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77378803863827312829317008636359414012","series":"1.2.826.0.1.3680043.8.498.12509695703732158342282005635643350237","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77416382267531567401698555072875486512","series":"1.2.826.0.1.3680043.8.498.71805342511900542452370329506031278555","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77433065090757862487482575794398458402","series":"1.2.826.0.1.3680043.8.498.24739093036519711258999230622190511814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77513348456432770689938407624768928320","series":"1.2.826.0.1.3680043.8.498.11916129071086491056021597522329253229","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77577135626047243581253857225991426727","series":"1.2.826.0.1.3680043.8.498.34711195917739588444595321905995402101","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77433065090757862487482575794398458402","series":"1.2.826.0.1.3680043.8.498.69506505647776954385374600297464413231","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77626578713099735143745396709359607445","series":"1.2.826.0.1.3680043.8.498.60143347930555299752247887975033513736","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77716260980321491689913730053309892120","series":"1.2.826.0.1.3680043.8.498.15853849771217582096739735020636314883","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77498121906981512168120474112422492971","series":"1.2.826.0.1.3680043.8.498.66213856727957564772704255694607426308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77740672313593763051900703940745812581","series":"1.2.826.0.1.3680043.8.498.18796958603702754179242560413378582661","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77760453966349959552328150395676117417","series":"1.2.826.0.1.3680043.8.498.10422157181074909976688773730576999418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77740672313593763051900703940745812581","series":"1.2.826.0.1.3680043.8.498.11114438654149519912156377368989428275","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77801860555102174958520570751272642387","series":"1.2.826.0.1.3680043.8.498.53694457980434257041384412165458503873","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77593859518024366170735307975173961777","series":"1.2.826.0.1.3680043.8.498.12286395481155776691859658904164533047","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77893704208186903913532950495036889026","series":"1.2.826.0.1.3680043.8.498.10872005957091797807039260879648155982","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77843765997593346714211135369376676497","series":"1.2.826.0.1.3680043.8.498.14537597152983416098187219169872488917","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77933128806366660573360274958258917742","series":"1.2.826.0.1.3680043.8.498.93435326478599563896562827715787345099","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77933128806366660573360274958258917742","series":"1.2.826.0.1.3680043.8.498.13670229115015396985784595445422020257","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.78012314847102407375138809615977711653","series":"1.2.826.0.1.3680043.8.498.13008015395074886554658244032093917109","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78012314847102407375138809615977711653","series":"1.2.826.0.1.3680043.8.498.29075792787363932864889586742377503240","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.78084594560404633421210837175859618437","series":"1.2.826.0.1.3680043.8.498.44673210757772776101594814355444588537","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78236036694398253902871510456985685293","series":"1.2.826.0.1.3680043.8.498.92438382555971945162881922271399336998","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78188863861812651079751164964756388411","series":"1.2.826.0.1.3680043.8.498.37874691952016285472892559731121125593","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78476873086705419136436170130940915568","series":"1.2.826.0.1.3680043.8.498.13101524764059595716328033792804524360","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78296478954955171036523274995555716268","series":"1.2.826.0.1.3680043.8.498.52107487346247804232111871748008979893","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78282552802075313852355443512555224586","series":"1.2.826.0.1.3680043.8.498.13230367408177330010209362415519933593","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78521396404736679127326025240279747196","series":"1.2.826.0.1.3680043.8.498.53351551074089569967518897008551700762","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78667999747216717251638754803697959701","series":"1.2.826.0.1.3680043.8.498.67915855945777940161401650432225595432","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052","series":"1.2.826.0.1.3680043.8.498.11209805925237684981503342044595343744","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78079900750417018394321006978343945239","series":"1.2.826.0.1.3680043.8.498.73221684703446793358328974965134611618","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.78808320605816811006382053060951793555","series":"1.2.826.0.1.3680043.8.498.78176092794245958588048841255448253934","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79038153803632889185429988483097161887","series":"1.2.826.0.1.3680043.8.498.61457190924665529503047823518495376009","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78685661415919658264222803849222055515","series":"1.2.826.0.1.3680043.8.498.33037135996590913248676775710644128805","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79080521163097116147379189510104170611","series":"1.2.826.0.1.3680043.8.498.29533992834524158967126571781481536856","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79117047831806895544174734229992359986","series":"1.2.826.0.1.3680043.8.498.63471163738554771627089019310304254812","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79165531588966776246377098630025444815","series":"1.2.826.0.1.3680043.8.498.21676668049071126965759942921489193083","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78853931928960361455025737444267199231","series":"1.2.826.0.1.3680043.8.498.95082199162561012718590100997093835515","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.79225094031423353857449795712402404317","series":"1.2.826.0.1.3680043.8.498.12921195805635550811364487141402856865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79247871688929479411597859576284194165","series":"1.2.826.0.1.3680043.8.498.60117259269781967851866197281490602201","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79505118958475739396320647697583979079","series":"1.2.826.0.1.3680043.8.498.70009451822299279121609993749720777540","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79165531588966776246377098630025444815","series":"1.2.826.0.1.3680043.8.498.56499686866694625282161981189331298152","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942","series":"1.2.826.0.1.3680043.8.498.36898834529003026596255105381017001101","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79551456190098389081551755482042135791","series":"1.2.826.0.1.3680043.8.498.11532754827112896038346820719643503615","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79609204436595306110253502961172135807","series":"1.2.826.0.1.3680043.8.498.22563981180567210481524526103875276850","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79648517065511897282938523065537768426","series":"1.2.826.0.1.3680043.8.498.10338615286687316344246160414194936938","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942","series":"1.2.826.0.1.3680043.8.498.25755074934969872546200523029047101535","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.79725520325008084984331823107016254775","series":"1.2.826.0.1.3680043.8.498.12380431690809899560619025766259112214","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79798779812052605703861699094268016306","series":"1.2.826.0.1.3680043.8.498.64410132725489840253677263673675921323","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80058869639900582578987760750483296668","series":"1.2.826.0.1.3680043.8.498.45657043992357273226721677452918817348","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80146250663015309456767479960653638934","series":"1.2.826.0.1.3680043.8.498.10251284258779540589661026563101041123","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80207056225476761432712249807198836187","series":"1.2.826.0.1.3680043.8.498.12377984937793060485220183983609749210","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79977436705480715175480363175316375576","series":"1.2.826.0.1.3680043.8.498.11943878544195223720125728159332576721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80319546205646617603546887163539306146","series":"1.2.826.0.1.3680043.8.498.53477424301291864424919993540851998946","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80312559977515578735238310472949287111","series":"1.2.826.0.1.3680043.8.498.72090217567502176302926821111639400682","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80391378873630377472192017862302619947","series":"1.2.826.0.1.3680043.8.498.18979249056911895380221252979341779314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80458507774711142534182397228254444268","series":"1.2.826.0.1.3680043.8.498.76005864791787691140225902505176386909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80458507774711142534182397228254444268","series":"1.2.826.0.1.3680043.8.498.18506326813514587243205621531675125727","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.80557835452734362083552169469427388003","series":"1.2.826.0.1.3680043.8.498.12679140843771621047127466728471125277","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80612964209397056653334784776523875647","series":"1.2.826.0.1.3680043.8.498.85733529822693643868048384338888332078","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80629565747112614845746475820152813307","series":"1.2.826.0.1.3680043.8.498.42392777333884830786644281676812540367","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80643480554795312280333659144941545925","series":"1.2.826.0.1.3680043.8.498.13084323451878816167379438588307033143","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80661392139668653552110941896475066204","series":"1.2.826.0.1.3680043.8.498.13083935522965801117655424555281172808","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80735285867775404153100236162962472952","series":"1.2.826.0.1.3680043.8.498.21845561934260309725404434539954197687","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80861108320341696163987804408524915999","series":"1.2.826.0.1.3680043.8.498.63211653683743404705377624506747481551","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80758336123908522542787716219939751704","series":"1.2.826.0.1.3680043.8.498.48418634714338866127373932657817838084","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80931292574057303692838531323879513417","series":"1.2.826.0.1.3680043.8.498.29707052960329851604338563290650414145","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81103809943451790668501089584752866199","series":"1.2.826.0.1.3680043.8.498.12436302417871992264653952906492673938","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81018911907787774214591206085583545057","series":"1.2.826.0.1.3680043.8.498.17248753995787997622627171616633692427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80698510014594772882040509716764819716","series":"1.2.826.0.1.3680043.8.498.31553326482145637113764783420659851047","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81142440700884513521007992957945568924","series":"1.2.826.0.1.3680043.8.498.12354830307920158990659600931604321847","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81342416992724702402763591633702802359","series":"1.2.826.0.1.3680043.8.498.94263995612690845107507613870412964770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81269884425000228604280521306949721859","series":"1.2.826.0.1.3680043.8.498.26749957507791318502604646487919110076","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81351574724227319632426427855692606694","series":"1.2.826.0.1.3680043.8.498.83256269396916900149626237265659865616","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81342416992724702402763591633702802359","series":"1.2.826.0.1.3680043.8.498.10001451031710804784840301648844254892","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81462603621780919719466264356289554669","series":"1.2.826.0.1.3680043.8.498.32621951442871996108012552329548692895","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81629704995225424568187579791704967233","series":"1.2.826.0.1.3680043.8.498.64775921849792713493673731765676948139","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81692996137620934043276253858381812472","series":"1.2.826.0.1.3680043.8.498.64998082693097953609029600267444478489","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81629704995225424568187579791704967233","series":"1.2.826.0.1.3680043.8.498.10899361608375770671667431056077608329","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81738571920712657790861120957086010772","series":"1.2.826.0.1.3680043.8.498.50662716624860950107973769066962655604","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81790990482142045640192729399175106255","series":"1.2.826.0.1.3680043.8.498.11306329132962003688857114435399592921","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81692996137620934043276253858381812472","series":"1.2.826.0.1.3680043.8.498.33161472925160877415643524959927536016","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81796385087779065457668769443782458844","series":"1.2.826.0.1.3680043.8.498.72872027878747496661809027185898300873","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81927409938015158663782061022054347955","series":"1.2.826.0.1.3680043.8.498.86506438995822965018676401023124007000","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81832287065207835637503039077685315236","series":"1.2.826.0.1.3680043.8.498.77496211030168448227100573543841167427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81964000393640770822199916465038759069","series":"1.2.826.0.1.3680043.8.498.11108614151446859362431203806636694098","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81981151447274200345866874172324295172","series":"1.2.826.0.1.3680043.8.498.35826552493392860159507134200674833503","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82074883987134049971242727164506046476","series":"1.2.826.0.1.3680043.8.498.10507146993605706465158309675206728735","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82012532783011843175172575198326783678","series":"1.2.826.0.1.3680043.8.498.95163967175857022513221785372878527737","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81819209302731920404237374320026807282","series":"1.2.826.0.1.3680043.8.498.11550376991779079869320540650085228835","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.82144602300666174450721045102799681426","series":"1.2.826.0.1.3680043.8.498.36194919842596510955209245388670337122","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82272793832748838882752586678639564181","series":"1.2.826.0.1.3680043.8.498.94495603007412142954473041330964331372","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254","series":"1.2.826.0.1.3680043.8.498.30205011752939967279543537379010794909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82498616123349854719801627024073383874","series":"1.2.826.0.1.3680043.8.498.13147190584295199097758938898988524157","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82414828460683289235183375457850096746","series":"1.2.826.0.1.3680043.8.498.57501815741742069746179287802321723865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82858981777405241601610241542173548956","series":"1.2.826.0.1.3680043.8.498.10056911984646058856766964243376225790","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82744514729782879778828512295184306538","series":"1.2.826.0.1.3680043.8.498.11951582938344630781203276192739841071","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254","series":"1.2.826.0.1.3680043.8.498.10018402039504397857968533816812816572","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.82579118531566493365068788437906656162","series":"1.2.826.0.1.3680043.8.498.34082265790297123564700633659272938005","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82858981777405241601610241542173548956","series":"1.2.826.0.1.3680043.8.498.13793753431964742965987199149755377798","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.82925474533853765570570726554869886925","series":"1.2.826.0.1.3680043.8.498.10213522702855644549318723032727986581","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83055401642066749019091368358783700783","series":"1.2.826.0.1.3680043.8.498.94184901983654938070265493727732603017","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83030371526667747898709613600825477966","series":"1.2.826.0.1.3680043.8.498.19635739428844317290418324120194947027","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83000639623679673469640458882929412717","series":"1.2.826.0.1.3680043.8.498.12538064500925848830181754494496429314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83108474538388106770920972885900596508","series":"1.2.826.0.1.3680043.8.498.97906866752714228642116375785635738625","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83253778356746892201320443660102495790","series":"1.2.826.0.1.3680043.8.498.12873356086028632445667675260169597230","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83235374118671754689663732057314083332","series":"1.2.826.0.1.3680043.8.498.35955931361877219512211675270324024511","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83495448827553002152296206085110883602","series":"1.2.826.0.1.3680043.8.498.44451508654259552189685656411529960784","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83376388091077619281568190257019735638","series":"1.2.826.0.1.3680043.8.498.91027913832433079712689892470836838284","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.83253778356746892201320443660102495790","series":"1.2.826.0.1.3680043.8.498.25557406479924895086794592761134442150","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.83747418172021065605877274971789603771","series":"1.2.826.0.1.3680043.8.498.33699222672574848167977614612582297181","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83691358175810149975084604975065122220","series":"1.2.826.0.1.3680043.8.498.67029174950551394667256977933360193715","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83956129905762530953192730040209719910","series":"1.2.826.0.1.3680043.8.498.10962599003400604860239488523801592336","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83691358175810149975084604975065122220","series":"1.2.826.0.1.3680043.8.498.15223450556059808513268992552245338731","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.83569065014916050100714967945484998462","series":"1.2.826.0.1.3680043.8.498.20841604508834566928700995786980434997","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84047134932018909998648460546980598926","series":"1.2.826.0.1.3680043.8.498.12566901385310692651379835554025193445","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84069767112794890061709127671908414399","series":"1.2.826.0.1.3680043.8.498.98978225204650147157083685535791742301","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84230432137191939390680841389370206819","series":"1.2.826.0.1.3680043.8.498.74480389940283871734031880500356295516","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84230432137191939390680841389370206819","series":"1.2.826.0.1.3680043.8.498.49757233778194150033043840758080244617","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84292569246911697141902796190329671261","series":"1.2.826.0.1.3680043.8.498.26434948058778766901134296005379149157","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652","series":"1.2.826.0.1.3680043.8.498.12948020789996463062577357257441366023","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84483080389344581106129884869678629612","series":"1.2.826.0.1.3680043.8.498.47589844573115635635947161400862967336","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84062602050757262792437893691591415560","series":"1.2.826.0.1.3680043.8.498.32443228471846694668100035252254850357","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84709539128629951937294117586370340042","series":"1.2.826.0.1.3680043.8.498.46379645306674958523106528083326484859","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84419297518873367123595080276985526259","series":"1.2.826.0.1.3680043.8.498.13571094474282934367974627224519457400","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482","series":"1.2.826.0.1.3680043.8.498.80825774132333777977638840890997555617","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84855341377363129926967387137216814518","series":"1.2.826.0.1.3680043.8.498.40880644513173049911520157577567668688","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482","series":"1.2.826.0.1.3680043.8.498.11086545566910011247716170962453698871","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066","series":"1.2.826.0.1.3680043.8.498.24474710828019883103282399955578907138","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85036616529391787087213173897728060798","series":"1.2.826.0.1.3680043.8.498.64401634656189212743693926425027215875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85108719393967663738580518985514195538","series":"1.2.826.0.1.3680043.8.498.72405248625771380966902743141599346635","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066","series":"1.2.826.0.1.3680043.8.498.10815112581716407910129044433300788938","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85108719393967663738580518985514195538","series":"1.2.826.0.1.3680043.8.498.11607208573915230192774701469504870513","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85004373913191467860228868467924401547","series":"1.2.826.0.1.3680043.8.498.17085170045815658895380450583731144476","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85294435004325211584698794441083866921","series":"1.2.826.0.1.3680043.8.498.62209313801260400687864127421056585487","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85194002080793317087210934629814349622","series":"1.2.826.0.1.3680043.8.498.80303252606793377011446169877431372120","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85464150757448712711832350617010924327","series":"1.2.826.0.1.3680043.8.498.11617338159312411055222791565081347123","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85397267100597758920280635204297906286","series":"1.2.826.0.1.3680043.8.498.11114464043462381774640824770025243641","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85431902914673414035492907270636012892","series":"1.2.826.0.1.3680043.8.498.64509467710713766029970610389392275782","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85347263026430684291021533170619978114","series":"1.2.826.0.1.3680043.8.498.21852273366232440940105264107329998939","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85566900782101872452970957532250214590","series":"1.2.826.0.1.3680043.8.498.30697992540418304903004738130621300282","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85687351346612079711706269264454944647","series":"1.2.826.0.1.3680043.8.498.68371541544754281087869738280527054693","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85624441072689653093363046226467141762","series":"1.2.826.0.1.3680043.8.498.46485974091860453017519079340625804011","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85687351346612079711706269264454944647","series":"1.2.826.0.1.3680043.8.498.19947391893023696730622359751892838453","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85719235937472088594405415004352845221","series":"1.2.826.0.1.3680043.8.498.49893436904175863101120702742306439555","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85735729179216019275685564112535593988","series":"1.2.826.0.1.3680043.8.498.89504731996576821954046411050092720536","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85890890545815190189345870912381925290","series":"1.2.826.0.1.3680043.8.498.53416920592541912226331371042841973033","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85882078780717905717989373451578844863","series":"1.2.826.0.1.3680043.8.498.80149167635600194368066719697797970856","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85760360279564672425828346507309104394","series":"1.2.826.0.1.3680043.8.498.61150618359313203258636972954206733931","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85870577082315172424837183881210532800","series":"1.2.826.0.1.3680043.8.498.56476586152889649918753901981217578832","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85976668143927055109184433855426232055","series":"1.2.826.0.1.3680043.8.498.66495562601847189130089461141940187316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86029947807373604003071651721618330827","series":"1.2.826.0.1.3680043.8.498.59992515998173487364479424415989744099","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85943737587102032370951456635167782343","series":"1.2.826.0.1.3680043.8.498.10280565840456588930835406767507889712","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85976668143927055109184433855426232055","series":"1.2.826.0.1.3680043.8.498.10203913153895404444730787301598588741","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86251314013737658351649446042801810374","series":"1.2.826.0.1.3680043.8.498.72818731892123246955697759965748490389","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86273853115722509980053190345047859057","series":"1.2.826.0.1.3680043.8.498.81178167153820994479027949421814125457","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86351141114604781108401133410934252655","series":"1.2.826.0.1.3680043.8.498.77338707475453874744207452790588218053","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86409878732405036646840908998274047391","series":"1.2.826.0.1.3680043.8.498.32800735087079563814684465746394538725","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86067849233330188988240710585112183308","series":"1.2.826.0.1.3680043.8.498.87541031377110409635053457453321840377","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86332035495941070455209594003499488889","series":"1.2.826.0.1.3680043.8.498.54787257581258866147034143601046699491","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251","series":"1.2.826.0.1.3680043.8.498.75490224058819385453983161734150451147","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86532643576908316499388520854820002133","series":"1.2.826.0.1.3680043.8.498.13389488580276587676453301319647459459","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86733565200815744326667230375642936291","series":"1.2.826.0.1.3680043.8.498.13382831330123363166859944796847946545","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86881633316011036335786865282908198831","series":"1.2.826.0.1.3680043.8.498.69320418525937827171503970573469308386","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86974853023888933977090539174537083409","series":"1.2.826.0.1.3680043.8.498.73833058690141197304115351217603262814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86733565200815744326667230375642936291","series":"1.2.826.0.1.3680043.8.498.28737992266179111425351874951901725319","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86435890312227285216810145658548164773","series":"1.2.826.0.1.3680043.8.498.89767874283411320208589587786038173712","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87018440624359538250658207994111333051","series":"1.2.826.0.1.3680043.8.498.10061018232179518089734137452530499136","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87064372531225943854092598173945043861","series":"1.2.826.0.1.3680043.8.498.50383782510102936025395657048635862759","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87102698749288269439517333660896096520","series":"1.2.826.0.1.3680043.8.498.10131822381875504921387974445796750260","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87253761427012478584873622994909841751","series":"1.2.826.0.1.3680043.8.498.11500009810356170205611549501131923131","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87198191797585172912979594638355685739","series":"1.2.826.0.1.3680043.8.498.96131416298582963128783630031245606980","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87253761427012478584873622994909841751","series":"1.2.826.0.1.3680043.8.498.28332722459390086443700316489641485808","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171","series":"1.2.826.0.1.3680043.8.498.20217532227581655328539723777325542116","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171","series":"1.2.826.0.1.3680043.8.498.88919541785385002414903531908299271460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87563449504927531224663601218159863518","series":"1.2.826.0.1.3680043.8.498.39112969435218845529032392985573641806","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87214831397290763080243320084625847148","series":"1.2.826.0.1.3680043.8.498.72937567078307507484768379998518308017","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87550037046735743504446944390018580426","series":"1.2.826.0.1.3680043.8.498.12892729042628830626553305517261167274","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87650297321648368205324227429768256110","series":"1.2.826.0.1.3680043.8.498.15966648716022679940788003643309543814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87563449504927531224663601218159863518","series":"1.2.826.0.1.3680043.8.498.56282299025671023058643630874958938493","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87680825247567493080163504511649789348","series":"1.2.826.0.1.3680043.8.498.12144325867630537047963938440779454987","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87797202230328868466634480198668365648","series":"1.2.826.0.1.3680043.8.498.31347934847681914035852198738713758832","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87845297884497964690686366000533570840","series":"1.2.826.0.1.3680043.8.498.15531741542871951806966838995551669185","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87900330305833722536121052961334699743","series":"1.2.826.0.1.3680043.8.498.39920858312928249558084371262238193550","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87761573479019057997334508266888503487","series":"1.2.826.0.1.3680043.8.498.25300898727036340186893896779808410901","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88022456207467042544403752144674777523","series":"1.2.826.0.1.3680043.8.498.10339128212959657604888704813541605732","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435","series":"1.2.826.0.1.3680043.8.498.10326918552454411368494543557296881553","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88466268868654468582661984391239692466","series":"1.2.826.0.1.3680043.8.498.74179315631369412454565579166912906525","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88466268868654468582661984391239692466","series":"1.2.826.0.1.3680043.8.498.64955293682204330582309234445604132190","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.88565389747258713872180859306907392080","series":"1.2.826.0.1.3680043.8.498.10402131223843888288334718856087780792","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88274753211076645976496220746740420894","series":"1.2.826.0.1.3680043.8.498.26965219198821138441199110640759097000","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88571985843649843469694556575325271560","series":"1.2.826.0.1.3680043.8.498.10182079484010866336352024084895732044","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88653085234639096911379842849193874077","series":"1.2.826.0.1.3680043.8.498.13118700326506228618964397308107263159","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88679328523895187500714090963655363787","series":"1.2.826.0.1.3680043.8.498.51070716672988366376622479595569699309","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88726804476694411363564819498296119763","series":"1.2.826.0.1.3680043.8.498.10526727703711025722578792374886894488","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88609906210780568637209291182572648571","series":"1.2.826.0.1.3680043.8.498.43179406803085529058831531019275803292","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88896246957673838955012057210098034177","series":"1.2.826.0.1.3680043.8.498.12031239741428324037895571470034957240","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88872155446935518174304990308380906112","series":"1.2.826.0.1.3680043.8.498.50962363226949753622288847652705843630","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88753415430913291809543943690942744997","series":"1.2.826.0.1.3680043.8.498.12268546440883267953527728350857826273","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89026102874017923932934673207699042800","series":"1.2.826.0.1.3680043.8.498.65297547223639681711818565897803744951","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89026102874017923932934673207699042800","series":"1.2.826.0.1.3680043.8.498.11061248166953059964701105014673035599","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89048907638162719840222508855779172366","series":"1.2.826.0.1.3680043.8.498.81004829384088362603071917065957316030","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89048907638162719840222508855779172366","series":"1.2.826.0.1.3680043.8.498.10443980500881724793181652454399951341","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88753415430913291809543943690942744997","series":"1.2.826.0.1.3680043.8.498.79840446333263648104839621457831341695","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89110271437785174288044757208973128908","series":"1.2.826.0.1.3680043.8.498.62018716226369535010620260506531423256","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89190511197564127079697151213577962649","series":"1.2.826.0.1.3680043.8.498.11695344721391685327217675783426604913","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89302191571937190529967168726521003428","series":"1.2.826.0.1.3680043.8.498.34209961107909519435383762795660114661","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89326347774248795083360112232993521530","series":"1.2.826.0.1.3680043.8.498.71936333704787073723476705484433798089","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89110271437785174288044757208973128908","series":"1.2.826.0.1.3680043.8.498.10683414454273810085173686414614693971","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89357823844365374776758395347764601030","series":"1.2.826.0.1.3680043.8.498.30133596071525793612266168031258338764","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89346975863478164334871204856160071946","series":"1.2.826.0.1.3680043.8.498.11580735692366186501390685933687226909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89423600201830685256510321877396547497","series":"1.2.826.0.1.3680043.8.498.36608150962215396262441099387785638537","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89464179959474561696319051639748844244","series":"1.2.826.0.1.3680043.8.498.37371162368578026770768748237545355459","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89576499383950909889665633610791093334","series":"1.2.826.0.1.3680043.8.498.94344730193093118761394608977460474202","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89590880980421560369495600204866789646","series":"1.2.826.0.1.3680043.8.498.10930927799692209031534544159242228546","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89729486637365590877469535082675069395","series":"1.2.826.0.1.3680043.8.498.12504345164525907272058721660204761685","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89423600201830685256510321877396547497","series":"1.2.826.0.1.3680043.8.498.44585219108984688315594303186872384905","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89829843399218627892046362324057240535","series":"1.2.826.0.1.3680043.8.498.12276557310495585517870872891996770026","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89857801102910604766959864464865910585","series":"1.2.826.0.1.3680043.8.498.80206670618462105500381498860362387022","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90057130466053377217652879986591534745","series":"1.2.826.0.1.3680043.8.498.12600446116142828596509105791894997369","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89967772422936726289461201791871771011","series":"1.2.826.0.1.3680043.8.498.65994939942196703776845322053746029468","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90003472003278237920648198243464374280","series":"1.2.826.0.1.3680043.8.498.37939901699599309335587174426676564917","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89857801102910604766959864464865910585","series":"1.2.826.0.1.3680043.8.498.13437184027839750122198191832357293057","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.90172333170788218623219231170360850947","series":"1.2.826.0.1.3680043.8.498.32947317900404253628349635544686299367","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90264300606047579088362539636050224478","series":"1.2.826.0.1.3680043.8.498.11826580423130736642070604555322620024","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90227414485305096825508911050013279704","series":"1.2.826.0.1.3680043.8.498.24588981968822714362346239896557245406","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90313529150165069133283531436628039942","series":"1.2.826.0.1.3680043.8.498.14825369532658473697632889010354474927","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90288598546566291188788375261370919222","series":"1.2.826.0.1.3680043.8.498.68825013301983908544465847600886327354","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90274711315692820326973277297753398634","series":"1.2.826.0.1.3680043.8.498.10424430860040158579336427029862115408","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90433488790413321102007716267340900056","series":"1.2.826.0.1.3680043.8.498.12119038963128830836301952224302332475","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90340114291852810896219650388952790164","series":"1.2.826.0.1.3680043.8.498.74021936627911165906674164883736387402","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90490891782778285565462083536468192410","series":"1.2.826.0.1.3680043.8.498.29235529133284658373259653136047814314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293","series":"1.2.826.0.1.3680043.8.498.45124647116330655844115942870681640148","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90573769367031841574114648773801425781","series":"1.2.826.0.1.3680043.8.498.12176139889509434951130748907630069123","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293","series":"1.2.826.0.1.3680043.8.498.10604473055133273149151774796247402968","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.90466423642678253933409966898120881010","series":"1.2.826.0.1.3680043.8.498.84166593982084294378070669700075462657","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90668646188784648688693620159738574680","series":"1.2.826.0.1.3680043.8.498.46585491643085331988722703348452483500","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90751504670572401790063555695835439317","series":"1.2.826.0.1.3680043.8.498.10806697524052310064438860955417177545","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90976227335921831531105845602655777296","series":"1.2.826.0.1.3680043.8.498.10095489287070547112103322013518017221","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90764713690393618577990540005126771712","series":"1.2.826.0.1.3680043.8.498.39875925137244170078812776650286887284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90668646188784648688693620159738574680","series":"1.2.826.0.1.3680043.8.498.13246607698983641560129888931264283105","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.91064647744940506223536204418057554573","series":"1.2.826.0.1.3680043.8.498.11313820274172413803691377391333426404","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91122873808284731048016330734204992698","series":"1.2.826.0.1.3680043.8.498.68731028511941089241983696629901269138","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91168270990167204047014161851340054645","series":"1.2.826.0.1.3680043.8.498.63880000435741232032702977136566723577","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91125742927944703615970502025043944089","series":"1.2.826.0.1.3680043.8.498.11920052156775850630715515266315411180","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91346221361264140052760282748336176394","series":"1.2.826.0.1.3680043.8.498.97191915009073287261467823753909738143","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91220255464004258635083940797356822178","series":"1.2.826.0.1.3680043.8.498.40226447511462762737588662749248996024","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91380337141071062166571333367919885984","series":"1.2.826.0.1.3680043.8.498.88421313148497313021719003634381533418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91391446167553025113367459399831345295","series":"1.2.826.0.1.3680043.8.498.10492934695106070473586113335057668771","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91468214041781427810665883790357021637","series":"1.2.826.0.1.3680043.8.498.14016766690825227020335239619646027878","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91323765268136520804399328350809179765","series":"1.2.826.0.1.3680043.8.498.41514225220149391394897154395127935565","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91499489440453244654238202136675083579","series":"1.2.826.0.1.3680043.8.498.23785450805727872389118576071634314819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91727248305411866145131034915485728037","series":"1.2.826.0.1.3680043.8.498.65940652757551612459730200070763092517","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91660278231317256353374962767408532443","series":"1.2.826.0.1.3680043.8.498.33383947227249919552845156809774641353","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91913894091933783677568170210064807665","series":"1.2.826.0.1.3680043.8.498.10331337904364362223730809416231577770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91897419008563027309752122760214275182","series":"1.2.826.0.1.3680043.8.498.37913458830377326594647766447993859741","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91468214041781427810665883790357021637","series":"1.2.826.0.1.3680043.8.498.11181025571146209360935309996927283015","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.92071199603599654827155753204585082303","series":"1.2.826.0.1.3680043.8.498.35841406816793769153952500834913848829","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016","series":"1.2.826.0.1.3680043.8.498.13343960847951042335151765410452308495","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92127426417720107708158713612888411025","series":"1.2.826.0.1.3680043.8.498.97997487775999652191797778045822305349","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92386519544130078776672685639580497728","series":"1.2.826.0.1.3680043.8.498.13108125610102826319317251512708374037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92301915122994899889226415477164044164","series":"1.2.826.0.1.3680043.8.498.35381246584622156114980955077720060488","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016","series":"1.2.826.0.1.3680043.8.498.11018061973861079072736753223379511583","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745","series":"1.2.826.0.1.3680043.8.498.11336613741310546076404814266244472577","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92626490033376759309712417622692944095","series":"1.2.826.0.1.3680043.8.498.11306295453589938021990765462368989820","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92581137608969847756039955144478716506","series":"1.2.826.0.1.3680043.8.498.39616911482773847334483864196956612360","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92655291659311409459093330375014445330","series":"1.2.826.0.1.3680043.8.498.12629288281741626188383805937315143682","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745","series":"1.2.826.0.1.3680043.8.498.74243763118970753834389206474439561602","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.92715232057122971236003034574127234433","series":"1.2.826.0.1.3680043.8.498.33620833092004492608106647909390017547","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92758635328468215409701857842011454608","series":"1.2.826.0.1.3680043.8.498.10262159277771764098654661323118689739","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055","series":"1.2.826.0.1.3680043.8.498.70649239796527400722399907758429026565","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92854445271746966212168817610305595078","series":"1.2.826.0.1.3680043.8.498.12871977925779305440243735750930061290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055","series":"1.2.826.0.1.3680043.8.498.11945432383342102965348605833257124787","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.93141871723058796160969424330424882014","series":"1.2.826.0.1.3680043.8.498.12655269587680545610271021841473899302","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93182602591865019295859776146674822677","series":"1.2.826.0.1.3680043.8.498.79699901089415742413404456747117749427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93344237572402384894445814841725878706","series":"1.2.826.0.1.3680043.8.498.69268597642480711940957758189393965718","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93222021909191248270818335069738080963","series":"1.2.826.0.1.3680043.8.498.78450875181134717328009324295953749755","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93385433312281352893457225415812880948","series":"1.2.826.0.1.3680043.8.498.13341521410256326813903015962232847425","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93054327536380380487760790221613186305","series":"1.2.826.0.1.3680043.8.498.40590250646909543190672186024486706836","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.93301337627536436086612556500687157032","series":"1.2.826.0.1.3680043.8.498.81988624245965590770122638353071698525","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.93653421810951484288813767943453706233","series":"1.2.826.0.1.3680043.8.498.71870238318207165071849293437988989788","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93722352425168280299901366261374293722","series":"1.2.826.0.1.3680043.8.498.63745190551153949540496571911559687856","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93921993164883084178408030385007465064","series":"1.2.826.0.1.3680043.8.498.46053977402302229106254332733879544969","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93747304211180517979657679136059719120","series":"1.2.826.0.1.3680043.8.498.19827244834297437246470153444076951118","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93848205752465320137433228658870667418","series":"1.2.826.0.1.3680043.8.498.22377476551065187183070072370363779409","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93555726419626687118723083838655757151","series":"1.2.826.0.1.3680043.8.498.48336846076682721579398272466824762812","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94027547274004055924993074754507802303","series":"1.2.826.0.1.3680043.8.498.56581023620358020414657452470495331824","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94069670011071682445520414370532541612","series":"1.2.826.0.1.3680043.8.498.42540465894547225058377872560412813180","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93985093305024344873337249433396141838","series":"1.2.826.0.1.3680043.8.498.12164498489874517556406576130970466666","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94296943940827583101208235208463322140","series":"1.2.826.0.1.3680043.8.498.10779618068700868413205443781177875879","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94393348753936295687030819683853459606","series":"1.2.826.0.1.3680043.8.498.99451239771402863798163062983712186463","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94335780236370744781443631231792792083","series":"1.2.826.0.1.3680043.8.498.11813590797358826168071384923647519224","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94553664348118184200007654265863262684","series":"1.2.826.0.1.3680043.8.498.53753608720667242905303121168343302573","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94244885518758327449236878653096161475","series":"1.2.826.0.1.3680043.8.498.54432398805474937978935731693377674356","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94607524665407029611795868588964734480","series":"1.2.826.0.1.3680043.8.498.10336961431591119011091249060412966585","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94553664348118184200007654265863262684","series":"1.2.826.0.1.3680043.8.498.13306075200739671927704715029730147076","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94677297888135961827159679484207792618","series":"1.2.826.0.1.3680043.8.498.40517785234021193764786206681499758648","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94729634553835820944505159643192054175","series":"1.2.826.0.1.3680043.8.498.93901156442969248292717514004654456994","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94822220672465914426995315295921716764","series":"1.2.826.0.1.3680043.8.498.60045735733196352701999425431282498875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442","series":"1.2.826.0.1.3680043.8.498.11402094225973310242779488202591581754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94635110520016467882028406590905441740","series":"1.2.826.0.1.3680043.8.498.53146770833545422553795476834234561551","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94774472240035721003264951868030913589","series":"1.2.826.0.1.3680043.8.498.13335202991842748692366032759124750149","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442","series":"1.2.826.0.1.3680043.8.498.25239358560098660131109110275059837125","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95060380238431209646600378585422128745","series":"1.2.826.0.1.3680043.8.498.26169321183228070047240426969401614661","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95142886737239438773421088967802136318","series":"1.2.826.0.1.3680043.8.498.63270633372361767201067851098078651295","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95196756237527666678816095708949563380","series":"1.2.826.0.1.3680043.8.498.51588551017390676795340332424605064430","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95321750682026667237717880394536750835","series":"1.2.826.0.1.3680043.8.498.67885736434421530167603900123023770190","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94954489705497211843190734684710878341","series":"1.2.826.0.1.3680043.8.498.32333086551619757435278507389561127511","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95376417067109020835342111882595126109","series":"1.2.826.0.1.3680043.8.498.15372202856667758888928356664094666088","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95256518245659249144569318023469597946","series":"1.2.826.0.1.3680043.8.498.86536433263504830053662120905968528909","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95555755747044090477918262640906805885","series":"1.2.826.0.1.3680043.8.498.11216583101535872247673097383757844168","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95570207430778332247200880037915693375","series":"1.2.826.0.1.3680043.8.498.13063663166685726649208874621379496352","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95464807346156547985763374990661923815","series":"1.2.826.0.1.3680043.8.498.57964960232336523011845134852370066253","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95705755527332814775935856896567782522","series":"1.2.826.0.1.3680043.8.498.62610738826530176020932306207670579745","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95588576819758266413765165887081039863","series":"1.2.826.0.1.3680043.8.498.97812812884402305079043279910226378003","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95727100148189624148200652225439854460","series":"1.2.826.0.1.3680043.8.498.11841395465807846192188377788880507756","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95777611643095956714301310992151723603","series":"1.2.826.0.1.3680043.8.498.23521008752845241327110101328862833142","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95850844945129153291759207212815927078","series":"1.2.826.0.1.3680043.8.498.15883051124256861363159912164108640600","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916","series":"1.2.826.0.1.3680043.8.498.10873452264345909510827388652681331335","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95918254432056542701730614049432316728","series":"1.2.826.0.1.3680043.8.498.11507446989622545459192974104729976530","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916","series":"1.2.826.0.1.3680043.8.498.10615333233251747030601663920740833365","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96158169859491339496094295508194547199","series":"1.2.826.0.1.3680043.8.498.90329165803497361417571658749000250156","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96158169859491339496094295508194547199","series":"1.2.826.0.1.3680043.8.498.92565540815571060151431599460806588850","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96448229695053562621553132798380456420","series":"1.2.826.0.1.3680043.8.498.21306960528768727266075001710085401857","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96293531288862652713666056362328893428","series":"1.2.826.0.1.3680043.8.498.27254860014884070203333933481210457543","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96561978771090020509310535265236331946","series":"1.2.826.0.1.3680043.8.498.66955923697534070352085755661006202677","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96561978771090020509310535265236331946","series":"1.2.826.0.1.3680043.8.498.10215373994982244069875252005804351235","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96564709711765982786409987515735443801","series":"1.2.826.0.1.3680043.8.498.32946887228488740981740017928444177745","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96614828054790815108190478795539254570","series":"1.2.826.0.1.3680043.8.498.87996768261527369042819027522754992646","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96818533353374094273022322108861564926","series":"1.2.826.0.1.3680043.8.498.10428446043070032107590589193895865706","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96944926470854551064948200936284887742","series":"1.2.826.0.1.3680043.8.498.96916049578120287025288423314240574302","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96950011369238410949579095568317214261","series":"1.2.826.0.1.3680043.8.498.49856074684723163314188932278352601110","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97042446513400958172996637321308032556","series":"1.2.826.0.1.3680043.8.498.36288401335593644898563054343333091999","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96657465815875587552665111166265086308","series":"1.2.826.0.1.3680043.8.498.88759047486975239517379387765045225832","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96944926470854551064948200936284887742","series":"1.2.826.0.1.3680043.8.498.10678013207985010271532548296745045383","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97087958215554026148585108910609248523","series":"1.2.826.0.1.3680043.8.498.38120645547388638052175289796502769428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97220604868276521004168045790587997687","series":"1.2.826.0.1.3680043.8.498.29782639754905522453285295875787544867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97248397528214121944398119979516268902","series":"1.2.826.0.1.3680043.8.498.90760449851947234877589521762844690352","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97135552871905937467775630206570358085","series":"1.2.826.0.1.3680043.8.498.95334590582655798992228516336719312305","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97248397528214121944398119979516268902","series":"1.2.826.0.1.3680043.8.498.58877976317578758398442088136405526052","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97348112008590316355392924312704081684","series":"1.2.826.0.1.3680043.8.498.12886698767826739293263527282880624445","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97485476951995325021464683747847126869","series":"1.2.826.0.1.3680043.8.498.12847743113855742782109137769831697327","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662","series":"1.2.826.0.1.3680043.8.498.12306898793869915364383953312846266667","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97496319463700021339637597270948309369","series":"1.2.826.0.1.3680043.8.498.25088566792964463766562356301938904722","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97520184067825971918316082134210579391","series":"1.2.826.0.1.3680043.8.498.67481737394483300868610391791476990225","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97496319463700021339637597270948309369","series":"1.2.826.0.1.3680043.8.498.16758853006497098427910475062504821725","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97520184067825971918316082134210579391","series":"1.2.826.0.1.3680043.8.498.10119208797992146649409853558226502365","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97689748704772735695349883980413529091","series":"1.2.826.0.1.3680043.8.498.12739078443617777692476602089301541013","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97564564380731258764291146761742841527","series":"1.2.826.0.1.3680043.8.498.72494370606119148800334095158780932285","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97696436660690920809826140686268769509","series":"1.2.826.0.1.3680043.8.498.10265126837592905636369238529672028708","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97731968872354496470096174559789586056","series":"1.2.826.0.1.3680043.8.498.10719818108060719105431094671275269391","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97849030682292314880463675622456900933","series":"1.2.826.0.1.3680043.8.498.11254064448148853027672991420341256773","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97731968872354496470096174559789586056","series":"1.2.826.0.1.3680043.8.498.16886945038251896783044681883815984886","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97797693721516872008302328638374252668","series":"1.2.826.0.1.3680043.8.498.22455908897950711378087994996517364775","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97944261713902372188894456162512065499","series":"1.2.826.0.1.3680043.8.498.46066382287111309319960146442580223853","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98028018520101798863795620289151810642","series":"1.2.826.0.1.3680043.8.498.17424628130711114814131643315897602627","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97944261713902372188894456162512065499","series":"1.2.826.0.1.3680043.8.498.55432596409955791082538068881886502680","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98102475310924339032647376491442498974","series":"1.2.826.0.1.3680043.8.498.10787768731062469786059516313700264424","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98146155044178593810385943365143374814","series":"1.2.826.0.1.3680043.8.498.31145361798136458903283678003595838792","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98064941971367073978404461105852658265","series":"1.2.826.0.1.3680043.8.498.20375159001261904582329827106763725530","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98333658788330318335997032513277632526","series":"1.2.826.0.1.3680043.8.498.18509622150647189320886581028259321892","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98181040478767526925214101502401448074","series":"1.2.826.0.1.3680043.8.498.10302285978260813179259685440784145651","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98351891935507349137982199658103205287","series":"1.2.826.0.1.3680043.8.498.12518984357265256809227906755565766807","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98581537941208552065033680853629122764","series":"1.2.826.0.1.3680043.8.498.61323586466521802116408482256596179062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98333658788330318335997032513277632526","series":"1.2.826.0.1.3680043.8.498.54769010889792163797485985524342014971","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98454206323401382140655387224146512225","series":"1.2.826.0.1.3680043.8.498.98440274254760510687660632317020920984","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98781117432577714521231232440503778261","series":"1.2.826.0.1.3680043.8.498.54477836812200948029283139446614584038","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98781117432577714521231232440503778261","series":"1.2.826.0.1.3680043.8.498.32259642468465962513345235402044969322","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98890851347757949328159691828768030366","series":"1.2.826.0.1.3680043.8.498.43730653114557935285560193112707390418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98676079331571304021387833410440136393","series":"1.2.826.0.1.3680043.8.498.91132362994230440055419142012417114120","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98958651320983196982606333548522846934","series":"1.2.826.0.1.3680043.8.498.11641597169026035810166608388029353143","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98871650330162988912501872031191530068","series":"1.2.826.0.1.3680043.8.498.11874714274705575951774619993461731072","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99077241529686301278201179461896593659","series":"1.2.826.0.1.3680043.8.498.16112574584701202383065705160509289323","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99108147913292332418681050058312852494","series":"1.2.826.0.1.3680043.8.498.33306791204492974988927084749008845118","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99077241529686301278201179461896593659","series":"1.2.826.0.1.3680043.8.498.12976264252535354068376587854559718055","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99243902497374852166139435719549686908","series":"1.2.826.0.1.3680043.8.498.57243907673573292081252677002805586466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99303079292320085374389086609761974820","series":"1.2.826.0.1.3680043.8.498.24700385801146576572340687965149447050","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99444110025206148999881196086066868280","series":"1.2.826.0.1.3680043.8.498.26283827235516621111588239380119001761","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99454657594216441037514296544803013407","series":"1.2.826.0.1.3680043.8.498.51830287579366468569087365077710221183","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99438805806853675793560604625783460513","series":"1.2.826.0.1.3680043.8.498.11556975429759304805676039582571310285","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99493257395588095270208592176887619190","series":"1.2.826.0.1.3680043.8.498.87755678221224118916955956056613583766","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206","series":"1.2.826.0.1.3680043.8.498.17280432617865183599445746558731534322","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99438805806853675793560604625783460513","series":"1.2.826.0.1.3680043.8.498.10439909693459604914092659917124386703","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206","series":"1.2.826.0.1.3680043.8.498.43732551983597414830499856301126265854","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99725523560254732094432244556623376711","series":"1.2.826.0.1.3680043.8.498.11779575322707075398677134300943825902","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99768154649604643975127490783989114506","series":"1.2.826.0.1.3680043.8.498.21045528243146351718412097869011750587","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99635784824711754116112241806442235135","series":"1.2.826.0.1.3680043.8.498.42725121902512084429460282795346233409","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99708702196778120583644711299950439036","series":"1.2.826.0.1.3680043.8.498.12060332525024522944496759188091091266","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99926624968240681772735102303531613209","series":"1.2.826.0.1.3680043.8.498.53465343867315157604776974742591748313","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99957611207313098696146908027229590966","series":"1.2.826.0.1.3680043.8.498.75079892939579407106916363523996097406","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99864345386635409953374818045984558970","series":"1.2.826.0.1.3680043.8.498.32924482106953812582918145667303011423","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99880021280666757817587834481314366555","series":"1.2.826.0.1.3680043.8.498.20295166215924439531177264827446165291","plane":"Coronal","fluid":0,"slot":"COR_T1"}]')
print('series a convertir (UID figes depuis crop130 224) :', len(jobs))
assert len(jobs) == 1988
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    # pad/crop spatial identique : si shapes divergent (rare), resize au mode
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w)

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    band = vol[a:b + 1]
    idx = np.linspace(0, band.shape[0] - 1, N_SLICES).round().astype(int)
    band = band[idx]

    out = np.empty((N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for i in range(N_SLICES):
        sl = cv2.resize(band[i], (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        out[i] = np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    # fallback : parfois series a la racine de train_series
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_320_proto_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / "manifest_proto_part0.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "note": "1 COR_T1 + 1 AX_T1, UID figes depuis crop130 224. Crop 130 mm (PixelSpacing) sinon centre 70 %. Bande profondeur 20-80 %, linspace 24, resize 320, uint8.",
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

# ZipFile n'est pas thread-safe en ecriture : workers convertissent, le
# thread principal ecrit le zip.
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

# Sanity : recharger 3 NPZ, shapes, moments
import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    means = []
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        print(name, vol.shape, vol.dtype, "min", int(vol.min()), "max", int(vol.max()),
              "mean", float(vol.mean()), "std", float(vol.std()))
        assert vol.shape == (N_SLICES, IMG_SIZE, IMG_SIZE), vol.shape
        means.append(float(vol.mean()))
print("mean uint8 (3 samples)", means)
print(f"attendu : shape ({N_SLICES}, {IMG_SIZE}, {IMG_SIZE}) uint8")
print("cible historique NPZ communautaires : mean~42 std~49.5 (crop 130 mm => moments differents, c'est normal)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
